# Нейросетевой синтез четырёхплоскостного ДОЭ

Задание хакатона: 4 одноканальных PNG 128×128 → фазовый ДОЭ 128×128 →
4 реконструкции методом Френеля; λ=532 нм, пиксель ПВМС 10 мкм,
z=[24.1,25.3,26.5,27.7] мм. Экспорт: 8-битные PNG, Pearson и SSIM.

**Run All** запускает обучение, тестовую оценку и экспорт. Включите
Settings → Accelerator → GPU T4 ×2 при доступной квоте; код также работает на CPU.
Интернет и Kaggle API-токен внутри notebook не требуются.

В документах нет четырёх исходных объектных PNG: по умолчанию используется явно
обозначенный процедурный демонстрационный набор. Для своих данных укажите четыре
пути в `INPUT_IMAGES` ниже. Их порядок соответствует возрастающему z.

Для дообучения на текстурах загрузите ZIP и исходные веса как Kaggle Input,
заполните `TEXTURE_ZIP` и `INITIAL_CHECKPOINT`. Тогда notebook разделит данные
по семействам, исключит Cracks/Gabor/Perlin/Voronoi из обучения и сформирует
PDF/CSV-отчёт с результатами до и после.

Метод — фиксированная развёрнутая нейросеть с оптическими проекциями и обучаемыми
CNN-коррекциями. Обучение без эталонных голограмм; при инференсе нет шагов оптимизатора.
Тестовые изображения не участвуют в обучении и выборе checkpoint.


In [ ]:
# Embedded source: inspectable, no Git clone or external download.
from pathlib import Path
import json, os, sys
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'notebook-workdir'
PROJECT = BASE / 'nuclear-hack-source'
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCE_FILES = {'doe/__init__.py': '"""Four-plane neural diffractive optical element synthesis."""\n\n__version__ = "0.1.0"\n', 'doe/__main__.py': 'from .cli import main\n\nmain()\n', 'doe/camera_optics.py': '"""Finite-aperture single-FFT Fresnel model with a declared camera sampling grid."""\nfrom dataclasses import dataclass\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom .optics import OpticsConfig,fft2c,ifft2c\n\n\n@dataclass(frozen=True)\nclass CameraConfig(OpticsConfig):\n    fft_padding: int = 2\n\n    def __post_init__(self):\n        super().__post_init__()\n        if self.fft_padding<1 or not isinstance(self.fft_padding,int):\n            raise ValueError(\'FFT padding must be a positive integer\')\n\n    @property\n    def fft_size(self):return self.size*self.fft_padding\n\n    @property\n    def output_pixel_pitches_m(self):\n        return tuple(self.wavelength_m*z/(self.fft_size*self.pixel_pitch_m) for z in self.distances_m)\n\n    def sampling_report(self):\n        report=super().sampling_report()\n        report.update(fft_size=self.fft_size,controlled_slm_size=self.size,camera_size=self.size,\n                      camera_definition=f\'central {self.size}x{self.size} samples of zero-padded Fresnel FFT; fixed before synthesis; all camera pixels scored\')\n        for p in report[\'planes\']:\n            p[\'output_full_grid_nyquist_ratio\']=p[\'output_chirp_nyquist_ratio\']/self.fft_padding\n            p[\'output_chirp_nyquist_ratio\']/=self.fft_padding**2\n        report[\'interpretation\']=\'Declared finer camera grid; input pupil and phase remain 128x128. Light outside the camera is retained in full-field outputs and energy accounting.\'\n        return report\n\n\nclass CameraFresnel(nn.Module):\n    def __init__(self,config=CameraConfig(),dtype=torch.float32):\n        super().__init__();self.config=config\n        n,m=config.size,config.fft_size\n        dx,lam=config.pixel_pitch_m,config.wavelength_m\n        z=torch.tensor(config.distances_m,dtype=torch.float64)[:,None,None]\n        axis=(torch.arange(n,dtype=torch.float64)-n//2)*dx\n        yy,xx=torch.meshgrid(axis,axis,indexing=\'ij\')\n        phase=math.pi*(xx.square()+yy.square())[None]/(lam*z)\n        out_axis=(torch.arange(m,dtype=torch.float64)-m//2)[None]*(lam*z[:,0,0]/(m*dx))[:,None]\n        out_phase=math.pi*(out_axis[:,:,None].square()+out_axis[:,None,:].square())/(lam*z)\n        cdtype=torch.complex128 if dtype==torch.float64 else torch.complex64\n        self.register_buffer(\'input_chirp\',torch.polar(torch.ones_like(phase),phase).to(cdtype))\n        self.register_buffer(\'output_chirp\',torch.polar(torch.ones_like(out_phase),out_phase).to(cdtype))\n        factor=m*dx**2/(lam*z)*torch.polar(torch.ones_like(z),2*math.pi*z/lam-math.pi/2)\n        self.register_buffer(\'physical_factor\',factor.to(cdtype))\n\n    @property\n    def margin(self):return (self.config.fft_size-self.config.size)//2\n\n    def crop(self,x):\n        p=self.margin;n=self.config.size\n        return x[...,p:p+n,p:p+n]\n\n    def embed(self,x):\n        p=self.margin\n        return F.pad(x,(p,p,p,p))\n\n    def full_field(self,field):\n        return self.output_chirp*fft2c(self.embed(field[:,None]*self.input_chirp))\n\n    def field(self,field):return self.crop(self.full_field(field))\n\n    def forward(self,phase):return self.field(torch.polar(torch.ones_like(phase),phase))\n\n    def full_physical(self,phase):\n        return self.full_field(torch.polar(torch.ones_like(phase),phase))*self.physical_factor\n\n    def physical(self,phase):return self(phase)*self.physical_factor\n\n    def full_adjoint(self,fields):\n        return self.crop(ifft2c(fields*self.output_chirp.conj()))*self.input_chirp.conj()\n\n    def adjoint(self,fields):return self.full_adjoint(self.embed(fields))\n\n    def camera_energy_fraction(self,phase):\n        full=self.full_physical(phase).abs().square()\n        return self.crop(full).sum((-2,-1))/full.sum((-2,-1)).clamp_min(1e-12)\n', 'doe/cli.py': 'import argparse\nimport json\nfrom pathlib import Path\n\nfrom .data import generate_dataset\nfrom .engine import TrainConfig, evaluate, predict, train\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description="Four-plane neural DOE synthesis, single-FFT Fresnel optics")\n    sub = parser.add_subparsers(dest="command", required=True)\n    generate = sub.add_parser("generate", help="Generate independent procedural train/validation/test quartets")\n    generate.add_argument("--output", required=True)\n    generate.add_argument("--train", type=int, default=128)\n    generate.add_argument("--validation", type=int, default=24)\n    generate.add_argument("--test", type=int, default=24)\n    generate.add_argument("--seed", type=int, default=2026)\n    fit = sub.add_parser("train", help="Train with differentiable Fresnel reconstruction, without DOE labels")\n    fit.add_argument("--data", required=True)\n    fit.add_argument("--output", required=True)\n    fit.add_argument("--initial-checkpoint", help="Fine-tune existing weights; optimizer starts fresh")\n    fit.add_argument("--objective",choices=("physics","perceptual"),default="physics")\n    fit.add_argument(\'--architecture\',choices=(\'fixed\',\'wavefront\',\'adaptive\'),default=\'fixed\')\n    fit.add_argument(\'--initializer-width\',type=int,default=8)\n    fit.add_argument(\'--initializer-depth\',type=int,choices=(3,4),default=3)\n    for flag, kind, default in (("epochs", int, 24), ("batch-size", int, 4), ("stages", int, 6),\n                                ("width", int, 12), ("learning-rate", float, 0.005), ("seed", int, 42)):\n        fit.add_argument("--" + flag, type=kind, default=default)\n    infer = sub.add_parser("predict", help="Export DOE, four amplitude PNGs and metrics for exactly four input PNGs")\n    infer.add_argument("--checkpoint", required=True)\n    infer.add_argument("--images", nargs=4, required=True, metavar="PNG")\n    infer.add_argument("--output", required=True)\n    assess = sub.add_parser("evaluate", help="Held-out comparison against untrained network and GS references")\n    assess.add_argument("--data", required=True)\n    assess.add_argument("--checkpoint", required=True)\n    assess.add_argument("--output", required=True)\n    assess.add_argument("--batch-size", type=int, default=4)\n    assess.add_argument("--gs-steps", type=int, default=80)\n    for p in (fit, infer, assess):\n        p.add_argument("--device", default="auto")\n        p.add_argument("--threads", type=int, default=2)\n    for p in (infer,assess):\n        p.add_argument(\'--refine-steps\',type=int,default=0,help=\'Optional per-input optimizer; zero preserves pure neural inference\')\n        p.add_argument(\'--refine-lr\',type=float,default=0.08)\n    args = vars(parser.parse_args(argv))\n    command = args.pop("command")\n    if command == "generate":\n        generate_dataset(args.pop("output"), **args)\n    elif command == "train":\n        data, output = args.pop("data"), args.pop("output")\n        train(data, output, TrainConfig(**args))\n    elif command == "predict":\n        report = predict(**args)\n        print(json.dumps({"mean_pearson": report["mean_pearson"], "mean_ssim": report["mean_ssim"]}, indent=2))\n    else:\n        evaluate(**args)\n\n\nif __name__ == "__main__":\n    main()\n', 'doe/data.py': '"""Procedural, seed-separated image quartets; no external dataset is required."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter, ImageFont\nimport torch\nfrom torch.utils.data import Dataset\n\n\ndef image_array(path, size=128):\n    with Image.open(path) as image:\n        if image.format != "PNG" or image.mode != "L" or image.size != (size, size):\n            raise ValueError(f"{path}: expected an 8-bit grayscale PNG ({size}x{size}), got {image.format}/{image.mode}/{image.size}")\n        return np.asarray(image, dtype=np.uint8).copy()\n\n\ndef load_quartet(paths, size=128):\n    if len(paths) != 4:\n        raise ValueError("supply exactly four PNGs in increasing distance order")\n    arrays = np.stack([image_array(path, size) for path in paths])\n    if (arrays.max(axis=(-2, -1)) == 0).any():\n        raise ValueError("completely black target planes are not supported")\n    return torch.from_numpy(arrays.astype(np.float32) / 255)[None]\n\n\ndef procedural_image(seed, size=128):\n    rng = np.random.default_rng(seed)\n    scale = 2\n    image = Image.new("L", (size * scale, size * scale), 0)\n    draw = ImageDraw.Draw(image)\n    kind = int(rng.integers(0, 4))\n    if kind == 0:\n        font = ImageFont.load_default(size=int(rng.integers(38, 80)) * scale)\n        text = str(rng.integers(0, 10))\n        box = draw.textbbox((0, 0), text, font=font)\n        w, h = box[2] - box[0], box[3] - box[1]\n        x = (size * scale - w) // 2 + int(rng.integers(-12, 13)) * scale\n        y = (size * scale - h) // 2 - box[1] + int(rng.integers(-12, 13)) * scale\n        draw.text((x, y), text, font=font, fill=255)\n    else:\n        for _ in range(int(rng.integers(1, 5))):\n            x, y = rng.integers(size // 8, size * 3 // 4, 2)\n            w, h = rng.integers(size // 8, size // 2, 2)\n            bounds = tuple(int(v * scale) for v in (x, y, min(x + w, size - 5), min(y + h, size - 5)))\n            value = int(rng.integers(140, 256))\n            if kind == 1:\n                draw.ellipse(bounds, fill=value)\n            elif kind == 2:\n                draw.rectangle(bounds, fill=value)\n            else:\n                draw.ellipse(bounds, outline=value, width=int(rng.integers(3, 8)) * scale)\n        if kind == 3:\n            draw.line([(20 * scale, 105 * scale), (100 * scale, 30 * scale)], fill=220, width=3 * scale)\n    image = image.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.5, 1.6)) * scale))\n    image = image.resize((size, size), Image.Resampling.LANCZOS)\n    array = np.array(image)\n    array = np.rint(array.astype(np.float64) * (255 / max(int(array.max()), 1))).astype(np.uint8)\n    return array\n\n\ndef generate_dataset(root, train=128, validation=24, test=24, seed=2026):\n    root = Path(root)\n    if root.exists() and any(root.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite existing dataset: {root}")\n    if min(train, validation, test) < 1:\n        raise ValueError("all three splits must be nonempty")\n    root.mkdir(parents=True, exist_ok=True)\n    manifest = {"source": "procedurally generated demonstration data, not official hackathon inputs",\n                "seed": seed, "size": 128, "mode": "L", "splits": {}}\n    for split_id, (split, count) in enumerate((("train", train), ("validation", validation), ("test", test))):\n        records = []\n        for index in range(count):\n            sample = root / split / f"sample_{index:04d}"\n            sample.mkdir(parents=True)\n            seeds = []\n            hashes = []\n            for plane in range(4):\n                image_seed = seed + split_id * 1_000_000 + index * 16 + plane * 101\n                path = sample / f"plane_{plane + 1}.png"\n                Image.fromarray(procedural_image(image_seed)).save(path)\n                seeds.append(image_seed)\n                hashes.append(hashlib.sha256(path.read_bytes()).hexdigest())\n            records.append({"path": str(sample.relative_to(root)), "image_seeds": seeds, "sha256": hashes})\n        manifest["splits"][split] = records\n    (root / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    return manifest\n\n\nclass QuartetDataset(Dataset):\n    def __init__(self, root, split):\n        root = Path(root)\n        manifest = json.loads((root / "manifest.json").read_text())\n        self.samples = [root / item["path"] for item in manifest["splits"][split]]\n        if not self.samples:\n            raise ValueError(f"empty dataset split: {split}")\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, index):\n        return load_quartet([self.samples[index] / f"plane_{i}.png" for i in range(1, 5)])[0]\n', 'doe/engine.py': '"""Reproducible training, held-out evaluation and inference."""\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom .data import QuartetDataset, load_quartet\nfrom .model import FresnelNet, build_model, gerchberg_saxton, physics_loss\nfrom .optics import OpticsConfig\nfrom .quality import perceptual_loss_per_item\nfrom .refinement import refine_phase\nfrom .results import export_result, phase_metrics\n\n\n@dataclass\nclass TrainConfig:\n    epochs: int = 24\n    batch_size: int = 4\n    learning_rate: float = 0.005\n    stages: int = 6\n    width: int = 12\n    seed: int = 42\n    device: str = "auto"\n    threads: int = 2\n    initial_checkpoint: str | None = None\n    objective: str = "physics"\n    architecture: str = "fixed"\n    initializer_width: int = 8\n    initializer_depth: int = 3\n\n\ndef setup(seed=42, device="auto", threads=2):\n    if threads < 1:\n        raise ValueError("threads must be >= 1")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.set_num_threads(threads)\n    if device == "auto":\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device.startswith("cuda") and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but unavailable; select --device cpu")\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n    return torch.device(device)\n\n\ndef sync(device):\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n\n\ndef load_model(checkpoint, device):\n    state = torch.load(checkpoint, map_location=device, weights_only=True)\n    if state.get("format_version") != 1:\n        raise ValueError("unsupported checkpoint format")\n    optics = OpticsConfig(**state["optics"])\n    model = build_model(optics, state["architecture"]).to(device)\n    model.load_state_dict(state["model"])\n    model.eval()\n    return model, state\n\n\n@torch.no_grad()\ndef validation_loss(model, loader, device, objective="physics"):\n    model.eval()\n    total, count = 0.0, 0\n    for targets in loader:\n        targets = targets.to(device)\n        phase=model(targets)\n        loss = (perceptual_loss_per_item(model.optics,phase,targets).mean() if objective=="perceptual"\n                else physics_loss(model.optics,phase,targets))\n        total += float(loss) * len(targets)\n        count += len(targets)\n    if not count:\n        raise ValueError("validation set is empty")\n    return total / count\n\n\ndef train(data, output, config=TrainConfig()):\n    if config.epochs < 1 or config.batch_size < 1 or config.learning_rate <= 0:\n        raise ValueError("epochs, batch size and learning rate must be positive")\n    if config.objective not in ("physics","perceptual"):\n        raise ValueError("objective must be physics or perceptual")\n    if config.architecture not in (\'fixed\',\'wavefront\',\'adaptive\'):\n        raise ValueError(\'architecture must be fixed, wavefront or adaptive\')\n    device = setup(config.seed, config.device, config.threads)\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite training run: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    train_data, val_data = QuartetDataset(data, "train"), QuartetDataset(data, "validation")\n    loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True,\n                        generator=torch.Generator().manual_seed(config.seed), num_workers=0)\n    val_loader = DataLoader(val_data, batch_size=config.batch_size, num_workers=0)\n    parent = None\n    requested={\'kind\':config.architecture,\'stages\':config.stages,\'width\':config.width}\n    if config.architecture!=\'fixed\':\n        requested.update(initializer_width=config.initializer_width,initializer_depth=config.initializer_depth)\n    if config.initial_checkpoint:\n        model, previous = load_model(config.initial_checkpoint, device)\n        old_arch=model.architecture();old_stages=model.stages\n        if model.width!=config.width or config.stages<old_stages:\n            raise ValueError(\'width must match and stage count cannot decrease during fine-tuning\')\n        if config.architecture==\'fixed\' and (old_arch[\'kind\']!=\'fixed\' or config.stages!=old_stages):\n            raise ValueError(\'requested fixed architecture differs from checkpoint\')\n        if old_arch[\'kind\']==\'adaptive\' and config.architecture!=\'adaptive\':\n            raise ValueError(\'cannot discard adaptive projection layers during fine-tuning\')\n        if old_arch[\'kind\']!=\'fixed\' and (config.initializer_width!=old_arch[\'initializer_width\'] or config.initializer_depth<old_arch[\'initializer_depth\']):\n            raise ValueError(\'initializer width must match and depth cannot decrease\')\n        if old_arch!=requested:\n            expanded=build_model(model.optics.config,requested).to(device)\n            transferred=dict(model.state_dict())\n            if config.stages!=old_stages:\n                indices=torch.arange(config.stages,device=device)%old_stages\n                transferred[\'plane_logits\']=transferred[\'plane_logits\'][indices].clone()\n                transferred[\'relax_logits\']=transferred[\'relax_logits\'][indices].clone()\n            missing,unexpected=expanded.load_state_dict(transferred,strict=False)\n            allowed=lambda k:k.startswith((\'phase_initializer.\',\'amplitude_modulators.\')) or k in (\'reference_phase\',\'coordinate_grid\',\'deep_scale\')\n            assert not unexpected and all(allowed(k) for k in missing)\n            model=expanded\n        parent = {"filename": Path(config.initial_checkpoint).name,\n                  "sha256": hashlib.sha256(Path(config.initial_checkpoint).read_bytes()).hexdigest(),\n                  "epoch": previous["epoch"],"architecture":old_arch}\n    else:\n        model = build_model(OpticsConfig(),requested).to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)\n    initial = validation_loss(model, val_loader, device, config.objective)\n    initial_parameters = {name: p.detach().clone() for name, p in model.named_parameters()}\n    best_loss, best_epoch = initial, 0\n    history = [{"epoch": 0, "train_loss": None, "validation_loss": initial}]\n    manifest_hash = hashlib.sha256((Path(data) / "manifest.json").read_bytes()).hexdigest()\n\n    def save(path, epoch, val_loss):\n        torch.save({"format_version": 1, "model": model.state_dict(),\n                    "architecture": model.architecture(),\n                    "optics": model.optics.config.to_dict(), "training": asdict(config),\n                    "epoch": epoch, "validation_loss": val_loss,\n                    "dataset_manifest_sha256": manifest_hash, "parent_checkpoint": parent}, path)\n\n    save(output / "initial.pt", 0, initial)\n    save(output / "best.pt", 0, initial)\n    start = time.perf_counter()\n    print(f"device={device}; train={len(train_data)} validation={len(val_data)}; initial_val={initial:.6f}", flush=True)\n    for epoch in range(1, config.epochs + 1):\n        model.train()\n        total = 0.0\n        for targets in loader:\n            targets = targets.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            phase = model(targets)\n            loss = (perceptual_loss_per_item(model.optics,phase,targets).mean() if config.objective=="perceptual"\n                    else physics_loss(model.optics,phase,targets))\n            if not torch.isfinite(loss):\n                raise FloatingPointError("nonfinite training loss")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n            optimizer.step()\n            total += float(loss.detach()) * len(targets)\n        val_loss = validation_loss(model, val_loader, device, config.objective)\n        if val_loss < best_loss:\n            best_loss, best_epoch = val_loss, epoch\n            save(output / "best.pt", epoch, val_loss)\n        record = {"epoch": epoch, "train_loss": total / len(train_data), "validation_loss": val_loss,\n                  "learning_rate": optimizer.param_groups[0]["lr"]}\n        history.append(record)\n        scheduler.step()\n        print(f"epoch={epoch:03d} train={record[\'train_loss\']:.6f} val={val_loss:.6f} best={best_loss:.6f}", flush=True)\n    sync(device)\n    save(output / "last.pt", config.epochs, history[-1]["validation_loss"])\n    drift = sum(float((p.detach() - initial_parameters[name]).square().sum()) for name, p in model.named_parameters())**0.5\n    report = {"config": asdict(config), "device": str(device), "torch": torch.__version__,\n              "train_count": len(train_data), "validation_count": len(val_data),\n              "selection": f"minimum validation {config.objective} loss; test split never used for selection",\n              "initial_validation_loss": initial, "best_validation_loss": best_loss, "best_epoch": best_epoch,\n              "parameter_l2_change_last_vs_initial": drift,\n              "elapsed_seconds": time.perf_counter() - start, "history": history,\n              "dataset_manifest_sha256": manifest_hash, "parent_checkpoint": parent,\n              "optimizer": "fresh Adam state; fine-tuning weights, not resuming optimizer state"}\n    (output / "training.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n    return report\n\n\n@torch.no_grad()\ndef evaluate(data, checkpoint, output, device="auto", batch_size=4, gs_steps=80, threads=2, refine_steps=0, refine_lr=0.08):\n    device = setup(device=device, threads=threads)\n    model, state = load_model(checkpoint, device)\n    initial = build_model(model.optics.config, model.architecture()).to(device).eval()\n    # Identical analytical initialization; CNN starts as exact zero correction.\n    test = QuartetDataset(data, "test")\n    loader = DataLoader(test, batch_size=batch_size, num_workers=0)\n    methods = {"neural": lambda x: model(x), "untrained_network": lambda x: initial(x),\n               f"gs_{model.stages}": lambda x: gerchberg_saxton(model.optics, x, model.stages),\n               f"gs_{gs_steps}": lambda x: gerchberg_saxton(model.optics, x, gs_steps)}\n    if refine_steps:\n        methods[\'neural_refined\']=lambda x: refine_phase(model.optics,model(x),x,steps=refine_steps,lr=refine_lr)[0]\n    records = {name: [] for name in methods}\n    seconds = {name: 0.0 for name in methods}\n    # Exclude one warmup batch per method from performance measurements.\n    warmup = next(iter(loader)).to(device)\n    for name,operation in methods.items():\n        # Warm kernels without repeating a potentially long optimization run.\n        if name==\'neural_refined\':\n            refine_phase(model.optics,model(warmup),warmup,steps=min(2,refine_steps),lr=refine_lr)\n        else:\n            operation(warmup)\n    for targets in loader:\n        targets = targets.to(device)\n        for name, operation in methods.items():\n            sync(device)\n            start = time.perf_counter()\n            phase = operation(targets)\n            sync(device)\n            seconds[name] += time.perf_counter() - start\n            records[name].extend(phase_metrics(model.optics, phase, targets))\n    summary = {}\n    for name, values in records.items():\n        correlations = [v["pearson"] for v in values if v["pearson"] is not None]\n        summary[name] = {"count": len(values), "mean_pearson": float(np.mean(correlations)) if correlations else None,\n                         "mean_ssim": float(np.mean([v["ssim"] for v in values])),\n                         "seconds_per_quartet_batched": seconds[name] / len(values)}\n    report = {"split": "test", "sample_count": len(test), "device": str(device), "batch_size": batch_size,\n              "architecture":model.architecture(),\n              "checkpoint_epoch": state["epoch"], "checkpoint_sha256": hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest(),\n              "quantization": "8-bit phase and 8-bit reconstructed amplitudes, same for all methods",\n              "timing": "one warmed pass per test batch; includes phase synthesis, excludes image IO and metric computation; batched throughput, not single-sample latency",\n              "summary": summary, "per_sample": records}\n    if refine_steps:\n        report[\'refinement\']={\'steps\':refine_steps,\'lr\':refine_lr,\'method\':\'neural initializer + per-input phase Adam; not a pure network\'}\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    if output.exists():\n        raise FileExistsError(f"Refusing to overwrite evaluation: {output}")\n    output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n    print(json.dumps(summary, indent=2), flush=True)\n    return report\n\n\n@torch.no_grad()\ndef predict(checkpoint, images, output, device="auto", threads=2, source="user PNGs",refine_steps=0,refine_lr=0.08):\n    device = setup(device=device, threads=threads)\n    model, state = load_model(checkpoint, device)\n    targets = load_quartet(images).to(device)\n    model(targets)  # warmup\n    sync(device)\n    start = time.perf_counter()\n    phase = model(targets)\n    sync(device)\n    elapsed = time.perf_counter() - start\n    refinement=None\n    if refine_steps:\n        phase,refinement=refine_phase(model.optics,phase,targets,steps=refine_steps,lr=refine_lr)\n    metadata = {"method": "FresnelNet", "network_stages": model.stages, "cnn_width": model.width,\n                "architecture":model.architecture(),\n                "source": source, "device": str(device), "checkpoint_epoch": state["epoch"],\n                "checkpoint_sha256": hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest(),\n                "network_forward_seconds": elapsed, "test_time_optimizer_steps": refine_steps}\n    if refinement:\n        metadata[\'method\']=\'FresnelNet + per-input phase Adam\'\n        metadata[\'refinement\']=refinement\n    return export_result(output, model.optics, phase, images, metadata)\n', 'doe/model.py': '"""A fixed-depth, trainable, physics-informed unrolled network."""\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom .optics import Fresnel, OpticsConfig, normalize_targets, unit_complex\n\n\ndef initialization(optics, targets):\n    # Deterministic random output phases reduce coherent bias. Same seed at test time.\n    generator = torch.Generator().manual_seed(1701)\n    phase = (torch.rand((4, optics.config.size, optics.config.size), generator=generator) * 2 - 1) * math.pi\n    phase = phase.to(device=targets.device, dtype=targets.dtype)\n    fields = targets * torch.polar(torch.ones_like(phase), phase)\n    return unit_complex(optics.adjoint(fields).mean(dim=1))\n\n\nclass FresnelNet(nn.Module):\n    """Six-to-eight optical projection layers with learned CNN phase corrections.\n\n    Inference is one fixed-depth network forward pass, without optimizer steps.\n    This is an unrolled neural method, not a claim of zero Fourier transforms.\n    """\n\n    def __init__(self, config=OpticsConfig(), stages=6, width=12):\n        super().__init__()\n        if stages < 1 or width < 1:\n            raise ValueError("stages and width must be positive")\n        self.optics = Fresnel(config)\n        self.stages = stages\n        self.width = width\n        self.plane_logits = nn.Parameter(torch.zeros(stages, 4))\n        # Start at the classical projection (rho=1); learn under/over-relaxation in (0,2).\n        self.relax_logits = nn.Parameter(torch.zeros(stages))\n        self.corrector = nn.Sequential(\n            nn.Conv2d(4, width, 3, padding=1, padding_mode="circular"), nn.SiLU(),\n            nn.Conv2d(width, width, 3, padding=1, padding_mode="circular"), nn.SiLU(),\n            nn.Conv2d(width, 1, 1),\n        )\n        nn.init.zeros_(self.corrector[-1].weight)\n        nn.init.zeros_(self.corrector[-1].bias)\n\n    def architecture(self):\n        return {"kind":"fixed","stages":self.stages,"width":self.width}\n\n    def initial_field(self,amplitude):\n        return initialization(self.optics,amplitude)\n\n    def constrained_fields(self,amplitude,reconstructed,stage):\n        return amplitude*unit_complex(reconstructed)\n\n    def forward(self, targets):\n        amplitude = normalize_targets(targets)\n        field = self.initial_field(amplitude)\n        for stage in range(self.stages):\n            reconstructed = self.optics.field(field)\n            constrained = self.constrained_fields(amplitude,reconstructed,stage)\n            back = self.optics.adjoint(constrained)\n            weights = self.plane_logits[stage].softmax(dim=0)[None, :, None, None]\n            projected = unit_complex((weights * back).sum(dim=1))\n            features = torch.stack((field.real, field.imag, projected.real, projected.imag), dim=1)\n            correction = 0.35 * torch.tanh(self.corrector(features)[:, 0])\n            relax = 2 * self.relax_logits[stage].sigmoid()\n            field = unit_complex((1 - relax) * field + relax * projected)\n            field = field * torch.polar(torch.ones_like(correction), correction)\n        return torch.angle(field)\n\n\nclass WavefrontBlock(nn.Module):\n    def __init__(self,cin,cout):\n        super().__init__()\n        self.net=nn.Sequential(\n            nn.Conv2d(cin,cout,3,padding=1,padding_mode=\'circular\'),nn.GroupNorm(4,cout),nn.SiLU(),\n            nn.Conv2d(cout,cout,3,padding=1,padding_mode=\'circular\'),nn.GroupNorm(4,cout),nn.SiLU())\n    def forward(self,x):\n        return self.net(x)\n\n\nclass WavefrontFresnelNet(FresnelNet):\n    """Learn four object-plane phases with a nonlocal U-Net before shared-DOE synthesis."""\n    def __init__(self,config=OpticsConfig(),stages=6,width=12,initializer_width=8,initializer_depth=3):\n        super().__init__(config,stages,width)\n        if initializer_width<4 or initializer_width%4:\n            raise ValueError(\'initializer width must be a positive multiple of 4\')\n        if initializer_depth not in (3,4):raise ValueError(\'initializer depth must be 3 or 4\')\n        w=initializer_width;self.initializer_width=w;self.initializer_depth=initializer_depth\n        self.phase_initializer=nn.ModuleDict({\n            \'first\':WavefrontBlock(10,w),\'down1\':WavefrontBlock(w,2*w),\n            \'down2\':WavefrontBlock(2*w,4*w),\'up1\':WavefrontBlock(6*w,2*w),\n            \'up0\':WavefrontBlock(3*w,w),\'head\':nn.Conv2d(w,4,1)})\n        nn.init.zeros_(self.phase_initializer[\'head\'].weight)\n        nn.init.zeros_(self.phase_initializer[\'head\'].bias)\n        if initializer_depth==4:\n            self.phase_initializer[\'down3\']=WavefrontBlock(4*w,8*w)\n            self.phase_initializer[\'up2\']=WavefrontBlock(12*w,4*w)\n            self.deep_scale=nn.Parameter(torch.zeros(()))\n        n=config.size\n        axis=torch.arange(n,dtype=torch.float32)-n//2\n        yy,xx=torch.meshgrid(axis,axis,indexing=\'ij\')\n        beta=sum(math.pi*config.pixel_pitch_m**2/(config.wavelength_m*z) for z in config.distances_m)/4\n        carrier=(math.pi/n-beta)*(xx.square()+yy.square())\n        with torch.no_grad():reference=unit_complex(self.optics(carrier[None]))[0]\n        self.register_buffer(\'reference_phase\',reference)\n        self.register_buffer(\'coordinate_grid\',torch.stack((xx/(n/2),yy/(n/2)),dim=0))\n\n    def architecture(self):\n        return {\'kind\':\'wavefront\',\'stages\':self.stages,\'width\':self.width,\'initializer_width\':self.initializer_width,\n                \'initializer_depth\':self.initializer_depth}\n\n    def initial_field(self,amplitude):\n        peak=amplitude.amax(dim=(-2,-1),keepdim=True).clamp_min(1e-8)\n        coordinates=self.coordinate_grid[None].expand(len(amplitude),-1,-1,-1)\n        x=torch.cat((amplitude,amplitude/peak,coordinates),dim=1)\n        layers=self.phase_initializer\n        a=layers[\'first\'](x)\n        b=layers[\'down1\'](F.avg_pool2d(a,2))\n        c=layers[\'down2\'](F.avg_pool2d(b,2))\n        if self.initializer_depth==4:\n            deep=layers[\'down3\'](F.avg_pool2d(c,2))\n            correction=layers[\'up2\'](torch.cat((F.interpolate(deep,size=c.shape[-2:],mode=\'bilinear\',align_corners=False),c),dim=1))\n            c=c+self.deep_scale*correction\n        b2=layers[\'up1\'](torch.cat((F.interpolate(c,size=b.shape[-2:],mode=\'bilinear\',align_corners=False),b),dim=1))\n        a2=layers[\'up0\'](torch.cat((F.interpolate(b2,size=a.shape[-2:],mode=\'bilinear\',align_corners=False),a),dim=1))\n        theta=math.pi*torch.tanh(layers[\'head\'](a2))\n        fields=amplitude*self.reference_phase*torch.polar(torch.ones_like(theta),theta)\n        return unit_complex(self.optics.adjoint(fields).mean(dim=1))\n\n\nclass AdaptiveWavefrontNet(WavefrontFresnelNet):\n    """Learn the amplitude freedom and phase correction of each optical projection."""\n    def __init__(self,config=OpticsConfig(),stages=12,width=12,initializer_width=8,initializer_depth=4):\n        super().__init__(config,stages,width,initializer_width,initializer_depth)\n        self.amplitude_modulators=nn.ModuleList()\n        for _ in range(stages):\n            net=nn.Sequential(nn.Conv2d(20,8,3,padding=1,padding_mode=\'circular\'),nn.SiLU(),\n                              nn.Conv2d(8,8,3,padding=1,padding_mode=\'circular\'),nn.SiLU(),nn.Conv2d(8,12,1))\n            nn.init.zeros_(net[-1].weight);nn.init.zeros_(net[-1].bias)\n            with torch.no_grad():net[-1].bias[4:8].fill_(-4)\n            self.amplitude_modulators.append(net)\n\n    def architecture(self):\n        arch=super().architecture();arch[\'kind\']=\'adaptive\';return arch\n\n    def constrained_fields(self,amplitude,reconstructed,stage):\n        magnitude=reconstructed.abs()\n        direction=unit_complex(reconstructed)\n        features=torch.cat((amplitude,magnitude,amplitude-magnitude,direction.real,direction.imag),dim=1)\n        modulation=self.amplitude_modulators[stage](features)\n        desired=amplitude*torch.exp(.75*torch.tanh(modulation[:,:4]))\n        freedom=modulation[:,4:8].sigmoid()*magnitude\n        phase=math.pi*torch.tanh(modulation[:,8:])\n        return (desired+freedom)*direction*torch.polar(torch.ones_like(phase),phase)\n\n\ndef build_model(config,architecture):\n    args=dict(architecture)\n    kind=args.pop(\'kind\',\'fixed\')\n    if kind==\'fixed\':return FresnelNet(config,**args)\n    if kind==\'wavefront\':return WavefrontFresnelNet(config,**args)\n    if kind==\'adaptive\':return AdaptiveWavefrontNet(config,**args)\n    raise ValueError(f\'unknown architecture kind: {kind}\')\n\n\n@torch.no_grad()\ndef gerchberg_saxton(optics, targets, steps=80):\n    """Multi-plane alternating-projection reference, using the identical physics."""\n    if steps < 1:\n        raise ValueError("steps must be positive")\n    amplitude = normalize_targets(targets)\n    field = initialization(optics, amplitude)\n    for _ in range(steps):\n        reconstructed = optics.field(field)\n        field = unit_complex(optics.adjoint(amplitude * unit_complex(reconstructed)).mean(dim=1))\n    return torch.angle(field)\n\n\ndef physics_loss(optics, phase, targets):\n    """Supervision-free amplitude loss; no ground-truth holograms are needed."""\n    desired = normalize_targets(targets)\n    amplitude = optics(phase).abs()\n    loss = F.mse_loss(amplitude, desired)\n    for scale in (2, 4):\n        loss = loss + 0.15 * F.mse_loss(F.avg_pool2d(amplitude, scale), F.avg_pool2d(desired, scale))\n    centered_a = amplitude - amplitude.mean(dim=(-2, -1), keepdim=True)\n    centered_b = desired - desired.mean(dim=(-2, -1), keepdim=True)\n    corr = (centered_a * centered_b).mean(dim=(-2, -1)) / (\n        centered_a.square().mean(dim=(-2, -1)) * centered_b.square().mean(dim=(-2, -1))\n    ).clamp_min(1e-10).sqrt()\n    return loss + 0.1 * (1 - corr).mean()\n', 'doe/optics.py': '"""Centered single-FFT Fresnel propagation on its natural output grids."""\nfrom dataclasses import asdict, dataclass\nimport math\n\nimport torch\nfrom torch import nn\n\n\n@dataclass(frozen=True)\nclass OpticsConfig:\n    size: int = 128\n    wavelength_m: float = 532e-9\n    pixel_pitch_m: float = 10e-6\n    distances_m: tuple[float, ...] = (0.0241, 0.0253, 0.0265, 0.0277)\n\n    def __post_init__(self):\n        if self.size < 8 or self.size % 2:\n            raise ValueError("size must be even and >= 8")\n        if not math.isfinite(self.wavelength_m) or self.wavelength_m <= 0:\n            raise ValueError("wavelength must be finite and positive")\n        if not math.isfinite(self.pixel_pitch_m) or self.pixel_pitch_m <= 0:\n            raise ValueError("pixel pitch must be finite and positive")\n        if len(self.distances_m) != 4 or any(not math.isfinite(z) or z <= 0 for z in self.distances_m):\n            raise ValueError("exactly four finite positive distances are required")\n\n    @property\n    def output_pixel_pitches_m(self):\n        return tuple(self.wavelength_m * z / (self.size * self.pixel_pitch_m) for z in self.distances_m)\n\n    def to_dict(self):\n        return asdict(self)\n\n    def sampling_report(self):\n        """Conservative edge-frequency checks for the two quadratic chirps.\n\n        The output chirp is only a pointwise factor, never FFT-transformed again;\n        its sampling affects interpolation of complex phase, not sampled |U|.\n        """\n        critical = self.size * self.pixel_pitch_m**2 / self.wavelength_m\n        return {\n            "critical_distance_m": critical,\n            "criterion": "conservative edge chirp frequency / Nyquist; <=1 passes",\n            "planes": [{"distance_m": z, "input_chirp_nyquist_ratio": critical / z,\n                        "output_chirp_nyquist_ratio": z / critical,\n                        "input_chirp_passes": critical / z <= 1} for z in self.distances_m],\n            "interpretation": "input chirp must be resolved before FFT; output chirp may exceed Nyquist for complex-field interpolation, but is pointwise unit-modulus and does not alter sampled amplitude. Do not claim both chirps are aliasing-free.",\n        }\n\n\ndef fft2c(field):\n    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))\n\n\ndef ifft2c(field):\n    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(field, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))\n\n\nclass Fresnel(nn.Module):\n    """Unitary preconditioned Fresnel operator; physical() restores its prefactor.\n\n    Input: phase [B,H,W], uniform unit-amplitude illumination.\n    Output: complex field [B,4,H,W]. Each plane has its own output pixel pitch.\n    The unitary normalization changes only one constant per plane, not its image.\n    """\n\n    def __init__(self, config=OpticsConfig(), dtype=torch.float32):\n        super().__init__()\n        self.config = config\n        n, dx, lam = config.size, config.pixel_pitch_m, config.wavelength_m\n        # Compute oscillatory chirps in double precision before optional casting.\n        axis = (torch.arange(n, dtype=torch.float64) - n // 2) * dx\n        yy, xx = torch.meshgrid(axis, axis, indexing="ij")\n        z = torch.tensor(config.distances_m, dtype=torch.float64)[:, None, None]\n        input_phase = math.pi * (xx.square() + yy.square())[None] / (lam * z)\n        output_axis = (torch.arange(n, dtype=torch.float64) - n // 2)[None] * (lam * z[:, 0, 0] / (n * dx))[:, None]\n        output_phase = math.pi * (output_axis[:, :, None].square() + output_axis[:, None, :].square()) / (lam * z)\n        complex_dtype = torch.complex128 if dtype == torch.float64 else torch.complex64\n        self.register_buffer("input_chirp", torch.polar(torch.ones_like(input_phase), input_phase).to(complex_dtype))\n        self.register_buffer("output_chirp", torch.polar(torch.ones_like(output_phase), output_phase).to(complex_dtype))\n        global_phase = 2 * math.pi * z / lam - math.pi / 2\n        factor = n * dx**2 / (lam * z) * torch.polar(torch.ones_like(global_phase), global_phase)\n        self.register_buffer("physical_factor", factor.to(complex_dtype))\n\n    def field(self, field):\n        if field.shape[-2:] != (self.config.size, self.config.size):\n            raise ValueError("field size does not match optics configuration")\n        return self.output_chirp * fft2c(field[:, None] * self.input_chirp)\n\n    def forward(self, phase):\n        return self.field(torch.polar(torch.ones_like(phase), phase))\n\n    def adjoint(self, fields):\n        """Per-plane unitary adjoint, without summing over planes."""\n        return ifft2c(fields * self.output_chirp.conj()) * self.input_chirp.conj()\n\n    def physical(self, phase):\n        return self(phase) * self.physical_factor\n\n\ndef unit_complex(field, eps=1e-7):\n    # Stable fallback at zeros, including blank regions in target back-projection.\n    return torch.where(field.abs() > eps, field / field.abs().clamp_min(eps), torch.ones_like(field))\n\n\ndef normalize_targets(targets):\n    if targets.ndim != 4 or targets.shape[1] != 4:\n        raise ValueError("targets must have shape [batch,4,height,width]")\n    if not torch.isfinite(targets).all() or (targets < 0).any():\n        raise ValueError("targets must be finite nonnegative amplitudes")\n    rms = targets.square().mean(dim=(-2, -1), keepdim=True).sqrt()\n    if (rms <= 1e-8).any():\n        raise ValueError("a completely black plane is incompatible with unit-amplitude phase-only illumination")\n    return targets / rms\n', 'doe/quality.py': '"""Differentiable counterparts of the reported Pearson and local SSIM metrics."""\nimport math\n\nimport torch\nfrom torch.nn import functional as F\n\n\nclass _SoftPeakGradient(torch.autograd.Function):\n    """Exact maximum forward; distributed logsumexp surrogate in backward only."""\n    @staticmethod\n    def forward(ctx,amplitude,temperature):\n        weights=(amplitude.flatten(2)/temperature).softmax(dim=-1).reshape_as(amplitude)\n        ctx.save_for_backward(weights)\n        return amplitude.amax(dim=(-2,-1),keepdim=True)\n\n    @staticmethod\n    def backward(ctx,gradient):\n        weights,=ctx.saved_tensors\n        return gradient*weights,None\n\n\ndef ssim_per_plane(a, b, window=7, return_map=False):\n    """Match skimage SSIM: uniform valid window, sample covariance, data_range=1."""\n    if a.shape != b.shape or a.ndim != 4:\n        raise ValueError("SSIM expects equally sized [B,planes,H,W] tensors")\n    if window % 2 != 1 or min(a.shape[-2:]) < window:\n        raise ValueError("window must be odd and fit the images")\n    shape = a.shape[:2]\n    x, y = a.flatten(0, 1)[:, None], b.flatten(0, 1)[:, None]\n    mean_x, mean_y = F.avg_pool2d(x, window, stride=1), F.avg_pool2d(y, window, stride=1)\n    correction = window**2 / (window**2 - 1)\n    var_x = correction * (F.avg_pool2d(x*x, window, stride=1) - mean_x.square())\n    var_y = correction * (F.avg_pool2d(y*y, window, stride=1) - mean_y.square())\n    cov = correction * (F.avg_pool2d(x*y, window, stride=1) - mean_x*mean_y)\n    c1, c2 = 0.01**2, 0.03**2\n    score = ((2*mean_x*mean_y+c1) * (2*cov+c2) /\n             ((mean_x.square()+mean_y.square()+c1) * (var_x+var_y+c2)))\n    if return_map:return score[:,0].reshape(*shape,*score.shape[-2:])\n    return score.mean(dim=(1, 2, 3)).reshape(shape)\n\n\ndef pearson_per_plane(a, b):\n    x = a - a.mean(dim=(-2, -1), keepdim=True)\n    y = b - b.mean(dim=(-2, -1), keepdim=True)\n    numerator = (x*y).mean(dim=(-2, -1))\n    denominator = (x.square().mean(dim=(-2, -1)) * y.square().mean(dim=(-2, -1))).clamp_min(1e-16).sqrt()\n    return numerator / denominator\n\n\ndef quantized_phase_ste(phase):\n    """Forward: actual 8-bit phase. Backward: straight-through derivative."""\n    wrapped = torch.atan2(phase.sin(), phase.cos())\n    precise = wrapped.to(torch.float64).clamp(-math.pi, math.pi)\n    code = torch.round((precise + math.pi) * (255 / (2*math.pi)))\n    decoded = (code * (2*math.pi/255) - math.pi).to(phase.dtype)\n    return wrapped + (decoded-wrapped).detach()\n\n\ndef display_amplitudes(optics, phase, quantize=True,max_temperature=0):\n    actual_phase = quantized_phase_ste(phase) if quantize else phase\n    amplitude = optics.physical(actual_phase).abs()\n    peak=(_SoftPeakGradient.apply(amplitude,max_temperature) if max_temperature>0\n          else amplitude.amax(dim=(-2,-1),keepdim=True))\n    display = amplitude / peak.clamp_min(1e-8)\n    if quantize:\n        rounded = (display*255).round() / 255\n        display = display + (rounded-display).detach()\n    return display\n\n\ndef perceptual_loss_per_item(optics, phase, targets, quantize=True,max_temperature=0):\n    display = display_amplitudes(optics, phase, quantize=quantize,max_temperature=max_temperature)\n    ssim = ssim_per_plane(display, targets)\n    pearson = pearson_per_plane(display, targets)\n    # Constant target planes have undefined Pearson; omit its term for those planes.\n    varying = targets.var(dim=(-2, -1), unbiased=False) > 1e-12\n    correlation_loss = ((1-pearson)*varying).sum(dim=1) / varying.sum(dim=1).clamp_min(1)\n    mse = (display-targets).square().mean(dim=(1, 2, 3))\n    return 0.5*(1-ssim).mean(dim=1) + 0.5*correlation_loss + 0.05*mse\n\n\ndef robust_quality_loss_per_item(optics,phase,targets,quantize=True,max_temperature=0):\n    """Logarithmic quality penalties emphasize weak-contrast and badly matched windows.\n\n    Used only as an optimization objective; final PNG metrics are unchanged.\n    Softplus prevents zero gradients or undefined logs for negative initial SSIM.\n    """\n    display=display_amplitudes(optics,phase,quantize=quantize,max_temperature=max_temperature)\n    local=ssim_per_plane(display,targets,return_map=True)\n    correlation=pearson_per_plane(display,targets)\n    varying=targets.var(dim=(-2,-1),unbiased=False)>1e-12\n    pcc_penalty=-torch.log(F.softplus(10*correlation)/10+1e-8)\n    pcc_loss=(pcc_penalty*varying).sum(dim=1)/varying.sum(dim=1).clamp_min(1)\n    ssim_loss=-torch.log(F.softplus(20*local)/20+1e-8).mean(dim=(1,2,3))\n    mse=(display-targets).square().mean(dim=(1,2,3))\n    return .5*pcc_loss+.5*ssim_loss+.05*mse\n', 'doe/refinement.py': '"""Optional per-input phase optimization, explicitly separate from network inference."""\nimport math\nimport time\n\nimport torch\n\nfrom .optics import normalize_targets\nfrom .quality import perceptual_loss_per_item\n\n\ndef refine_phase(optics, initial_phase, targets, steps=600, lr=0.08, check_every=25):\n    if steps < 0 or lr <= 0 or check_every < 1:\n        raise ValueError("steps must be >=0; lr and check_every must be positive")\n    if initial_phase.ndim != 3 or targets.shape != (len(initial_phase), 4, *initial_phase.shape[-2:]):\n        raise ValueError("one shared phase [B,H,W] and four target planes are required")\n    normalize_targets(targets)\n    if not torch.isfinite(initial_phase).all():\n        raise ValueError("initial phase must be finite")\n    started=time.perf_counter()\n    history=[]\n    with torch.enable_grad():\n        phase=torch.nn.Parameter(initial_phase.detach().clone())\n        target=targets.detach()\n        optimizer=torch.optim.Adam([phase],lr=lr)\n        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,steps),eta_min=lr/20)\n        with torch.no_grad():\n            best_loss=perceptual_loss_per_item(optics,phase,target)\n            best=phase.detach().clone()\n            best_step=torch.zeros(len(phase),dtype=torch.long,device=phase.device)\n            history.append({\'step\':0,\'loss\':float(best_loss.mean())})\n        for step in range(1,steps+1):\n            optimizer.zero_grad(set_to_none=True)\n            loss=perceptual_loss_per_item(optics,phase,target).mean()\n            if not torch.isfinite(loss):\n                raise FloatingPointError("nonfinite refinement loss")\n            loss.backward()\n            optimizer.step();scheduler.step()\n            if step%check_every==0 or step==steps:\n                with torch.no_grad():\n                    current=perceptual_loss_per_item(optics,phase,target)\n                    improve=current<best_loss\n                    best[improve]=phase.detach()[improve]\n                    best_step[improve]=step\n                    best_loss=torch.minimum(best_loss,current)\n                    history.append({\'step\':step,\'loss\':float(best_loss.mean())})\n    if initial_phase.is_cuda:\n        torch.cuda.synchronize(initial_phase.device)\n    phase=torch.atan2(best.sin(),best.cos()).detach()\n    report={\'method\':\'per-input Adam on shared phase, with straight-through 8-bit quantization\',\n            \'optimizer_steps\':steps,\'learning_rate\':lr,\'best_step_per_input\':best_step.cpu().tolist(),\n            \'seconds\':time.perf_counter()-started,\'history\':history,\n            \'warning\':\'This is a neural initializer plus iterative phase solver, not a single network forward pass.\'}\n    return phase,report\n', 'doe/results.py': '"""Quantized deliverables and independently computable image quality metrics."""\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom skimage.metrics import structural_similarity\nimport torch\n\nfrom .data import load_quartet\n\n\ndef encode_phase(phase):\n    phase = np.asarray(phase, dtype=np.float64)\n    if not np.isfinite(phase).all() or (np.abs(phase) > math.pi + 1e-6).any():\n        raise ValueError("phase must be finite and in [-pi, pi]")\n    return np.rint((np.clip(phase, -math.pi, math.pi) + math.pi) * (255 / (2 * math.pi))).astype(np.uint8)\n\n\ndef decode_phase(image):\n    image = np.asarray(image)\n    if image.dtype != np.uint8:\n        raise ValueError("phase PNG samples must be uint8")\n    return image.astype(np.float64) * (2 * math.pi / 255) - math.pi\n\n\ndef amplitude_png(amplitude):\n    amplitude = np.asarray(amplitude, dtype=np.float64)\n    if not np.isfinite(amplitude).all() or (amplitude < 0).any():\n        raise ValueError("amplitude must be finite and nonnegative")\n    peak = float(amplitude.max())\n    return np.rint(255 * amplitude / max(peak, 1e-30)).astype(np.uint8), peak\n\n\ndef metrics(target, reconstructed):\n    """Metrics of actual 8-bit images. No fitting, denoising or target-dependent gain."""\n    a = np.asarray(target, dtype=np.float64) / 255\n    b = np.asarray(reconstructed, dtype=np.float64) / 255\n    if a.shape != b.shape or a.ndim != 2:\n        raise ValueError("metrics require two equally-sized grayscale images")\n    if min(a.shape) < 7:\n        raise ValueError("SSIM requires dimensions >= 7")\n    ac, bc = a - a.mean(), b - b.mean()\n    denominator = float(np.linalg.norm(ac) * np.linalg.norm(bc))\n    correlation = float(np.clip(np.sum(ac * bc) / denominator, -1, 1)) if denominator > 1e-14 else None\n    return {"pearson": correlation, "ssim": float(structural_similarity(a, b, data_range=1.0, win_size=7))}\n\n\ndef phase_metrics(optics, phase, targets):\n    """Evaluate after phase quantization and amplitude PNG quantization."""\n    encoded = encode_phase(phase.detach().cpu().numpy())\n    decoded = torch.as_tensor(decode_phase(encoded), device=phase.device, dtype=phase.dtype)\n    with torch.no_grad():\n        amplitudes = np.abs(optics.physical(decoded).cpu().numpy())\n    source = np.rint(targets.detach().cpu().numpy() * 255).astype(np.uint8)\n    records = []\n    for b in range(len(source)):\n        planes = [metrics(source[b, i], amplitude_png(amplitudes[b, i])[0]) for i in range(4)]\n        correlations = [v["pearson"] for v in planes if v["pearson"] is not None]\n        records.append({"pearson": float(np.mean(correlations)) if correlations else None,\n                        "ssim": float(np.mean([v["ssim"] for v in planes])), "planes": planes})\n    return records\n\n\ndef export_result(output, optics, phase, image_paths, metadata=None):\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite existing result: {output}")\n    targets = load_quartet(image_paths)\n    if phase.ndim != 3 or phase.shape[0] != 1:\n        raise ValueError("export expects one phase map [1,128,128]")\n    output.mkdir(parents=True, exist_ok=True)\n    encoded = encode_phase(phase[0].detach().cpu().numpy())\n    phase_path = output / "doe_phase.png"\n    Image.fromarray(encoded).save(phase_path)\n    # Intentionally read the saved file back: all reported reconstructions use its quantized phase.\n    with Image.open(phase_path) as saved:\n        decoded = decode_phase(np.array(saved))\n    decoded_tensor = torch.as_tensor(decoded, device=phase.device, dtype=phase.dtype)[None]\n    with torch.no_grad():\n        physical = optics.physical(decoded_tensor)[0].cpu().numpy()\n    amplitudes = np.abs(physical)\n    np.save(output / "phase_rad.npy", decoded)\n    np.save(output / "reconstructed_amplitudes.npy", amplitudes)\n    np.save(output / "reconstructed_complex_fields.npy", physical)\n    input_arrays = np.rint(targets[0].numpy() * 255).astype(np.uint8)\n    rows = []\n    images = []\n    for i, z in enumerate(optics.config.distances_m):\n        image, peak = amplitude_png(amplitudes[i])\n        images.append(image)\n        Image.fromarray(input_arrays[i]).save(output / f"target_{i + 1}.png")\n        Image.fromarray(image).save(output / f"reconstruction_{i + 1}.png")\n        rows.append({"plane": i + 1, "distance_mm": z * 1000,\n                     "output_pixel_pitch_um": optics.config.output_pixel_pitches_m[i] * 1e6,\n                     "amplitude_peak_before_display_scaling": peak,\n                     **metrics(input_arrays[i], image)})\n    report = {\n        "optics": optics.config.to_dict(), "sampling": optics.config.sampling_report(),\n        "metadata": metadata or {}, "planes": rows,\n        "mean_pearson": float(np.mean([row["pearson"] for row in rows if row["pearson"] is not None]))\n                        if any(row["pearson"] is not None for row in rows) else None,\n        "mean_ssim": float(np.mean([row["ssim"] for row in rows])),\n        "phase_encoding": "uint8 0 -> -pi; 255 -> +pi; round to nearest",\n        "reconstruction": "single-FFT Fresnel, reconstructed from the saved quantized DOE PNG",\n        "png_scaling": "amplitude (not intensity); each plane divided by its own maximum; no background subtraction or fitted gain",\n        "ssim": {"implementation": "skimage.metrics.structural_similarity", "win_size": 7, "data_range": 1.0},\n        "pearson_constant_image": "null (undefined), never a fabricated zero or one",\n        "doe_sha256": hashlib.sha256(phase_path.read_bytes()).hexdigest(),\n        "inputs": [{"name": Path(path).name, "sha256": hashlib.sha256(Path(path).read_bytes()).hexdigest()} for path in image_paths],\n    }\n    (output / "metrics.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n    with (output / "metrics.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    plot_result(output / "comparison.png", input_arrays, np.stack(images), encoded, rows)\n    return report\n\n\ndef plot_result(path, targets, reconstructions, phase, rows):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    fig, axes = plt.subplots(4, 3, figsize=(10, 11), constrained_layout=True)\n    for i, row in enumerate(rows):\n        axes[i, 0].imshow(targets[i], cmap="gray", vmin=0, vmax=255)\n        axes[i, 0].set_title(f"Target {i+1}, z={row[\'distance_mm\']:.1f} mm")\n        axes[i, 1].imshow(reconstructions[i], cmap="gray", vmin=0, vmax=255)\n        corr = row["pearson"]\n        axes[i, 1].set_title(f"Reconstruction |U|; r={corr:.3f}" if corr is not None else "Reconstruction; r undefined")\n        axes[i, 2].imshow(np.abs(targets[i].astype(float) - reconstructions[i]), cmap="magma", vmin=0, vmax=255)\n        axes[i, 2].set_title(f"Absolute difference; SSIM={row[\'ssim\']:.3f}")\n        for ax in axes[i]:\n            ax.axis("off")\n    fig.savefig(path, dpi=140)\n    plt.close(fig)\n    fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)\n    display = ax.imshow(decode_phase(phase), cmap="twilight", vmin=-math.pi, vmax=math.pi)\n    ax.set_title("Quantized DOE phase (radians)")\n    fig.colorbar(display, ax=ax)\n    fig.savefig(Path(path).with_name("phase_preview.png"), dpi=140)\n    plt.close(fig)\n', 'tests/test_camera_optics.py': "import math\n\nimport numpy as np\nimport torch\n\nfrom doe.camera_optics import CameraConfig,CameraFresnel\nfrom doe.optics import Fresnel,OpticsConfig\n\n\ndef test_padding_one_matches_original_model():\n    torch.manual_seed(60)\n    a=Fresnel(OpticsConfig(size=16),dtype=torch.float64)\n    b=CameraFresnel(CameraConfig(size=16,fft_padding=1),dtype=torch.float64)\n    phase=torch.rand(2,16,16,dtype=torch.float64)\n    torch.testing.assert_close(a.physical(phase),b.physical(phase),rtol=0,atol=0)\n\n\ndef test_finer_camera_matches_independent_direct_diffraction_sum():\n    cfg=CameraConfig(size=8,fft_padding=2)\n    optics=CameraFresnel(cfg,dtype=torch.float64)\n    phase=np.random.default_rng(61).uniform(-math.pi,math.pi,(8,8))\n    actual=optics.physical(torch.tensor(phase)[None])[0].numpy()\n    axis=(np.arange(8)-4)*cfg.pixel_pitch_m\n    u,v=np.meshgrid(axis,axis)\n    for i,z in enumerate(cfg.distances_m):\n        camera=(np.arange(8)-4)*cfg.output_pixel_pitches_m[i]\n        x,y=np.meshgrid(camera,camera)\n        r2=(x[:,:,None,None]-u)**2+(y[:,:,None,None]-v)**2\n        direct=(np.exp(1j*phase)*np.exp(1j*math.pi*r2/(cfg.wavelength_m*z))).sum((-2,-1))\n        direct*=cfg.pixel_pitch_m**2*np.exp(2j*math.pi*z/cfg.wavelength_m)/(1j*cfg.wavelength_m*z)\n        np.testing.assert_allclose(actual[i],direct,rtol=1e-8,atol=1e-8)\n\n\ndef test_camera_adjoint_and_full_energy_are_correct():\n    torch.manual_seed(62)\n    cfg=CameraConfig(size=16,fft_padding=2)\n    optics=CameraFresnel(cfg,dtype=torch.float64)\n    source=torch.randn(2,16,16,dtype=torch.complex128)\n    observed=torch.randn(2,4,16,16,dtype=torch.complex128)\n    lhs=(optics.field(source).conj()*observed).sum()\n    rhs=(source[:,None].conj()*optics.adjoint(observed)).sum()\n    torch.testing.assert_close(lhs,rhs)\n    phase=torch.rand(2,16,16,dtype=torch.float64)\n    field=optics.full_physical(phase)\n    pitches=torch.tensor(cfg.output_pixel_pitches_m,dtype=torch.float64)\n    power=field.abs().square().sum((-2,-1))*pitches.square()\n    torch.testing.assert_close(power,torch.full_like(power,cfg.size**2*cfg.pixel_pitch_m**2))\n    fraction=optics.camera_energy_fraction(phase)\n    assert ((fraction>0)&(fraction<=1)).all()\n    assert cfg.sampling_report()['fft_size']==32\n", 'tests/test_deliverables.py': 'import json\nimport math\n\nimport numpy as np\nfrom PIL import Image\nimport pytest\nimport torch\n\nfrom doe.data import generate_dataset, load_quartet, QuartetDataset\nfrom doe.model import FresnelNet\nfrom doe.optics import Fresnel\nfrom doe.results import amplitude_png, decode_phase, encode_phase, export_result, metrics\n\n\ndef test_phase_endpoints_and_quantization_error():\n    phi = np.linspace(-math.pi, math.pi, 1001)\n    encoded = encode_phase(phi)\n    assert encoded.dtype == np.uint8\n    assert encoded[0] == 0 and encoded[-1] == 255\n    np.testing.assert_allclose(decode_phase(np.array([0, 255], dtype=np.uint8)), [-math.pi, math.pi])\n    assert np.max(np.abs(decode_phase(encoded) - phi)) <= math.pi / 255 + 1e-12\n    with pytest.raises(ValueError):\n        encode_phase(np.array([math.nan]))\n\n\ndef test_pearson_ssim_known_cases():\n    a = np.arange(128 * 128).reshape(128, 128).astype(np.uint8)\n    assert metrics(a, a) == {"pearson": 1.0, "ssim": 1.0}\n    assert metrics(a, 255 - a)["pearson"] == pytest.approx(-1.0)\n    assert metrics(np.zeros_like(a), a)["pearson"] is None\n    assert metrics(np.zeros_like(a), np.zeros_like(a))["ssim"] == 1.0\n\n\ndef test_export_reconstructs_saved_quantized_doe_and_metrics(tmp_path):\n    generate_dataset(tmp_path / "data", train=1, validation=1, test=1)\n    inputs = [tmp_path / "data/test/sample_0000" / f"plane_{i}.png" for i in range(1, 5)]\n    optics = Fresnel()\n    torch.manual_seed(11)\n    phase = torch.rand(1, 128, 128) * 2 * math.pi - math.pi\n    out = tmp_path / "result"\n    report = export_result(out, optics, phase, inputs)\n    with Image.open(out / "doe_phase.png") as image:\n        assert image.mode == "L" and image.size == (128, 128)\n        decoded = decode_phase(np.array(image))\n    expected = np.abs(optics.physical(torch.tensor(decoded, dtype=torch.float32)[None])[0].numpy())\n    np.testing.assert_allclose(np.load(out / "reconstructed_amplitudes.npy"), expected)\n    for i in range(4):\n        with Image.open(out / f"reconstruction_{i + 1}.png") as image:\n            assert image.mode == "L" and image.size == (128, 128)\n            png = np.array(image)\n        np.testing.assert_array_equal(png, amplitude_png(expected[i])[0])\n        target = np.array(Image.open(inputs[i]))\n        assert report["planes"][i]["ssim"] == metrics(target, png)["ssim"]\n    assert len(report["planes"]) == 4\n    json.loads((out / "metrics.json").read_text())\n    with pytest.raises(FileExistsError):\n        export_result(out, optics, phase, inputs)\n\n\ndef test_strict_inputs_and_separate_splits(tmp_path):\n    manifest = generate_dataset(tmp_path / "data", train=2, validation=2, test=2)\n    seeds = [s for split in manifest["splits"].values() for row in split for s in row["image_seeds"]]\n    assert len(seeds) == len(set(seeds))\n    assert QuartetDataset(tmp_path / "data", "train")[0].shape == (4, 128, 128)\n    paths = [tmp_path / "data/test/sample_0000" / f"plane_{i}.png" for i in range(1, 5)]\n    Image.new("RGB", (128, 128)).save(tmp_path / "rgb.png")\n    with pytest.raises(ValueError, match="grayscale"):\n        load_quartet([tmp_path / "rgb.png", *paths[1:]])\n    with pytest.raises(ValueError, match="exactly four"):\n        load_quartet(paths[:3])\n    with pytest.raises(FileExistsError):\n        generate_dataset(tmp_path / "data", train=1, validation=1, test=1)\n', 'tests/test_finetuning.py': '"""Prevent silent training from scratch and source leakage during fine-tuning."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport runpy\nfrom zipfile import ZipFile\n\nimport numpy as np\nfrom PIL import Image\nimport pytest\nimport torch\n\nfrom doe.data import generate_dataset\nfrom doe.engine import TrainConfig, train\nfrom doe.model import FresnelNet\nfrom doe.optics import OpticsConfig\n\n\n@pytest.mark.parametrize("objective",["physics","perceptual"])\ndef test_finetuning_starts_from_the_supplied_weights(tmp_path,objective):\n    torch.manual_seed(13)\n    model = FresnelNet(stages=1, width=4)\n    with torch.no_grad():\n        model.relax_logits.fill_(0.71)\n        model.corrector[-1].weight.fill_(0.013)\n    checkpoint = tmp_path / "parent.pt"\n    torch.save({"format_version": 1, "model": model.state_dict(), "epoch": 7,\n                "optics": model.optics.config.to_dict(), "architecture": {"stages": 1, "width": 4}}, checkpoint)\n    generate_dataset(tmp_path / "data", train=1, validation=1, test=1)\n    report = train(tmp_path / "data", tmp_path / "run", TrainConfig(\n        epochs=1, batch_size=1, stages=1, width=4, learning_rate=0.0001,\n        initial_checkpoint=str(checkpoint), device="cpu",objective=objective))\n    initial = torch.load(tmp_path / "run/initial.pt", weights_only=True)\n    for name, tensor in model.state_dict().items():\n        torch.testing.assert_close(initial["model"][name], tensor, rtol=0, atol=0)\n    assert report["parent_checkpoint"]["sha256"] == hashlib.sha256(checkpoint.read_bytes()).hexdigest()\n    assert report["parent_checkpoint"]["epoch"] == 7\n    assert report["parameter_l2_change_last_vs_initial"] > 0\n\n\ndef test_texture_source_families_do_not_cross_splits(tmp_path):\n    archive = tmp_path / "textures.zip"\n    rng = np.random.default_rng(15)\n    with ZipFile(archive, "w") as zipped:\n        zipped.writestr("License.txt", "Test fixture, generated in test")\n        for family in ("Craters", "Super Perlin", "Streak", "Cracks"):\n            for i in range(4):\n                array = rng.integers(0, 256, (128, 128), dtype=np.uint8)\n                stream = io.BytesIO()\n                Image.fromarray(array).convert("RGB").save(stream, format="PNG")\n                zipped.writestr(f"128x128/{family}/{family} {i}.png", stream.getvalue())\n    scope = runpy.run_path(str(Path(__file__).resolve().parents[1] / "scripts/prepare_texture_dataset.py"))\n    manifest = scope["prepare"](archive, tmp_path / "data", train=2, validation=1, test=1)\n    families = {k: set(v) for k, v in manifest["families"].items()}\n    for a in families:\n        for b in families:\n            if a != b:\n                assert families[a].isdisjoint(families[b])\n    assert manifest["image_counts"]["reserved"] == 4\n    all_samples = [row for split in manifest["splits"].values() for row in split]\n    assert not any(im["family"] == "Cracks" for row in all_samples for im in row["images"])\n\n\ndef test_chirp_sampling_diagnostics_distinguish_input_and_output():\n    report = OpticsConfig().sampling_report()\n    assert report["critical_distance_m"] == pytest.approx(0.02406015037594)\n    assert all(plane["input_chirp_passes"] for plane in report["planes"])\n    assert all(plane["output_chirp_nyquist_ratio"] > 1 for plane in report["planes"])\n    for plane in report["planes"]:\n        assert plane["input_chirp_nyquist_ratio"] * plane["output_chirp_nyquist_ratio"] == pytest.approx(1)\n', 'tests/test_optics.py': '"""Independent quadrature and gradient checks, beyond testing FFT against itself."""\nimport math\n\nimport numpy as np\nimport pytest\nimport torch\n\nfrom doe.model import FresnelNet, physics_loss\nfrom doe.optics import Fresnel, OpticsConfig\n\n\ndef test_fresnel_matches_direct_diffraction_integral():\n    config = OpticsConfig(size=8)\n    optics = Fresnel(config, dtype=torch.float64)\n    rng = np.random.default_rng(17)\n    phase = rng.uniform(-math.pi, math.pi, (8, 8))\n    calculated = optics.physical(torch.tensor(phase)[None])[0].numpy()\n    axis = (np.arange(8) - 4) * config.pixel_pitch_m\n    u, v = np.meshgrid(axis, axis)\n    for i, z in enumerate(config.distances_m):\n        out_axis = (np.arange(8) - 4) * config.output_pixel_pitches_m[i]\n        x, y = np.meshgrid(out_axis, out_axis)\n        distance_squared = (x[:, :, None, None] - u)**2 + (y[:, :, None, None] - v)**2\n        integrand = np.exp(1j * phase) * np.exp(1j * math.pi * distance_squared / (config.wavelength_m * z))\n        direct = (integrand.sum(axis=(-2, -1)) * config.pixel_pitch_m**2\n                  * np.exp(2j * math.pi * z / config.wavelength_m) / (1j * config.wavelength_m * z))\n        np.testing.assert_allclose(calculated[i], direct, rtol=1e-8, atol=1e-8)\n\n\ndef test_adjoint_and_physical_power_with_different_output_pixel_sizes():\n    torch.manual_seed(7)\n    config = OpticsConfig(size=16)\n    optics = Fresnel(config, dtype=torch.float64)\n    field = torch.randn(2, 16, 16, dtype=torch.complex128)\n    propagated = optics.field(field)\n    arbitrary = torch.randn_like(propagated)\n    lhs = (propagated.conj() * arbitrary).sum()\n    rhs = (field[:, None].conj() * optics.adjoint(arbitrary)).sum()\n    torch.testing.assert_close(lhs, rhs)\n    phase = torch.rand(2, 16, 16, dtype=torch.float64)\n    power = optics.physical(phase).abs().square().sum(dim=(-2, -1))\n    pitches = torch.tensor(config.output_pixel_pitches_m, dtype=torch.float64)\n    np.testing.assert_allclose((power * pitches.square()).numpy(), config.size**2 * config.pixel_pitch_m**2, rtol=1e-9)\n\n\ndef test_phase_gradient_matches_finite_difference():\n    torch.manual_seed(20)\n    optics = Fresnel(OpticsConfig(size=8), dtype=torch.float64)\n    phase = torch.randn(1, 8, 8, dtype=torch.float64, requires_grad=True)\n    targets = torch.rand(1, 4, 8, 8, dtype=torch.float64) + 0.1\n    physics_loss(optics, phase, targets).backward()\n    index = (0, 2, 3)\n    step = 1e-5\n    plus, minus = phase.detach().clone(), phase.detach().clone()\n    plus[index] += step\n    minus[index] -= step\n    numerical = (physics_loss(optics, plus, targets) - physics_loss(optics, minus, targets)) / (2 * step)\n    torch.testing.assert_close(phase.grad[index], numerical, rtol=1e-5, atol=1e-7)\n\n\ndef test_neural_weights_receive_finite_nonzero_gradients():\n    torch.manual_seed(5)\n    model = FresnelNet(OpticsConfig(size=16), stages=2, width=4)\n    targets = torch.rand(2, 4, 16, 16) + 0.1\n    loss = physics_loss(model.optics, model(targets), targets)\n    loss.backward()\n    gradient = model.corrector[-1].weight.grad\n    assert torch.isfinite(gradient).all()\n    assert gradient.abs().max() > 0\n    assert torch.isfinite(model.plane_logits.grad).all()\n    with pytest.raises(ValueError, match="black"):\n        model(torch.zeros_like(targets))\n', 'tests/test_quality.py': "import math\nfrom pathlib import Path\nimport runpy\n\nimport numpy as np\nimport torch\nfrom skimage.metrics import structural_similarity\n\nfrom doe.quality import pearson_per_plane, quantized_phase_ste, ssim_per_plane,robust_quality_loss_per_item,_SoftPeakGradient\nfrom doe.results import decode_phase, encode_phase\nfrom doe.optics import Fresnel,OpticsConfig\nfrom doe.refinement import refine_phase\n\n\ndef test_differentiable_quality_matches_reported_definitions():\n    rng=np.random.default_rng(44)\n    a=rng.uniform(0,1,(2,4,16,16))\n    b=rng.uniform(0,1,(2,4,16,16))\n    ssim=ssim_per_plane(torch.tensor(a),torch.tensor(b)).numpy()\n    maps=ssim_per_plane(torch.tensor(a),torch.tensor(b),return_map=True)\n    np.testing.assert_allclose(maps.mean(dim=(-2,-1)).numpy(),ssim,atol=1e-12)\n    corr=pearson_per_plane(torch.tensor(a),torch.tensor(b)).numpy()\n    for i in range(2):\n        for j in range(4):\n            np.testing.assert_allclose(ssim[i,j],structural_similarity(a[i,j],b[i,j],data_range=1.0,win_size=7),atol=1e-12)\n            np.testing.assert_allclose(corr[i,j],np.corrcoef(a[i,j].ravel(),b[i,j].ravel())[0,1],atol=1e-12)\n\n\ndef test_ste_quantization_matches_saved_phase_encoding():\n    phi=torch.linspace(-math.pi,math.pi,1001,dtype=torch.float64,requires_grad=True)\n    # Interior values avoid the equivalent -pi/+pi endpoint wrapping ambiguity.\n    phi=phi[1:-1].detach().requires_grad_(True)\n    actual=quantized_phase_ste(phi)\n    expected=decode_phase(encode_phase(phi.detach().numpy()))\n    np.testing.assert_allclose(actual.detach().numpy(),expected,atol=1e-14)\n    actual.sum().backward()\n    torch.testing.assert_close(phi.grad,torch.ones_like(phi))\n\n\ndef test_ssim_gradient_matches_finite_difference():\n    torch.manual_seed(10)\n    a=torch.rand(1,4,9,9,dtype=torch.float64,requires_grad=True)\n    b=torch.rand_like(a)\n    ssim_per_plane(a,b).mean().backward()\n    idx=(0,1,4,5)\n    eps=1e-5\n    plus,minus=a.detach().clone(),a.detach().clone()\n    plus[idx]+=eps;minus[idx]-=eps\n    fd=(ssim_per_plane(plus,b).mean()-ssim_per_plane(minus,b).mean())/(2*eps)\n    torch.testing.assert_close(a.grad[idx],fd,atol=1e-8,rtol=1e-6)\n\n\ndef test_refinement_keeps_one_shared_phase_and_works_under_no_grad():\n    torch.manual_seed(9)\n    optics=Fresnel(OpticsConfig(size=16))\n    initial=torch.rand(1,16,16)*2*math.pi-math.pi\n    targets=torch.rand(1,4,16,16)+0.01\n    with torch.no_grad():\n        phase,report=refine_phase(optics,initial,targets,steps=3,check_every=1)\n    assert phase.shape==initial.shape and not phase.requires_grad\n    assert torch.isfinite(phase).all() and phase.abs().max()<=math.pi+1e-6\n    assert report['optimizer_steps']==3\n    assert report['history'][-1]['loss']<=report['history'][0]['loss']\n    assert len(report['best_step_per_input'])==1\n\n\ndef test_independent_plane_control_uses_the_same_physics_per_plane():\n    scope=runpy.run_path(str(Path(__file__).resolve().parents[1]/'scripts/probe_quality.py'))\n    optics=Fresnel(OpticsConfig(size=16))\n    control=scope['IndependentPlaneOptics'](optics)\n    phase=torch.rand(2,4,16,16)*2*math.pi-math.pi\n    actual=control.physical(phase)\n    for plane in range(4):\n        expected=optics.physical(phase[:,plane])[:,plane]\n        torch.testing.assert_close(actual[:,plane],expected)\n\n\ndef test_robust_quality_objective_has_finite_phase_gradients():\n    torch.manual_seed(51)\n    optics=Fresnel(OpticsConfig(size=16))\n    phase=torch.rand(1,16,16,requires_grad=True)\n    target=torch.rand(1,4,16,16)*.05+.7\n    loss=robust_quality_loss_per_item(optics,phase,target).mean()\n    loss.backward()\n    assert torch.isfinite(loss) and torch.isfinite(phase.grad).all() and phase.grad.abs().max()>0\n\n\ndef test_soft_peak_keeps_exact_forward_maximum_and_only_changes_gradient():\n    value=torch.tensor([[[[.1,.3],[.5,.7]]]],requires_grad=True)\n    peak=_SoftPeakGradient.apply(value,.2)\n    torch.testing.assert_close(peak,value.amax(dim=(-2,-1),keepdim=True),rtol=0,atol=0)\n    peak.sum().backward()\n    assert (value.grad>0).all()\n    torch.testing.assert_close(value.grad.sum(),torch.tensor(1.))\n", 'tests/test_wavefront.py': "import hashlib\n\nimport torch\n\nfrom doe.data import generate_dataset\nfrom doe.engine import TrainConfig,load_model,train\nfrom doe.model import FresnelNet,WavefrontFresnelNet,AdaptiveWavefrontNet,physics_loss\nfrom doe.optics import OpticsConfig\n\n\ndef test_wavefront_initializer_receives_nonzero_gradients():\n    torch.manual_seed(41)\n    model=WavefrontFresnelNet(OpticsConfig(size=16),stages=2,width=4,initializer_width=4)\n    targets=torch.rand(2,4,16,16)+.1\n    phase=model(targets)\n    assert phase.shape==(2,16,16)\n    physics_loss(model.optics,phase,targets).backward()\n    gradient=model.phase_initializer['head'].weight.grad\n    assert torch.isfinite(gradient).all() and gradient.abs().max()>0\n    assert phase.abs().max()<=torch.pi+1e-6\n\n\ndef test_wavefront_checkpoint_round_trip(tmp_path):\n    torch.manual_seed(42)\n    model=WavefrontFresnelNet(OpticsConfig(size=16),stages=1,width=4,initializer_width=4).eval()\n    target=torch.rand(1,4,16,16)+.1\n    path=tmp_path/'wavefront.pt'\n    torch.save({'format_version':1,'architecture':model.architecture(),'optics':model.optics.config.to_dict(),\n                'model':model.state_dict(),'epoch':0},path)\n    loaded,_=load_model(path,torch.device('cpu'))\n    assert isinstance(loaded,WavefrontFresnelNet)\n    with torch.no_grad():torch.testing.assert_close(loaded(target),model(target),rtol=0,atol=0)\n\n\ndef test_fixed_checkpoint_can_upgrade_without_losing_existing_weights(tmp_path):\n    torch.manual_seed(43)\n    base=FresnelNet(stages=1,width=4)\n    path=tmp_path/'parent.pt'\n    torch.save({'format_version':1,'architecture':{'stages':1,'width':4},'optics':base.optics.config.to_dict(),\n                'model':base.state_dict(),'epoch':3},path)\n    generate_dataset(tmp_path/'data',train=1,validation=1,test=1)\n    report=train(tmp_path/'data',tmp_path/'run',TrainConfig(epochs=1,batch_size=1,stages=1,width=4,\n        initializer_width=4,architecture='wavefront',objective='perceptual',initial_checkpoint=str(path),device='cpu'))\n    initial=torch.load(tmp_path/'run/initial.pt',weights_only=True)\n    for name,value in base.state_dict().items():\n        torch.testing.assert_close(initial['model'][name],value,rtol=0,atol=0)\n    assert initial['architecture']['kind']=='wavefront'\n    assert report['parent_checkpoint']['sha256']==hashlib.sha256(path.read_bytes()).hexdigest()\n    assert report['parameter_l2_change_last_vs_initial']>0\n\n\ndef test_adaptive_projection_and_deeper_initializer_have_gradients():\n    torch.manual_seed(48)\n    model=AdaptiveWavefrontNet(OpticsConfig(size=16),stages=2,width=4,initializer_width=4,initializer_depth=4)\n    target=torch.rand(2,4,16,16)+.1\n    phase=model(target)\n    physics_loss(model.optics,phase,target).backward()\n    grad=model.amplitude_modulators[0][-1].weight.grad\n    assert torch.isfinite(grad).all() and grad.abs().max()>0\n    # The phase head is zero-initialized. After its first update, the gated\n    # deeper branch must receive a signal without perturbing the warm start.\n    optimizer=torch.optim.Adam(model.parameters(),lr=.001)\n    optimizer.step();optimizer.zero_grad(set_to_none=True)\n    physics_loss(model.optics,model(target),target).backward()\n    assert torch.isfinite(model.deep_scale.grad) and model.deep_scale.grad.abs()>0\n    assert phase.shape==(2,16,16)\n", 'scripts/build_notebook.py': '"""Build a standalone, inspectable Kaggle notebook with embedded project source."""\nimport json\nfrom pathlib import Path\n\nimport nbformat as nbf\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef build():\n    files = [*sorted((ROOT / "doe").glob("*.py")), *sorted((ROOT / "tests").glob("*.py")),\n             *sorted((ROOT / "scripts").glob("*.py")), *sorted((ROOT / "docs").glob("*.md")),\n             ROOT / "pyproject.toml", ROOT / "requirements.txt", ROOT / "requirements-dev.txt", ROOT / "README.md"]\n    sources = {str(path.relative_to(ROOT)): path.read_text() for path in files}\n    cells = [nbf.v4.new_markdown_cell(r"""# Нейросетевой синтез четырёхплоскостного ДОЭ\n\nЗадание хакатона: 4 одноканальных PNG 128×128 → фазовый ДОЭ 128×128 →\n4 реконструкции методом Френеля; λ=532 нм, пиксель ПВМС 10 мкм,\nz=[24.1,25.3,26.5,27.7] мм. Экспорт: 8-битные PNG, Pearson и SSIM.\n\n**Run All** запускает обучение, тестовую оценку и экспорт. Включите\nSettings → Accelerator → GPU T4 ×2 при доступной квоте; код также работает на CPU.\nИнтернет и Kaggle API-токен внутри notebook не требуются.\n\nВ документах нет четырёх исходных объектных PNG: по умолчанию используется явно\nобозначенный процедурный демонстрационный набор. Для своих данных укажите четыре\nпути в `INPUT_IMAGES` ниже. Их порядок соответствует возрастающему z.\n\nДля дообучения на текстурах загрузите ZIP и исходные веса как Kaggle Input,\nзаполните `TEXTURE_ZIP` и `INITIAL_CHECKPOINT`. Тогда notebook разделит данные\nпо семействам, исключит Cracks/Gabor/Perlin/Voronoi из обучения и сформирует\nPDF/CSV-отчёт с результатами до и после.\n\nМетод — фиксированная развёрнутая нейросеть с оптическими проекциями и обучаемыми\nCNN-коррекциями. Обучение без эталонных голограмм; при инференсе нет шагов оптимизатора.\nТестовые изображения не участвуют в обучении и выборе checkpoint.\n"""), nbf.v4.new_code_cell(\n        "# Embedded source: inspectable, no Git clone or external download.\\n"\n        "from pathlib import Path\\nimport json, os, sys\\n"\n        "BASE = Path(\'/kaggle/working\') if Path(\'/kaggle/working\').exists() else Path.cwd() / \'notebook-workdir\'\\n"\n        "PROJECT = BASE / \'nuclear-hack-source\'\\nPROJECT.mkdir(parents=True, exist_ok=True)\\n"\n        + "SOURCE_FILES = " + repr(sources) + "\\n"\n        "for name, source in SOURCE_FILES.items():\\n"\n        "    path = PROJECT / name\\n"\n        "    path.parent.mkdir(parents=True, exist_ok=True)\\n"\n        "    if path.exists() and path.read_text() != source:\\n"\n        "        raise RuntimeError(f\'Existing source differs: {path}; preserve it and use a fresh work directory\')\\n"\n        "    path.write_text(source)\\n"\n        "sys.path.insert(0, str(PROJECT))\\n"\n        "import torch, numpy as np, PIL, skimage, matplotlib\\n"\n        "print({\'torch\': torch.__version__, \'cuda\': torch.cuda.is_available(), \'numpy\': np.__version__})\\n"\n        "if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))\\n"\n    ), nbf.v4.new_markdown_cell("""## Настройки\n\nЧетыре входных файла должны быть PNG, режим L, 8 бит, 128×128. Пустая плоскость\nнесовместима с фазовым ДОЭ при равномерном освещении и не принимается.\n`INPUT_IMAGES=[]` означает демонстрацию, а не выполнение на официальных данных.\nCPU использует меньший учебный набор; на GPU можно увеличить число эпох и примеров.\nПри повторном запуске создаётся новый каталог, предыдущие результаты сохраняются.\n"""), nbf.v4.new_code_cell("""from datetime import datetime, timezone\nfrom doe.engine import TrainConfig, train, evaluate, predict\nfrom doe.data import generate_dataset\nimport runpy\n\nINPUT_IMAGES = []  # [\'/kaggle/input/YOUR_DATA/image1.png\', ..., \'image4.png\']\nREFINE_STEPS = int(os.environ.get(\'DOE_REFINE_STEPS\', \'0\'))  # Optional, slower phase solver; raw network outputs remain separate\nMODEL_KIND = os.environ.get(\'DOE_MODEL_KIND\', \'inherit\')  # \'wavefront\' or \'adaptive\' upgrades a matching-width checkpoint\nTEXTURE_ZIP = os.environ.get(\'DOE_TEXTURE_ZIP\') or None  # \'/kaggle/input/YOUR_DATA/sbs...128x128.zip\'\nINITIAL_CHECKPOINT = os.environ.get(\'DOE_INITIAL_CHECKPOINT\') or None  # \'/kaggle/input/YOUR_MODEL/best.pt\'\nif TEXTURE_ZIP and not INITIAL_CHECKPOINT:\n    raise ValueError(\'Texture fine-tuning requires INITIAL_CHECKPOINT\')\nQUICK_CHECK = os.environ.get(\'DOE_QUICK_CHECK\') == \'1\'  # integration validation only\nGPU = torch.cuda.is_available()\nRUN = BASE / (\'doe-run-\' + datetime.now(timezone.utc).strftime(\'%Y%m%d-%H%M%S-%f\'))\nRUN.mkdir(parents=True)\nDATA = RUN / \'data\'\nTRAINING = RUN / \'training\'\nif QUICK_CHECK:\n    counts = (8, 4, 4)\n    config = TrainConfig(epochs=2, batch_size=2, stages=2, width=4)\nelse:\n    counts = (512, 64, 32) if GPU else (96, 16, 16)\n    config = TrainConfig(epochs=20 if GPU else 24, batch_size=8 if GPU else 4,\n                         stages=8 if GPU else 6, width=16 if GPU else 12)\nif TEXTURE_ZIP:\n    counts = (8, 4, 4) if QUICK_CHECK else (256, 32, 32)\n    config.epochs = 2 if QUICK_CHECK else 30\n    config.learning_rate = 0.001\nif INITIAL_CHECKPOINT:\n    parent = torch.load(INITIAL_CHECKPOINT, map_location=\'cpu\', weights_only=True)\n    config.stages = parent[\'architecture\'][\'stages\']\n    config.width = parent[\'architecture\'][\'width\']\n    config.architecture = parent[\'architecture\'].get(\'kind\',\'fixed\')\n    config.initializer_width = parent[\'architecture\'].get(\'initializer_width\',8)\n    config.initializer_depth = parent[\'architecture\'].get(\'initializer_depth\',3)\n    config.initial_checkpoint = INITIAL_CHECKPOINT\nif MODEL_KIND != \'inherit\':\n    if MODEL_KIND not in (\'fixed\',\'wavefront\',\'adaptive\'):\n        raise ValueError(\'Unknown model architecture\')\n    config.architecture = MODEL_KIND\n    if MODEL_KIND == \'adaptive\':\n        config.stages = max(config.stages, 12)\n        config.initializer_depth = 4\nprint(\'Run:\', RUN, \'\\\\nTraining:\', config)\n"""), nbf.v4.new_markdown_cell("""## Данные и обучение\n\nРазные seed для train/validation/test; сеть минимизирует ошибку амплитуд после\nдифференцируемого одно-FFT оператора Френеля. Лучшая модель выбирается по validation loss.\n`initial.pt` нужен для контроля того, чему научилась сеть. `best.pt` — для инференса.\n"""), nbf.v4.new_code_cell("""if TEXTURE_ZIP:\n    prepare = runpy.run_path(str(PROJECT / \'scripts/prepare_texture_dataset.py\'))[\'prepare\']\n    prepare(TEXTURE_ZIP, DATA, train=counts[0], validation=counts[1], test=counts[2])\nelse:\n    generate_dataset(DATA, train=counts[0], validation=counts[1], test=counts[2])\ntraining_report = train(DATA, TRAINING, config)\nassert training_report[\'parameter_l2_change_last_vs_initial\'] > 0\nprint(\'Best validation epoch:\', training_report[\'best_epoch\'])\n"""), nbf.v4.new_markdown_cell("""## Независимая тестовая оценка\n\nСравнение с той же сетью до обучения и многоплоскостным Герчбергом–Сэкстоном\nпри одинаковом и большем числе оптических шагов. Все методы оцениваются после\nквантования фазы и амплитуд в 8 бит. Время отражает batched throughput на текущем\nустройстве; ускорение относительно классического метода не предполагается заранее.\n"""), nbf.v4.new_code_cell("""benchmark = evaluate(DATA, TRAINING / \'best.pt\', RUN / \'benchmark.json\',\n                     batch_size=config.batch_size, gs_steps=12 if QUICK_CHECK else 80)\nprint(json.dumps(benchmark[\'summary\'], indent=2))\nif INITIAL_CHECKPOINT:\n    before_benchmark = evaluate(DATA, INITIAL_CHECKPOINT, RUN / \'benchmark-before.json\',\n                                batch_size=config.batch_size, gs_steps=12 if QUICK_CHECK else 80)\n"""), nbf.v4.new_markdown_cell("""## ДОЭ и восстановленные изображения\n\nСеть выдаёт фазу, кодируемую как `0 → −π`, `255 → +π` с округлением.\nРеконструкции рассчитываются заново **из сохранённого PNG ДОЭ**. Сохраняется модуль\nкомплексного поля, не квадрат модуля. Для отображения каждый модуль делится на свой\nмаксимум; без вычитания фона и без подгонки к целевому изображению.\nМетрики считаются по фактическим сохранённым 8-битным PNG. В `.npy` сохранены также\nмодули до масштабирования и комплексные поля. Шаг выходной сетки зависит от z:\nΔx_out = λz/(NΔu); изображения разных плоскостей задаются на соответствующих сетках.\n"""), nbf.v4.new_code_cell("""if INPUT_IMAGES:\n    if len(INPUT_IMAGES) != 4:\n        raise ValueError(\'Specify exactly four paths in increasing distance order\')\n    input_paths = [Path(p) for p in INPUT_IMAGES]\n    data_source = \'user-provided quartet\'\nelif TEXTURE_ZIP:\n    names = runpy.run_path(str(PROJECT / \'scripts/prepare_noise_quartet.py\'))[\'SELECTION\']\n    input_paths = [DATA / \'pool\' / Path(name).parent.name.replace(\' \', \'_\') / Path(name).name for name in names]\n    data_source = \'reserved texture families: Cracks, Gabor, Perlin, Voronoi; not used in training or validation\'\nelse:\n    input_paths = [DATA / \'test/sample_0000\' / f\'plane_{i}.png\' for i in range(1, 5)]\n    data_source = \'synthetic held-out demonstration, not official hackathon inputs\'\nresult = predict(TRAINING / \'best.pt\', input_paths, RUN / \'result\', source=data_source)\nprint(json.dumps(result[\'planes\'], indent=2))\nfrom IPython.display import display, Image as DisplayImage\ndisplay(DisplayImage(filename=str(RUN / \'result/comparison.png\')))\ndisplay(DisplayImage(filename=str(RUN / \'result/phase_preview.png\')))\nif REFINE_STEPS:\n    refined = predict(TRAINING / \'best.pt\', input_paths, RUN / \'result-refined\', source=data_source,\n                      refine_steps=REFINE_STEPS, refine_lr=0.08)\n    print(\'Neural initializer + phase optimizer (not raw network):\', refined[\'mean_pearson\'], refined[\'mean_ssim\'])\n    display(DisplayImage(filename=str(RUN / \'result-refined/comparison.png\')))\nif INITIAL_CHECKPOINT:\n    before_result = predict(INITIAL_CHECKPOINT, input_paths, RUN / \'result-before\', source=data_source)\nif TEXTURE_ZIP:\n    report = runpy.run_path(str(PROJECT / \'scripts/report_texture_finetuning.py\'))[\'generate\']\n    report(TRAINING / \'training.json\', DATA, RUN / \'benchmark-before.json\', RUN / \'benchmark.json\',\n           RUN / \'result-before\', RUN / \'result\', RUN / \'reports\')\n"""), nbf.v4.new_markdown_cell("""## Материалы для экспертов\n\nАрхив содержит код, полный сгенерированный набор, веса, историю обучения,\nтестовое сравнение, входные PNG, фазовый ДОЭ, четыре восстановленных PNG,\nметрики и исходные комплексные поля. Качество на официальной четвёрке необходимо\nизмерить отдельно после её предоставления. Это численное моделирование; оптическая\nпроверка на ПВМС здесь не выполняется.\n"""), nbf.v4.new_code_cell("""import shutil\nsubmission = RUN / \'submission\'\nsubmission.mkdir()\nfor name, source in SOURCE_FILES.items():\n    path = submission / \'source\' / name\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(source)\nfor name in (\'data\', \'training\', \'result\'):\n    shutil.copytree(RUN / name, submission / name)\nfor name in (\'result-before\', \'reports\', \'result-refined\'):\n    if (RUN / name).exists(): shutil.copytree(RUN / name, submission / name)\nif INITIAL_CHECKPOINT:\n    shutil.copy2(INITIAL_CHECKPOINT, submission / \'parent.pt\')\n    shutil.copy2(RUN / \'benchmark-before.json\', submission / \'benchmark-before.json\')\nshutil.copy2(RUN / \'benchmark.json\', submission / \'benchmark.json\')\n(submission / \'runtime.json\').write_text(json.dumps({\'torch\': torch.__version__, \'numpy\': np.__version__,\n    \'pillow\': PIL.__version__, \'skimage\': skimage.__version__, \'device\': benchmark[\'device\'],\n    \'input_source\': data_source, \'quick_check\': QUICK_CHECK}, indent=2))\narchive = shutil.make_archive(str(RUN / \'submission\'), \'zip\', submission)\nprint(\'Submission archive:\', archive)\nprint(\'Saved results:\', RUN / \'result\')\n""")]\n    for index, cell in enumerate(cells):\n        cell.id = f"section-{index:02d}"\n    notebook = nbf.v4.new_notebook(cells=cells, metadata={\n        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n        "language_info": {"name": "python"},\n        "kaggle": {"accelerator": "gpu", "isInternetEnabled": False},\n    })\n    nbf.validate(notebook)\n    out = ROOT / "notebooks/fresnel_doe_kaggle.ipynb"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    nbf.write(notebook, out)\n    metadata = {"id": "fiskarik/nuclear-hack-fresnel-doe", "title": "Nuclear Hack Fresnel DOE",\n                "code_file": out.name, "language": "python", "kernel_type": "notebook",\n                "is_private": True, "enable_gpu": True, "enable_internet": False,\n                "dataset_sources": [], "competition_sources": [], "kernel_sources": [], "model_sources": []}\n    (out.parent / "kernel-metadata.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(out)\n\n\nif __name__ == "__main__":\n    build()\n', 'scripts/compare_models.py': '"""Evaluate named checkpoints on exactly the same images and optical sampling grid."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom doe.data import QuartetDataset\nfrom doe.engine import load_model,setup,sync\nfrom doe.model import gerchberg_saxton\nfrom doe.results import phase_metrics\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data\',default=\'data/textures_v1\')\n    p.add_argument(\'--split\',default=\'test\',choices=(\'validation\',\'test\'))\n    p.add_argument(\'--checkpoint\',action=\'append\',required=True,help=\'LABEL=PATH\')\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--threads\',type=int,default=2)\n    p.add_argument(\'--batch-size\',type=int,default=4)\n    args=p.parse_args();out=Path(args.output)\n    if out.exists():raise FileExistsError(out)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=args.threads)\n    models={};checkpoints={};config=None\n    for value in args.checkpoint:\n        name,path=value.split(\'=\',1)\n        if name in models:raise ValueError(\'duplicate label\')\n        model,_=load_model(path,device)\n        spec=json.dumps(model.optics.config.to_dict(),sort_keys=True)\n        if config is not None and spec!=config:raise ValueError(\'cannot compare different optical sampling grids as a model gain\')\n        config=spec;models[name]=model\n        checkpoints[name]={\'path\':path,\'sha256\':hashlib.sha256(Path(path).read_bytes()).hexdigest(),\'architecture\':model.architecture()}\n    dataset=QuartetDataset(args.data,args.split)\n    loader=DataLoader(dataset,batch_size=args.batch_size,num_workers=0)\n    optics=next(iter(models.values())).optics\n    operations={name:model for name,model in models.items()}\n    operations[\'gs_80\']=lambda x:gerchberg_saxton(optics,x,80)\n    records={name:[] for name in operations};seconds={name:0.0 for name in operations}\n    with torch.no_grad():\n        warm=next(iter(loader)).to(device)\n        for fn in operations.values():fn(warm)\n        for targets in loader:\n            targets=targets.to(device)\n            for name,fn in operations.items():\n                sync(device);started=time.perf_counter();phase=fn(targets);sync(device)\n                seconds[name]+=time.perf_counter()-started\n                records[name].extend(phase_metrics(optics,phase,targets))\n    summary={name:{\'count\':len(values),\'pearson\':float(np.mean([v[\'pearson\'] for v in values])),\n                   \'ssim\':float(np.mean([v[\'ssim\'] for v in values])),\n                   \'seconds_per_quartet_batched\':seconds[name]/len(values)} for name,values in records.items()}\n    report={\'split\':args.split,\'sample_count\':len(dataset),\'threads\':args.threads,\'batch_size\':args.batch_size,\n            \'optics\':json.loads(config),\'checkpoints\':checkpoints,\'summary\':summary,\'per_sample\':records}\n    out.write_text(json.dumps(report,indent=2)+\'\\n\');print(json.dumps(summary,indent=2),flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'scripts/package_submission.py': '"""Package only declared project/data/run paths, never credentials or Git metadata."""\nimport argparse\nfrom pathlib import Path\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef package(data, run, output, holdout=None, quartet=None, quartet_results=None):\n    data, run, output = Path(data).resolve(), Path(run).resolve(), Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f"Refusing to overwrite {output}")\n    required = [data / "manifest.json", run / "best.pt", run / "training.json", run / "benchmark.json",\n                run / "result/doe_phase.png", run / "result/metrics.json", ROOT / "notebooks/fresnel_doe_kaggle.ipynb"]\n    if holdout:\n        required.append(Path(holdout) / "manifest.json")\n    if bool(quartet) != bool(quartet_results):\n        raise ValueError("provide both --quartet and --quartet-results")\n    if quartet:\n        required.extend([Path(quartet) / "manifest.json", Path(quartet_results) / "doe_phase.png",\n                         Path(quartet_results) / "metrics.json"])\n    for path in required:\n        if not path.is_file():\n            raise FileNotFoundError(path)\n    entries = []\n    for name in ("README.md", "pyproject.toml", "requirements.txt", "requirements-dev.txt", ".gitignore"):\n        entries.append((ROOT / name, "source/" + name))\n    for folder in ("doe", "tests", "scripts", "docs", "notebooks"):\n        for path in sorted((ROOT / folder).rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc":\n                entries.append((path, "source/" + str(path.relative_to(ROOT))))\n    for prefix, directory in (("data", data), ("run", run),\n                              ("independent_holdout", Path(holdout).resolve() if holdout else None),\n                              ("user_quartet/inputs", Path(quartet).resolve() if quartet else None),\n                              ("user_quartet/result", Path(quartet_results).resolve() if quartet_results else None)):\n        if directory is None:\n            continue\n        for path in sorted(directory.rglob("*")):\n            if path.is_file() and path != output:\n                entries.append((path, prefix + "/" + str(path.relative_to(directory))))\n    if len({name for _, name in entries}) != len(entries):\n        raise ValueError("duplicate archive paths")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(output, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path, name in entries:\n            archive.write(path, name)\n    with zipfile.ZipFile(output) as archive:\n        corrupt = archive.testzip()\n        if corrupt:\n            raise RuntimeError(f"Archive CRC failure: {corrupt}")\n    print(f"{output}: {len(entries)} files, {output.stat().st_size:,} bytes")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data", required=True)\n    parser.add_argument("--run", required=True)\n    parser.add_argument("--holdout")\n    parser.add_argument("--quartet")\n    parser.add_argument("--quartet-results")\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    package(**vars(args))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/prepare_noise_quartet.py': '"""Prepare four preselected texture images as strict hackathon PNG inputs."""\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport numpy as np\nfrom PIL import Image\n\n\n# First numbered variant in four distinct texture families. Selected before synthesis.\nSELECTION = (\n    "128x128/Cracks/Cracks 1 - 128x128.png",\n    "128x128/Gabor/Gabor 1 - 128x128.png",\n    "128x128/Perlin/Perlin 1 - 128x128.png",\n    "128x128/Voronoi/Voronoi 1 - 128x128.png",\n)\nDISTANCES_MM = (24.1, 25.3, 26.5, 27.7)\n\n\ndef prepare(archive, output):\n    archive, output = Path(archive), Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    (output / "originals").mkdir()\n    records = []\n    with ZipFile(archive) as zipped:\n        license_bytes = zipped.read("License.txt")\n        (output / "License.txt").write_bytes(license_bytes)\n        for i, (name, distance) in enumerate(zip(SELECTION, DISTANCES_MM), start=1):\n            raw = zipped.read(name)\n            with Image.open(io.BytesIO(raw)) as source:\n                if source.format != "PNG" or source.size != (128, 128):\n                    raise ValueError(f"Invalid source image: {name}")\n                original_mode = source.mode\n                rgba = source.convert("RGBA")\n                alpha = np.asarray(rgba)[..., 3]\n                background = Image.new("RGBA", source.size, (0, 0, 0, 255))\n                grayscale = Image.alpha_composite(background, rgba).convert("L")\n            filename = f"plane_{i}.png"\n            (output / "originals" / filename).write_bytes(raw)\n            grayscale.save(output / filename)\n            array = np.asarray(grayscale)\n            records.append({"plane": i, "distance_mm": distance, "archive_member": name,\n                            "source_mode": original_mode, "source_alpha_min": int(alpha.min()),\n                            "source_alpha_max": int(alpha.max()), "output": filename,\n                            "size": list(grayscale.size), "mode": grayscale.mode,\n                            "pixel_min": int(array.min()), "pixel_max": int(array.max()),\n                            "pixel_std": float(array.std()),\n                            "source_sha256": hashlib.sha256(raw).hexdigest(),\n                            "output_sha256": hashlib.sha256((output / filename).read_bytes()).hexdigest()})\n    manifest = {\n        "source_archive": archive.name, "source_archive_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),\n        "selection": "first numbered variant from Cracks, Gabor, Perlin, Voronoi; chosen before inference, not selected by achieved metrics",\n        "conversion": "Pillow alpha composite on black, then convert(\'L\'); no resize, contrast stretch, denoising or histogram adjustment",\n        "phase_plane_order": "increasing distance: 24.1, 25.3, 26.5, 27.7 mm",\n        "license": "License.txt supplied in archive states CC0/Public Domain, Screaming Brain Studios",\n        "images": records,\n    }\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    print(json.dumps(manifest, indent=2))\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--archive", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    prepare(args.archive, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/prepare_texture_dataset.py': '"""Build family-disjoint texture quartets from the supplied 128x128 SBS ZIP."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nfrom zipfile import ZipFile\n\nimport numpy as np\nfrom PIL import Image\n\nRESERVED = {"Cracks", "Gabor", "Perlin", "Voronoi"}\nVALIDATION = {"Super Perlin", "Turbulence"}\nTEST = {"Streak", "Vein"}\n\n\ndef prepare(archive, output, train=256, validation=32, test=32, seed=4187):\n    archive, output = Path(archive), Path(output)\n    if min(train, validation, test) < 1:\n        raise ValueError("all splits must be nonempty")\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(output)\n    output.mkdir(parents=True, exist_ok=True)\n    pools = {split: [] for split in ("train", "validation", "test", "reserved")}\n    catalog = []\n    with ZipFile(archive) as zipped:\n        (output / "License.txt").write_bytes(zipped.read("License.txt"))\n        for name in sorted(zipped.namelist()):\n            if not name.lower().endswith(".png"):\n                continue\n            member = PurePosixPath(name)\n            if member.is_absolute() or ".." in member.parts or len(member.parts) != 3:\n                raise ValueError(f"unexpected archive path: {name}")\n            family = member.parts[1]\n            split = "reserved" if family in RESERVED else "validation" if family in VALIDATION else "test" if family in TEST else "train"\n            raw = zipped.read(name)\n            with Image.open(io.BytesIO(raw)) as im:\n                if im.format != "PNG" or im.size != (128, 128):\n                    raise ValueError(f"expected 128x128 PNG: {name}")\n                rgba = im.convert("RGBA")\n                grayscale = Image.alpha_composite(Image.new("RGBA", im.size, (0, 0, 0, 255)), rgba).convert("L")\n            a = np.array(grayscale)\n            if a.max() == 0 or a.std() < 1e-6:\n                raise ValueError(f"blank or constant texture: {name}")\n            pixel_hash = hashlib.sha256(a.tobytes()).hexdigest()\n            relative = Path("pool") / family.replace(" ", "_") / member.name\n            target = output / relative\n            target.parent.mkdir(parents=True, exist_ok=True)\n            grayscale.save(target)\n            record = {"member": name, "family": family, "split": split, "path": str(relative),\n                      "source_sha256": hashlib.sha256(raw).hexdigest(), "pixel_sha256": pixel_hash,\n                      "png_sha256": hashlib.sha256(target.read_bytes()).hexdigest()}\n            catalog.append(record)\n            pools[split].append((a, record))\n    # Exact duplicates may share a split, but cannot leak across split boundaries.\n    assignment = {}\n    for item in catalog:\n        previous = assignment.setdefault(item["pixel_sha256"], item["split"])\n        if previous != item["split"]:\n            raise ValueError(f"duplicate pixels cross splits: {item[\'member\']}")\n    for split in ("train", "validation", "test"):\n        if len(pools[split]) < 4:\n            raise ValueError(f"need >=4 textures in {split}")\n    manifest = {"source": "user-provided SBS noise textures; not the inaccessible full OpenGameArt download",\n                "source_archive": archive.name, "source_archive_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),\n                "seed": seed, "size": 128, "mode": "L", "image_counts": {k: len(v) for k, v in pools.items()},\n                "families": {k: sorted({r[1]["family"] for r in v}) for k, v in pools.items()},\n                "split_policy": "disjoint texture families, plus cross-split exact grayscale-pixel duplicate check; reserved families never used for model selection",\n                "conversion": "alpha composite on black, then Pillow L; no rescaling or contrast stretch",\n                "train_augmentation": "random 90-degree rotations, horizontal reflection and toroidal translations; all transforms recorded",\n                "splits": {}}\n    for split_id, (split, count) in enumerate((("train", train), ("validation", validation), ("test", test))):\n        rng = np.random.default_rng(seed + split_id * 100_000)\n        samples = []\n        for index in range(count):\n            folder = output / split / f"sample_{index:04d}"\n            folder.mkdir(parents=True)\n            images, hashes = [], []\n            for plane, source_index in enumerate(rng.choice(len(pools[split]), size=4, replace=False), start=1):\n                array, record = pools[split][int(source_index)]\n                rotations = int(rng.integers(4)) if split == "train" else 0\n                reflection = bool(rng.integers(2)) if split == "train" else False\n                shift = tuple(int(x) for x in rng.integers(0, 128, 2)) if split == "train" else (0, 0)\n                augmented = np.rot90(array, rotations)\n                if reflection:\n                    augmented = augmented[:, ::-1]\n                augmented = np.roll(augmented, shift, axis=(0, 1))\n                path = folder / f"plane_{plane}.png"\n                Image.fromarray(augmented.copy()).save(path)\n                images.append({"member": record["member"], "family": record["family"],\n                               "source_pixel_sha256": record["pixel_sha256"],\n                               "rotation_90": rotations, "horizontal_reflection": reflection, "roll_yx": shift})\n                hashes.append(hashlib.sha256(path.read_bytes()).hexdigest())\n            samples.append({"path": str(folder.relative_to(output)), "images": images, "sha256": hashes})\n        manifest["splits"][split] = samples\n    (output / "catalog.json").write_text(json.dumps(catalog, indent=2) + "\\n")\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    print(json.dumps({"image_counts": manifest["image_counts"], "families": manifest["families"],\n                      "quartets": {k: len(v) for k, v in manifest["splits"].items()}}, indent=2))\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--archive", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--train", type=int, default=256)\n    parser.add_argument("--validation", type=int, default=32)\n    parser.add_argument("--test", type=int, default=32)\n    parser.add_argument("--seed", type=int, default=4187)\n    prepare(**vars(parser.parse_args()))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/probe_camera.py': '"""Validate a declared camera sampling grid; DOE stays 128x128 and all image pixels are scored."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom doe.camera_optics import CameraConfig,CameraFresnel\nfrom doe.data import QuartetDataset\nfrom doe.engine import setup\nfrom doe.model import initialization\nfrom doe.optics import normalize_targets,unit_complex\nfrom doe.quality import perceptual_loss_per_item\nfrom doe.refinement import refine_phase\nfrom doe.results import phase_metrics\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--padding\',type=int,default=2)\n    p.add_argument(\'--count\',type=int,default=4)\n    p.add_argument(\'--gs-steps\',type=int,default=500)\n    p.add_argument(\'--refine-steps\',type=int,default=1200)\n    p.add_argument(\'--minimum-gain\',type=float,default=.15)\n    p.add_argument(\'--initial-phase\')\n    p.add_argument(\'--initialization\',choices=(\'random\',\'flat\',\'minimum-x\'),default=\'random\')\n    args=p.parse_args();out=Path(args.output)\n    if out.exists():raise FileExistsError(out)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=2)\n    cfg=CameraConfig(fft_padding=args.padding);optics=CameraFresnel(cfg).to(device)\n    data=QuartetDataset(\'data/textures_v1\',\'validation\')\n    targets=torch.stack([data[i] for i in range(args.count)]).to(device)\n    amplitude=normalize_targets(targets)\n    started=time.perf_counter();history=[]\n    with torch.no_grad():\n        if args.initialization==\'random\':source=initialization(optics,amplitude)\n        else:\n            camera_chirp=optics.crop(optics.output_chirp)[None]\n            if args.initialization==\'flat\':desired_phase=torch.ones_like(amplitude,dtype=torch.complex64)\n            else:\n                n=cfg.size\n                frequency=torch.fft.fftfreq(n,device=device)*n\n                sign=frequency.sign();sign[0]=0;sign[n//2]=0\n                log_amplitude=amplitude.clamp_min(.02).log()\n                hilbert=torch.fft.ifft(torch.fft.fft(log_amplitude,dim=-1)*(-1j*sign),dim=-1).real\n                axis=torch.arange(n,device=device)-n//2\n                carrier=-2*torch.pi*(n/(4*args.padding))*axis/n\n                angle=hilbert+carrier\n                desired_phase=torch.polar(torch.ones_like(angle),angle)\n            source=unit_complex(optics.adjoint(amplitude*camera_chirp*desired_phase).mean(dim=1))\n        phase=torch.angle(source);best=phase.clone()\n        if args.initial_phase:\n            saved=np.load(args.initial_phase)\n            np.testing.assert_array_equal(saved[\'targets\'],targets.cpu().numpy())\n            phase=torch.from_numpy(saved[\'phase\']).to(device)\n            source=torch.polar(torch.ones_like(phase),phase);best=phase.clone()\n        best_loss=perceptual_loss_per_item(optics,best,targets)\n        for step in range(args.gs_steps+1):\n            if step%100==0 or step==args.gs_steps:\n                values=phase_metrics(optics,best,targets)\n                record={\'step\':step,\'pearson\':sum(v[\'pearson\'] for v in values)/len(values),\n                        \'ssim\':sum(v[\'ssim\'] for v in values)/len(values),\'per_sample\':values}\n                history.append(record);print({k:record[k] for k in (\'step\',\'pearson\',\'ssim\')},flush=True)\n            if step==args.gs_steps:break\n            fields=optics.full_field(source)\n            camera=optics.crop(fields)\n            scale=(camera.abs()*amplitude).mean((-2,-1),keepdim=True).clamp(args.minimum_gain,.8)\n            desired=amplitude*scale*unit_complex(camera)\n            constrained=fields.clone()\n            start=optics.margin;n=cfg.size\n            constrained[...,start:start+n,start:start+n]=desired\n            source=unit_complex(optics.full_adjoint(constrained).mean(dim=1))\n            if step%10==0:\n                phase=torch.angle(source)\n                loss=perceptual_loss_per_item(optics,phase,targets)\n                improve=loss<best_loss;best[improve]=phase[improve];best_loss=torch.minimum(best_loss,loss)\n        phase,trace=refine_phase(optics,best,targets,steps=args.refine_steps,lr=.05)\n        values=phase_metrics(optics,phase,targets)\n        fractions=optics.camera_energy_fraction(phase).cpu().tolist()\n    report={\'config\':vars(args),\'optics\':cfg.to_dict(),\'sampling\':cfg.sampling_report(),\'split\':\'validation\',\n            \'kind\':\'ONE shared 128x128 phase DOE; explicitly finer camera sampling, NOT the old full-period grid\',\n            \'seconds\':time.perf_counter()-started,\'history\':history,\'refinement\':trace,\'per_sample\':values,\n            \'camera_energy_fraction\':fractions,\n            \'pearson\':sum(v[\'pearson\'] for v in values)/len(values),\'ssim\':sum(v[\'ssim\'] for v in values)/len(values)}\n    out.write_text(json.dumps(report,indent=2)+\'\\n\')\n    np.savez_compressed(out.with_suffix(\'.npz\'),phase=phase.cpu().numpy(),targets=targets.cpu().numpy())\n    print({\'final_pearson\':report[\'pearson\'],\'final_ssim\':report[\'ssim\'],\'energy_fraction\':fractions},flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'scripts/probe_homotopy.py': '"""Reach the original four-plane inverse problem through an easier continuation path."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom doe.data import QuartetDataset\nfrom doe.engine import setup\nfrom doe.model import gerchberg_saxton\nfrom doe.optics import Fresnel,OpticsConfig\nfrom doe.refinement import refine_phase\nfrom doe.results import phase_metrics\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--count\',type=int,default=4)\n    p.add_argument(\'--steps-per-stage\',type=int,default=300)\n    p.add_argument(\'--target-only\',action=\'store_true\')\n    args=p.parse_args();out=Path(args.output)\n    if out.exists():raise FileExistsError(out)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=2)\n    cfg=OpticsConfig();original=Fresnel(cfg).to(device)\n    dataset=QuartetDataset(\'data/textures_v1\',\'validation\')\n    target=torch.stack([dataset[i] for i in range(args.count)]).to(device)\n    common=target.mean(dim=1,keepdim=True).expand(-1,4,-1,-1)\n    mean_z=sum(cfg.distances_m)/4\n    initial_cfg=cfg if args.target_only else OpticsConfig(distances_m=(mean_z,)*4)\n    virtual=Fresnel(initial_cfg).to(device)\n    phase=gerchberg_saxton(virtual,common,steps=100)\n    history=[];started=time.perf_counter()\n    for alpha in np.linspace(0,1,11):\n        distances=cfg.distances_m if args.target_only else tuple(mean_z+float(alpha)*(z-mean_z) for z in cfg.distances_m)\n        virtual=Fresnel(OpticsConfig(distances_m=distances)).to(device)\n        intermediate=(1-float(alpha))*common+float(alpha)*target\n        phase,trace=refine_phase(virtual,phase,intermediate,steps=args.steps_per_stage,lr=.05)\n        values=phase_metrics(original,phase,target)\n        record={\'alpha\':float(alpha),\'scoring\':\'original optics and original PNG targets, always\',\n                \'pearson\':sum(v[\'pearson\'] for v in values)/len(values),\n                \'ssim\':sum(v[\'ssim\'] for v in values)/len(values),\'per_sample\':values,\'refinement\':trace}\n        history.append(record);print({k:record[k] for k in (\'alpha\',\'pearson\',\'ssim\')},flush=True)\n    out.write_text(json.dumps({\'config\':vars(args),\'split\':\'validation\',\'seconds\':time.perf_counter()-started,\n                              \'kind\':\'homotopy optimization; scoring only on ORIGINAL four-plane task\',\n                              \'history\':history},indent=2)+\'\\n\')\n    np.savez_compressed(out.with_suffix(\'.npz\'),phase=phase.cpu().numpy(),targets=target.cpu().numpy())\n\n\nif __name__==\'__main__\':main()\n', 'scripts/probe_neural_fit.py': '"""Diagnostic target-specific network optimization, not an amortized test score."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom doe.data import QuartetDataset\nfrom doe.engine import load_model,setup\nfrom doe.quality import perceptual_loss_per_item\nfrom doe.results import phase_metrics\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--checkpoint\',default=\'artifacts/wavefront_v1/best.pt\')\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--steps\',type=int,default=1000)\n    p.add_argument(\'--lr\',type=float,default=.0005)\n    p.add_argument(\'--count\',type=int,default=1)\n    args=p.parse_args();out=Path(args.output)\n    if out.exists():raise FileExistsError(out)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=2)\n    model,_=load_model(args.checkpoint,device);model.train()\n    data=QuartetDataset(\'data/textures_v1\',\'validation\')\n    targets=torch.stack([data[i] for i in range(args.count)]).to(device)\n    optimizer=torch.optim.Adam(model.parameters(),lr=args.lr)\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=args.steps,eta_min=args.lr/20)\n    history=[];started=time.perf_counter();best_loss=float(\'inf\');best=None\n    for step in range(args.steps+1):\n        optimizer.zero_grad(set_to_none=True)\n        phase=model(targets)\n        loss=perceptual_loss_per_item(model.optics,phase,targets).mean()\n        if float(loss.detach())<best_loss:\n            best_loss=float(loss.detach());best=phase.detach().clone()\n        if step%100==0 or step==args.steps:\n            values=phase_metrics(model.optics,best,targets)\n            r={\'step\':step,\'pearson\':sum(v[\'pearson\'] for v in values)/len(values),\n               \'ssim\':sum(v[\'ssim\'] for v in values)/len(values),\'per_sample\':values}\n            history.append(r);print({k:r[k] for k in (\'step\',\'pearson\',\'ssim\')},flush=True)\n        if step==args.steps:break\n        if not torch.isfinite(loss):raise FloatingPointError(\'nonfinite loss\')\n        loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1,error_if_nonfinite=True)\n        optimizer.step();scheduler.step()\n    out.write_text(json.dumps({\'config\':vars(args),\'split\':\'validation\',\'seconds\':time.perf_counter()-started,\n                              \'kind\':\'target-specific network weight fitting; NOT an independent generalization score\',\n                              \'history\':history},indent=2)+\'\\n\')\n    np.savez_compressed(out.with_suffix(\'.npz\'),phase=best.cpu().numpy(),targets=targets.cpu().numpy())\n\n\nif __name__==\'__main__\':main()\n', 'scripts/probe_progressive.py': '"""Validation-only coarse-to-fine common-phase optimization, with unchanged scoring."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nfrom doe.data import QuartetDataset\nfrom doe.engine import load_model,setup\nfrom doe.quality import perceptual_loss_per_item,ssim_per_plane,pearson_per_plane\nfrom doe.results import phase_metrics\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--data\',default=\'data/textures_v1\')\n    p.add_argument(\'--checkpoint\',default=\'deliverables/texture_finetuned/best.pt\')\n    p.add_argument(\'--count\',type=int,default=4)\n    p.add_argument(\'--stage-steps\',type=int,default=500)\n    p.add_argument(\'--lr\',type=float,default=0.08)\n    p.add_argument(\'--chirp-order\',type=float,default=1)\n    p.add_argument(\'--fit-start\',action=\'store_true\',help=\'Intermediate loss fits amplitude gain; reported PNGs never use fitted gain\')\n    args=p.parse_args()\n    output=Path(args.output)\n    if output.exists():raise FileExistsError(output)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=2)\n    model,_=load_model(args.checkpoint,device);model.requires_grad_(False)\n    data=QuartetDataset(args.data,\'validation\')\n    targets=torch.stack([data[i] for i in range(args.count)]).to(device)\n    cfg=model.optics.config;n=cfg.size\n    axis=torch.arange(n,device=device)-n//2\n    yy,xx=torch.meshgrid(axis,axis,indexing=\'ij\')\n    beta=sum(math.pi*cfg.pixel_pitch_m**2/(cfg.wavelength_m*z) for z in cfg.distances_m)/4\n    carrier=((args.chirp_order*math.pi/n-beta)*(xx.square()+yy.square()))[None].expand(len(targets),-1,-1)\n    residual=torch.zeros(len(targets),1,8,8,device=device)\n    history=[];started=time.perf_counter();total=0\n    for resolution in (8,16,32,64,128):\n        parameter=torch.nn.Parameter(F.interpolate(residual,size=(resolution,resolution),mode=\'bilinear\',align_corners=False).clone())\n        optimizer=torch.optim.Adam([parameter],lr=args.lr)\n        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=args.stage_steps,eta_min=args.lr/10)\n        for step in range(args.stage_steps+1):\n            phase=carrier+F.interpolate(parameter,size=(n,n),mode=\'bilinear\',align_corners=False)[:,0]\n            if step%100==0 or step==args.stage_steps:\n                values=phase_metrics(model.optics,torch.atan2(phase.sin(),phase.cos()),targets)\n                record={\'resolution\':resolution,\'stage_step\':step,\'total_step\':total,\n                        \'pearson\':sum(v[\'pearson\'] for v in values)/len(values),\n                        \'ssim\':sum(v[\'ssim\'] for v in values)/len(values),\'per_sample\':values}\n                history.append(record)\n                print({k:record[k] for k in (\'resolution\',\'stage_step\',\'pearson\',\'ssim\')},flush=True)\n            if step==args.stage_steps:break\n            optimizer.zero_grad(set_to_none=True)\n            if args.fit_start and resolution<128:\n                amplitude=model.optics(phase).abs()\n                gain=(amplitude*targets).sum((-2,-1),keepdim=True)/amplitude.square().sum((-2,-1),keepdim=True).clamp_min(1e-8)\n                fitted=amplitude*gain\n                loss=.5*(1-pearson_per_plane(fitted,targets)).mean()+.5*(1-ssim_per_plane(fitted,targets)).mean()+.05*(fitted-targets).square().mean()\n            else:\n                loss=perceptual_loss_per_item(model.optics,phase,targets).mean()\n            if not torch.isfinite(loss):raise FloatingPointError(\'nonfinite loss\')\n            loss.backward();optimizer.step();scheduler.step();total+=1\n        residual=parameter.detach()\n    output.write_text(json.dumps({\'config\':vars(args),\'split\':\'validation\',\'kind\':\'coarse-to-fine optimization of ONE common phase; scoring unchanged\',\n                                 \'seconds\':time.perf_counter()-started,\'history\':history},indent=2)+\'\\n\')\n    np.savez_compressed(output.with_suffix(\'.npz\'),phase=torch.atan2(phase.sin(),phase.cos()).detach().cpu().numpy(),targets=targets.cpu().numpy())\n\n\nif __name__==\'__main__\':main()\n', 'scripts/probe_quality.py': '"""Validation-only experiments to assess achievable phase quality before model changes."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom doe.data import QuartetDataset\nfrom doe.engine import load_model, setup\nfrom doe.model import physics_loss\nfrom doe.optics import normalize_targets, unit_complex, fft2c\nfrom doe.quality import perceptual_loss_per_item,robust_quality_loss_per_item\nfrom doe.results import phase_metrics\n\n\nclass IndependentPlaneOptics:\n    """A noncompliant control: a different phase-only DOE for every target plane."""\n    def __init__(self,base):\n        self.base=base\n        self.config=base.config\n\n    def __call__(self,phase):\n        return self.base.output_chirp*fft2c(torch.polar(torch.ones_like(phase),phase)*self.base.input_chirp)\n\n    def physical(self,phase):\n        return self(phase)*self.base.physical_factor\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data\',default=\'data/textures_v1\')\n    p.add_argument(\'--checkpoint\',default=\'deliverables/texture_finetuned/best.pt\')\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--steps\',type=int,default=600)\n    p.add_argument(\'--count\',type=int,default=4)\n    p.add_argument(\'--lr\',type=float,default=0.05)\n    p.add_argument(\'--objective\',choices=[\'perceptual\',\'physics\',\'robust\'],default=\'perceptual\')\n    p.add_argument(\'--initialization\',choices=[\'neural\',\'chirp\',\'chirp_projected\',\'zero_projected\',\'random\'],default=\'neural\')\n    p.add_argument(\'--chirp-order\',type=float,default=1)\n    p.add_argument(\'--optimizer\',choices=[\'adam\',\'lbfgs\'],default=\'adam\')\n    p.add_argument(\'--continuous\',action=\'store_true\',help=\'Optimize continuous phase; still evaluate after 8-bit quantization\')\n    p.add_argument(\'--report-every\',type=int,default=50)\n    p.add_argument(\'--initial-phase\',help=\'Resume a diagnostic NPZ on exactly the same validation targets\')\n    p.add_argument(\'--independent-planes\',action=\'store_true\',help=\'NONCOMPLIANT ablation: four separate DOEs instead of one\')\n    p.add_argument(\'--softmax-temp\',type=float,default=0,help=\'Annealed surrogate for maximum gradient only; PNG forward/scoring unchanged\')\n    args=p.parse_args()\n    output=Path(args.output)\n    if output.exists(): raise FileExistsError(output)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=2)\n    model,_=load_model(args.checkpoint,device)\n    model.requires_grad_(False)\n    data=QuartetDataset(args.data,\'validation\')\n    target=torch.stack([data[i] for i in range(args.count)]).to(device)\n    with torch.no_grad():\n        if args.initialization==\'neural\':\n            initial=model(target)\n        elif args.initialization==\'random\':\n            initial=(torch.rand((len(target),128,128),device=device)*2-1)*math.pi\n        elif args.initialization==\'zero_projected\':\n            initial=torch.angle(model.optics.adjoint(normalize_targets(target).to(torch.complex64)).mean(dim=1))\n        else:\n            cfg=model.optics.config\n            axis=torch.arange(cfg.size,device=device)-cfg.size//2\n            yy,xx=torch.meshgrid(axis,axis,indexing=\'ij\')\n            beta=sum(math.pi*cfg.pixel_pitch_m**2/(cfg.wavelength_m*z) for z in cfg.distances_m)/4\n            carrier=(args.chirp_order*math.pi/cfg.size-beta)*(xx.square()+yy.square())\n            initial=carrier[None].expand(len(target),-1,-1).clone()\n            if args.initialization==\'chirp_projected\':\n                fields=model.optics(initial)\n                initial=torch.angle(model.optics.adjoint(normalize_targets(target)*unit_complex(fields)).mean(dim=1))\n    optics=model.optics\n    if args.independent_planes:\n        optics=IndependentPlaneOptics(model.optics)\n        initial=initial[:,None].expand(-1,4,-1,-1).clone()\n    if args.initial_phase:\n        saved=np.load(args.initial_phase)\n        np.testing.assert_array_equal(saved[\'targets\'],target.cpu().numpy())\n        initial=torch.from_numpy(saved[\'phase\']).to(device)\n    phase=torch.nn.Parameter(initial.clone())\n    if args.optimizer==\'adam\':\n        optimizer=torch.optim.Adam([phase],lr=args.lr)\n        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=args.steps,eta_min=args.lr/20)\n    else:\n        optimizer=torch.optim.LBFGS([phase],lr=args.lr,max_iter=10,history_size=20,line_search_fn=\'strong_wolfe\')\n        scheduler=None\n    evaluations=0\n    records=[]; started=time.perf_counter()\n    for step in range(args.steps+1):\n        if step%args.report_every==0 or step==args.steps:\n            wrapped=torch.atan2(phase.sin(),phase.cos())\n            values=phase_metrics(optics,wrapped,target)\n            row={\'step\':step,\'mean_pearson\':sum(v[\'pearson\'] for v in values)/len(values),\n                 \'mean_ssim\':sum(v[\'ssim\'] for v in values)/len(values),\'per_sample\':values}\n            records.append(row)\n            print({k:row[k] for k in (\'step\',\'mean_pearson\',\'mean_ssim\')},flush=True)\n        if step==args.steps: break\n        def closure():\n            nonlocal evaluations\n            optimizer.zero_grad(set_to_none=True)\n            temp=args.softmax_temp*(.01+.99*(1-step/max(1,args.steps)))\n            if args.objective==\'perceptual\':loss=perceptual_loss_per_item(optics,phase,target,quantize=not args.continuous,max_temperature=temp).mean()\n            elif args.objective==\'robust\':loss=robust_quality_loss_per_item(optics,phase,target,quantize=not args.continuous,max_temperature=temp).mean()\n            else:loss=physics_loss(optics,phase,target)\n            if not torch.isfinite(loss): raise FloatingPointError(\'nonfinite loss\')\n            loss.backward(); evaluations+=1\n            return loss\n        optimizer.step(closure)\n        if scheduler is not None: scheduler.step()\n    output.write_text(json.dumps({\'config\':vars(args),\'split\':\'validation\',\'seconds\':time.perf_counter()-started,\n                                 \'kind\':(\'NONCOMPLIANT control: four independent DOEs\' if args.independent_planes else \'per-input phase optimization diagnostic, not a trained network improvement\'),\n                                 \'objective_evaluations\':evaluations,\'history\':records},indent=2)+\'\\n\')\n    np.savez_compressed(output.with_suffix(\'.npz\'),phase=torch.atan2(phase.sin(),phase.cos()).detach().cpu().numpy(),targets=target.cpu().numpy())\n\n\nif __name__==\'__main__\': main()\n', 'scripts/probe_raar.py': '"""Validation-only relaxed alternating reflections with one common phase screen."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom doe.data import QuartetDataset\nfrom doe.engine import load_model,setup\nfrom doe.optics import normalize_targets,unit_complex\nfrom doe.quality import perceptual_loss_per_item\nfrom doe.refinement import refine_phase\nfrom doe.results import phase_metrics\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--beta\',type=float,default=.85)\n    p.add_argument(\'--steps\',type=int,default=1200)\n    p.add_argument(\'--refine-steps\',type=int,default=1200)\n    p.add_argument(\'--count\',type=int,default=4)\n    args=p.parse_args()\n    out=Path(args.output)\n    if out.exists():raise FileExistsError(out)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    device=setup(device=\'cpu\',threads=2)\n    model,_=load_model(\'deliverables/texture_finetuned/best.pt\',device);model.requires_grad_(False)\n    data=QuartetDataset(\'data/textures_v1\',\'validation\')\n    target=torch.stack([data[i] for i in range(args.count)]).to(device)\n    history=[];started=time.perf_counter()\n    with torch.no_grad():\n        initial=model(target)\n        x=model.optics(initial)\n        desired=normalize_targets(target)\n        best=initial.clone()\n        best_loss=perceptual_loss_per_item(model.optics,best,target)\n        for step in range(args.steps+1):\n            if step%100==0 or step==args.steps:\n                values=phase_metrics(model.optics,best,target)\n                r={\'step\':step,\'pearson\':sum(v[\'pearson\'] for v in values)/len(values),\n                   \'ssim\':sum(v[\'ssim\'] for v in values)/len(values),\'per_sample\':values}\n                history.append(r);print({k:r[k] for k in (\'step\',\'pearson\',\'ssim\')},flush=True)\n            if step==args.steps:break\n            a=desired*unit_complex(x)\n            field=unit_complex(model.optics.adjoint(2*a-x).mean(dim=1))\n            b=model.optics.field(field)\n            x=args.beta*b+args.beta*x+(1-2*args.beta)*a\n            if step%10==0:\n                phase=torch.angle(field)\n                loss=perceptual_loss_per_item(model.optics,phase,target)\n                improved=loss<best_loss\n                best[improved]=phase[improved]\n                best_loss=torch.minimum(best_loss,loss)\n        phase,trace=refine_phase(model.optics,best,target,steps=args.refine_steps,lr=.05)\n        final=phase_metrics(model.optics,phase,target)\n    result={\'config\':vars(args),\'split\':\'validation\',\'kind\':\'RAAR + Adam; ONE shared phase, original PNG scoring\',\n            \'seconds\':time.perf_counter()-started,\'history\':history,\'refinement\':trace,\'final\':final,\n            \'mean_pearson\':sum(v[\'pearson\'] for v in final)/len(final),\n            \'mean_ssim\':sum(v[\'ssim\'] for v in final)/len(final)}\n    out.write_text(json.dumps(result,indent=2)+\'\\n\')\n    np.savez_compressed(out.with_suffix(\'.npz\'),phase=phase.cpu().numpy(),targets=target.cpu().numpy())\n    print({\'final_pearson\':result[\'mean_pearson\'],\'final_ssim\':result[\'mean_ssim\']},flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'scripts/report_adaptive_research.py': '"""Publish measured adaptive-network results and separate sampling experiments."""\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\nimport textwrap\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.backends.backend_pdf import PdfPages\nimport numpy as np\nfrom PIL import Image\n\nplt.rcParams["pdf.fonttype"] = 42\n\n\ndef read(path):\n    return json.loads(Path(path).read_text())\n\n\ndef generate(output):\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(output)\n    output.mkdir(parents=True, exist_ok=True)\n    benchmark = read("artifacts/quality_research/adaptive_test.json")\n    training = {name: read(f"artifacts/{name}_v1/training.json") for name in ("wavefront", "adaptive")}\n    user = {steps: read(f"artifacts/adaptive_v1/user_{steps}/metrics.json") for steps in (0, 3000)}\n    rows = []\n\n    def add(split, method, pearson, ssim, grid="original", seconds=None):\n        rows.append(dict(split=split, method=method, pearson=pearson, ssim=ssim,\n                         sampling_grid=grid, seconds_per_quartet_batched=seconds,\n                         both_at_least_0_8=pearson >= .8 and ssim >= .8))\n\n    for name, value in benchmark["summary"].items():\n        add("test_32", name, value["pearson"], value["ssim"],\n            seconds=value["seconds_per_quartet_batched"])\n    for steps, value in user.items():\n        add("reserved_user_quartet", f"adaptive + {steps} phase Adam steps",\n            value["mean_pearson"], value["mean_ssim"])\n\n    probes = Path("artifacts/quality_research")\n    original_names = ["progressive_direct", "progressive_fitwarm", "raar_085", "raar_095",\n                      "homotopy_geometry", "homotopy_targets", "robust_from_expert", "softpeak_from_expert"]\n    for name in original_names:\n        probe = read(probes / f"{name}.json")\n        value = probe if "mean_pearson" in probe else probe["history"][-1]\n        add("validation_first_4", name, value.get("mean_pearson", value.get("pearson")),\n            value.get("mean_ssim", value.get("ssim")))\n    diagnostic = read(probes / "target_specific_wavefront.json")["history"][-1]\n    add("validation_first_1_fitted_diagnostic", "target-specific neural weights; NOT generalization",\n        diagnostic["pearson"], diagnostic["ssim"])\n    for path in sorted(probes.glob("camera_*.json")):\n        probe = read(path)\n        add("validation_first_4", path.stem, probe["pearson"], probe["ssim"],\n            grid=f"declared finer camera, padding={probe[\'config\'][\'padding\']}")\n    assert not any(row["both_at_least_0_8"] for row in rows)\n    with (output / "measurements.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows(rows)\n    summary = dict(goal=dict(pearson=.8, ssim=.8, one_shared_doe=True), goal_achieved=False,\n                   benchmark=benchmark, training=training, measurements=rows,\n                   validation=dict(pytest_passed=26, notebook_code_cells_executed=6,\n                                   notebook_mode="local quick integration; not full GPU training"),\n                   kaggle_status="staged, not submitted: no credential binding or applied Kaggle host access",\n                   interpretation="Compare model gains only on identical test images and original sampling grid. "\n                                  "Finer camera probes change the physical field of view and are separate research.")\n    (output / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\\n")\n\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)\n    methods = ["baseline", "adaptive", "gs_80"]\n    for ax, metric in zip(axes, ("pearson", "ssim")):\n        bars = ax.bar(range(3), [benchmark["summary"][n][metric] for n in methods],\n                      color=["#8798a5", "#237ab0", "#c48c49"])\n        ax.bar_label(bars, fmt="%.3f", padding=3)\n        ax.axhline(.8, color="#b03838", linestyle="--", label="Goal 0.8")\n        ax.set_xticks(range(3), ["Original NN", "Adaptive NN", "GS 80"])\n        ax.set(ylim=(0, 1), title=metric.upper(), ylabel="Same test: 32 quartets")\n        ax.legend()\n    fig.savefig(output / "test_quality.png", dpi=160)\n    plt.close(fig)\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)\n    for ax, (name, run) in zip(axes, training.items()):\n        ax.plot([r["epoch"] for r in run["history"]], [r["validation_loss"] for r in run["history"]], label="validation")\n        trained = run["history"][1:]\n        ax.plot([r["epoch"] for r in trained], [r["train_loss"] for r in trained], label="train")\n        ax.axvline(run["best_epoch"], color="gray", linestyle="--")\n        ax.set(title=name, xlabel="Epoch", ylabel="Perceptual loss")\n        ax.legend()\n    fig.savefig(output / "training.png", dpi=160)\n    plt.close(fig)\n    fig, axes = plt.subplots(4, 3, figsize=(10, 12), constrained_layout=True)\n    for i in range(4):\n        paths = [Path(f"artifacts/adaptive_v1/user_0/target_{i+1}.png")] + [\n            Path(f"artifacts/adaptive_v1/user_{n}/reconstruction_{i+1}.png") for n in (0, 3000)]\n        for j, (ax, path) in enumerate(zip(axes[i], paths)):\n            with Image.open(path) as im:\n                ax.imshow(np.array(im), cmap="gray", vmin=0, vmax=255)\n            ax.axis("off")\n            if j == 0:\n                ax.set_title(f"Target, z={user[0][\'planes\'][i][\'distance_mm\']:.1f} mm", fontsize=9)\n            else:\n                plane = user[(0, 3000)[j-1]]["planes"][i]\n                ax.set_title(f"{(\'Adaptive NN\', \'+ Adam 3000\')[j-1]}\\nr={plane[\'pearson\']:.3f}; SSIM={plane[\'ssim\']:.3f}", fontsize=9)\n    fig.savefig(output / "user_comparison.png", dpi=150)\n    plt.close(fig)\n\n    test = benchmark["summary"]\n    delta = {key: test["adaptive"][key] - test["baseline"][key] for key in ("pearson", "ssim")}\n    table = "| Набор | Метод | Pearson | SSIM | Сетка |\\n|---|---|---:|---:|---|\\n"\n    for row in rows:\n        table += f"| {row[\'split\']} | {row[\'method\']} | {row[\'pearson\']:.5f} | {row[\'ssim\']:.5f} | {row[\'sampling_grid\']} |\\n"\n    text = f"""# Adaptive wavefront: результаты обучения\n\n**Pearson ≥0.8 и SSIM ≥0.8 одновременно пока не достигнуты.**\nНовая чистая сеть на том же test из 32 четвёрок: **{test[\'adaptive\'][\'pearson\']:.5f} / {test[\'adaptive\'][\'ssim\']:.5f}**.\nПрежняя сеть в том же измерении: {test[\'baseline\'][\'pearson\']:.5f} / {test[\'baseline\'][\'ssim\']:.5f}.\nПрибавка: +{delta[\'pearson\']:.5f} Pearson и +{delta[\'ssim\']:.5f} SSIM.\nGS80 здесь качественнее обеих сетей; порог 0.8 также не достигнут.\n\n![Один test и одна физическая сетка](test_quality.png)\n\n## Что обучено\n\n1. Wavefront U-Net: 20 эпох, 6 оптических слоёв, инициализатор ширины 8 с тремя\n   уровнями. Старт из прежних texture-весов, perceptual loss с 8-битным STE.\n   Validation loss {training[\'wavefront\'][\'initial_validation_loss\']:.6f} → {training[\'wavefront\'][\'best_validation_loss\']:.6f},\n   лучшая эпоха {training[\'wavefront\'][\'best_epoch\']}; CPU {training[\'wavefront\'][\'elapsed_seconds\']:.1f} с.\n2. Adaptive: 16 эпох, 12 оптических слоёв, U-Net с четырьмя уровнями и обучаемая\n   пространственная амплитудная свобода/фазовая поправка каждого слоя. Модуль\n   итогового единственного ДОЭ всегда единица. Старт из выбранного wavefront checkpoint.\n   Validation loss {training[\'adaptive\'][\'initial_validation_loss\']:.6f} → {training[\'adaptive\'][\'best_validation_loss\']:.6f},\n   лучшая эпоха {training[\'adaptive\'][\'best_epoch\']}; CPU {training[\'adaptive\'][\'elapsed_seconds\']:.1f} с.\n\n![Истории обучения](training.png)\n\nОбучение использовало прежние 256 train-четвёрок, validation — 32. Семейства\ntrain/validation/test различны; Cracks/Gabor/Perlin/Voronoi зарезервированы.\nCheckpoint выбирался по validation loss. Test не использовался в loss,\nоптимизации весов или выборе эпохи. Повторные исследовательские сравнения на\nодном test означают, что его нельзя считать новым, ранее не виденным аудитом.\nДля итогового подтверждения новой архитектуры нужен дополнительный независимый набор.\nНа CPU, 1 поток, batch=4: исходная сеть {test[\'baseline\'][\'seconds_per_quartet_batched\']:.4f},\nadaptive {test[\'adaptive\'][\'seconds_per_quartet_batched\']:.4f}, GS80 {test[\'gs_80\'][\'seconds_per_quartet_batched\']:.4f} с/четвёрку.\nЭто время синтеза в пакете, без IO и расчёта метрик; новая сеть тяжелее прежней.\n\n## Выбранные пользовательские изображения\n\nAdaptive без оптимизатора: {user[0][\'mean_pearson\']:.5f}/{user[0][\'mean_ssim\']:.5f}.\nС 3000 шагами общей фазы: {user[3000][\'mean_pearson\']:.5f}/{user[3000][\'mean_ssim\']:.5f}.\nПоследний результат хуже опубликованного v2 QAT + 3000: 0.59732/0.46088.\nПоэтому v2 сохранён, новый гибрид не объявлен улучшением.\nЛучший опубликованный test-гибрид v2 + 600 остаётся 0.67111/0.60637;\nего нельзя напрямую сопоставлять со скоростью одного forward pass.\n\n![Реконструкции выбранных изображений](user_comparison.png)\n\n## Журнал дополнительных попыток\n\n{table}\n\nProgressive phase, RAAR, homotopy и новые loss использовали первые 4 validation-четвёрки.\nRobust/softpeak продолжают прежние 3000 шагов ещё на 1800; это больше вычислений.\nRAAR при обоих beta вернулся к одной сохранённой лучшей инициализации.\nTarget-specific neural fitting меняет веса на одном входе: это диагностика, не\nобобщение сети. Параметры опытов, история и итоговые фазы лежат в `../probes`.\n\n## Отдельное исследование сетки камеры\n\nВ приложенном ТЗ указан шаг ПВМС, но не физический шаг камеры. Проверена явно\nзаданная более частая сетка: нулевое дополнение **до** единственного FFT,\nзатем центральные 128×128 отсчётов. Управляемый ДОЭ всё ещё 128×128; оцениваются\nвсе 128×128 пикселей каждого исходного target. Выходной шаг и физическое поле\nзрения меняются относительно прежней сетки λz/(128Δu). Эти результаты **не являются\nулучшением модели при прежней дискретизации**, не включены в основной benchmark\nи не используются в production-инференсе. Учтена также энергия вне камеры.\nПодгонка gain применялась только внутри MRAF-инициализации; конечная оценка\nиспользует прежнюю нормировку модуля по собственному максимуму без target gain.\nНи одна проверенная сетка не дала оба показателя 0.8.\n\n## Следующий GPU-эксперимент\n\nГотов приватный Kaggle job: 24 оптических слоя, 4 уровня U-Net, 80 эпох,\n2048 train-четвёрок, те же 32 validation и 32 test. Исходные train-текстуры прежние:\nувеличено число комбинаций и аугментаций, не число независимых источников.\nNotebook самодостаточен и не требует Интернета внутри Kaggle. Целиком выполнены\n6 ячеек локального quick-прогона; это проверка интеграции, не GPU-обучение.\nУдалённый запуск **не отправлен**: нет KAGGLE_API_TOKEN в секретах и применённого\nдоступа к api.kaggle.com/www.kaggle.com. Подготовленные metadata и инструкции\nв `../kaggle`; скрипт подготовки `scripts/stage_kaggle_job.py`.\nБольший запуск не гарантирует 0.8. После него требуются фактические метрики.\n\n## Протокол и проверки\n\nОдин общий фазовый ДОЭ, λ=532 нм, шаг ПВМС 10 мкм, z=24.1/25.3/26.5/27.7 мм.\nДля основной таблицы физика и выходные сетки прежние. Оценка по реальным\n8-битным реконструкциям из квантованной фазы, SSIM skimage с окном 7 и\ndata_range=1, Pearson по всем пикселям. Нет фильтрации и подмены входных изображений.\n26 тестов прошли, включая прямую квадратуру Fresnel, сопряжённость и энергию\ncamera-модели, градиенты нового инициализатора и checkpoint round trip.\nФизический ПВМС не проверялся. Неудачные опыты не доказывают невозможность 0.8.\n"""\n    (output / "report.md").write_text(text)\n    with PdfPages(output / "report.pdf") as pdf:\n        fig = plt.figure(figsize=(8.27, 11.69))\n        fig.text(.07, .94, "Adaptive wavefront: измеренные результаты", fontsize=16, weight="bold")\n        paragraphs = [\n            "Цель: один общий фазовый ДОЭ, Pearson и SSIM не ниже 0.8. Цель пока не достигнута.",\n            f"Одинаковый test, 32 четвёрки: прежняя сеть {test[\'baseline\'][\'pearson\']:.5f}/{test[\'baseline\'][\'ssim\']:.5f}; adaptive {test[\'adaptive\'][\'pearson\']:.5f}/{test[\'adaptive\'][\'ssim\']:.5f}; GS80 {test[\'gs_80\'][\'pearson\']:.5f}/{test[\'gs_80\'][\'ssim\']:.5f} (Pearson/SSIM).",\n            "Обучены U-Net wavefront (20 эпох) и adaptive сеть с 12 оптическими слоями (16 эпох). Выбор checkpoint по 32 validation-четвёркам. Train: 256 четвёрок, семейства раздельны. Повторный test не является новым независимым аудитом.",\n            f"На выбранной пользовательской четвёрке adaptive: {user[0][\'mean_pearson\']:.5f}/{user[0][\'mean_ssim\']:.5f}, +3000 шагов общей фазы: {user[3000][\'mean_pearson\']:.5f}/{user[3000][\'mean_ssim\']:.5f}. Старый v2-гибрид лучше; он сохранён.",\n            "Дополнительно измерены progressive phase, RAAR, homotopy, robust loss и distributed max gradient. Порог не достигнут. Подгонка весов на одном target обозначена только как диагностика.",\n            "Более частая camera-сетка меняет физический шаг и поле зрения. Её оценки вынесены отдельно, не являются выигрышем сети на прежней сетке. Все пиксели targets оценены; энергия вне камеры учтена. Оба 0.8 также не достигнуты.",\n            "Подготовлен приватный Kaggle GPU job: 24 слоя, 80 эпох, 2048 train-четвёрок. Шесть ячеек notebook прошли локальный quick-прогон. Полный GPU job не отправлен: нет секрета Kaggle и применённого сетевого доступа. Обещания 0.8 нет.",\n            "26 тестов пройдены. Полный Markdown/CSV/JSON и исходные истории включены в поставку. Основные метрики: реальный 8-битный ДОЭ и PNG амплитуд, прежняя физика, без фильтрации и target gain. Физический эксперимент не выполнен. Неудачи не доказывают невозможности цели."\n        ]\n        y = .88\n        for paragraph in paragraphs:\n            lines = textwrap.wrap(paragraph, 90)\n            fig.text(.07, y, "\\n".join(lines), fontsize=10, va="top", linespacing=1.4)\n            y -= len(lines) * .019 + .025\n        pdf.savefig(fig)\n        plt.close(fig)\n        for name in ("test_quality.png", "training.png", "user_comparison.png"):\n            fig, ax = plt.subplots(figsize=(8.27, 11.69) if "user" in name else (10, 6))\n            with Image.open(output / name) as im:\n                ax.imshow(np.array(im))\n            ax.axis("off")\n            fig.tight_layout()\n            pdf.savefig(fig)\n            plt.close(fig)\n    print(json.dumps({"output": str(output), "goal_achieved": False, "adaptive_test": test["adaptive"]}, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", required=True)\n    generate(parser.parse_args().output)\n', 'scripts/report_quality_research.py': '"""Report valid single-DOE improvements separately from noncompliant controls."""\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\nimport textwrap\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.backends.backend_pdf import PdfPages\nimport numpy as np\nfrom PIL import Image\n\nplt.rcParams[\'pdf.fonttype\']=42\n\n\ndef read(path):\n    return json.loads(Path(path).read_text())\n\n\ndef generate(output):\n    output=Path(output)\n    if output.exists() and any(output.iterdir()): raise FileExistsError(output)\n    output.mkdir(parents=True,exist_ok=True)\n    base=read(\'deliverables/texture_finetuned/benchmark.json\')[\'summary\'][\'neural\']\n    benchmark=read(\'artifacts/quality_research/qat_test_with_refinement.json\')\n    training=read(\'artifacts/perceptual_qat_v1/training.json\')\n    user_base=read(\'deliverables/texture_finetuned/result/metrics.json\')\n    user={n:read(f\'artifacts/quality_research/qat_user_{n}/metrics.json\') for n in (0,600,3000)}\n    probes={p.stem:read(p) for p in sorted(Path(\'artifacts/quality_research\').glob(\'validation_*.json\'))}\n    rows=[]\n    def add(split,name,pearson,ssim,seconds=None,compliant=True):\n        rows.append({\'split\':split,\'method\':name,\'pearson\':pearson,\'ssim\':ssim,\'seconds_per_quartet\':seconds,\n                     \'one_shared_doe\':compliant,\'both_at_least_0_8\':bool(compliant and pearson>=0.8 and ssim>=0.8)})\n    add(\'test_32\',\'original neural\',base[\'mean_pearson\'],base[\'mean_ssim\'],base[\'seconds_per_quartet_batched\'])\n    for key in (\'neural\',\'gs_80\',\'neural_refined\'):\n        value=benchmark[\'summary\'][key]\n        add(\'test_32\',{\'neural\':\'QAT neural candidate\',\'gs_80\':\'GS 80\',\'neural_refined\':\'QAT neural + 600 Adam steps\'}[key],\n            value[\'mean_pearson\'],value[\'mean_ssim\'],value[\'seconds_per_quartet_batched\'])\n    add(\'user_quartet\',\'original neural\',user_base[\'mean_pearson\'],user_base[\'mean_ssim\'],user_base[\'metadata\'][\'network_forward_seconds\'])\n    for n,r in user.items():\n        seconds=r[\'metadata\'][\'network_forward_seconds\']+r[\'metadata\'].get(\'refinement\',{}).get(\'seconds\',0)\n        add(\'user_quartet\',f\'QAT neural + {n} Adam steps\',r[\'mean_pearson\'],r[\'mean_ssim\'],seconds)\n    for name,p in probes.items():\n        last=p[\'history\'][-1]\n        compliant=not p[\'config\'].get(\'independent_planes\',False)\n        add(\'validation_first_4\',name,last[\'mean_pearson\'],last[\'mean_ssim\'],p[\'seconds\']/p[\'config\'][\'count\'],compliant)\n    with (output/\'measurements.csv\').open(\'w\',newline=\'\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=list(rows[0]),lineterminator=\'\\n\')\n        writer.writeheader();writer.writerows(rows)\n    summary={\'goal\':{\'pearson\':0.8,\'ssim\':0.8,\'one_shared_doe\':True},\'goal_achieved\':False,\n             \'checkpoint_selection\':\'QAT selected by validation perceptual loss; test used only for final evaluation\',\n             \'production_weights_replaced\':False,\'measurements\':rows,\'qat_training\':training,\n             \'limitations\':[\'The four-independent-DOE control is not compliant with the original task.\',\n                            \'Failure of these experiments is not a mathematical impossibility proof.\',\n                            \'17 tests passed; GPU/Kaggle execution was not available.\']}\n    assert not any(r[\'both_at_least_0_8\'] for r in rows)\n    (output/\'summary.json\').write_text(json.dumps(summary,indent=2,allow_nan=False)+\'\\n\')\n    fig,axes=plt.subplots(1,2,figsize=(11,4.5),constrained_layout=True)\n    test_rows=[r for r in rows if r[\'split\']==\'test_32\']\n    for ax,key in zip(axes,(\'pearson\',\'ssim\')):\n        bars=ax.bar(np.arange(len(test_rows)),[r[key] for r in test_rows],color=[\'#7c8792\',\'#a5b5c3\',\'#d8984c\',\'#2f80b8\'])\n        ax.bar_label(bars,fmt=\'%.3f\',padding=3)\n        ax.axhline(.8,color=\'#b83030\',ls=\'--\',label=\'Goal 0.8\')\n        ax.set_xticks(np.arange(len(test_rows)),[\'Original NN\',\'QAT NN\',\'GS 80\',\'NN + Adam 600\'],rotation=18)\n        ax.set(ylim=(0,1),ylabel=key.upper(),title=\'Same test split: 32 quartets\')\n        ax.legend();ax.grid(axis=\'y\',alpha=.2)\n    fig.savefig(output/\'test_quality.png\',dpi=160);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4.5),constrained_layout=True)\n    selected=[\'validation_perceptual_600\',\'validation_neural_3000\',\'validation_independent_planes_600\']\n    labels=[\'1 DOE, 600 steps\',\'1 DOE, 3000 steps\',\'4 independent DOEs\\nNOT COMPLIANT\']\n    for ax,key in zip(axes,(\'mean_pearson\',\'mean_ssim\')):\n        values=[probes[n][\'history\'][-1][key] for n in selected]\n        bars=ax.bar(np.arange(3),values,color=[\'#668fba\',\'#2f80b8\',\'#cf8484\'])\n        bars[-1].set_hatch(\'//\')\n        ax.bar_label(bars,fmt=\'%.3f\',padding=3)\n        ax.axhline(.8,color=\'#b83030\',ls=\'--\')\n        ax.set_xticks(np.arange(3),labels,fontsize=8)\n        ax.set(ylim=(0,1.05),title=key.replace(\'mean_\',\'\').upper(),ylabel=\'Validation first 4 quartets\')\n    fig.savefig(output/\'constraint_control.png\',dpi=160);plt.close(fig)\n    fig,axes=plt.subplots(4,4,figsize=(12,12),constrained_layout=True)\n    for i in range(4):\n        paths=[Path(f\'artifacts/quality_research/qat_user_0/target_{i+1}.png\')]+[\n            Path(f\'artifacts/quality_research/qat_user_{n}/reconstruction_{i+1}.png\') for n in (0,600,3000)]\n        for j,(ax,path) in enumerate(zip(axes[i],paths)):\n            with Image.open(path) as image: ax.imshow(np.array(image),cmap=\'gray\',vmin=0,vmax=255)\n            ax.axis(\'off\')\n            if j==0: ax.set_title(f\'Target, z={user[0]["planes"][i]["distance_mm"]:.1f} mm\',fontsize=9)\n            else:\n                n=(0,600,3000)[j-1];r=user[n][\'planes\'][i]\n                ax.set_title(f\'{n} steps: r={r["pearson"]:.3f}\\nSSIM={r["ssim"]:.3f}\',fontsize=9)\n    fig.savefig(output/\'user_comparison.png\',dpi=150);plt.close(fig)\n    table=\'| Набор | Метод | Pearson | SSIM | Один общий ДОЭ |\\n|---|---|---:|---:|---|\\n\'\n    for r in rows:\n        table+=f"| {r[\'split\']} | {r[\'method\']} | {r[\'pearson\']:.5f} | {r[\'ssim\']:.5f} | {\'да\' if r[\'one_shared_doe\'] else \'**нет — контроль**\'} |\\n"\n    markdown=f\'\'\'# Исследование цели 0.8 по Pearson и SSIM\n\n**При одном общем ДОЭ цель пока не достигнута.** Это исследование и улучшение\nкачества, а не заявление о выполнении целевого порога.\n\n## Фактические результаты\n\n{table}\n\n![Тест](test_quality.png)\n\n![Контроль ограничения](constraint_control.png)\n\n![Пользовательская четвёрка](user_comparison.png)\n\n## Что изменено\n\n- Добавлен дифференцируемый SSIM, проверенный против skimage и конечными разностями.\n- Добавлено обучение с прямым perceptual loss и straight-through квантованием в 8 бит.\n  12 эпох QAT дали лишь малое улучшение validation loss; на test обе метрики чистой\n  сети немного снизились. Поэтому прежние production-веса не заменены.\n- Проверены Adam, LBFGS, разные chirp-инициализации, 600 и 3000 шагов фазы.\n- Добавлен опциональный `--refine-steps`: сеть плюс итеративный решатель одной общей\n  фазы. Это работает медленнее одного forward pass; число шагов и время явно записаны.\n- В контрольном опыте каждому изображению разрешён свой отдельный ДОЭ.\n  Высокие 0.99/0.95 относятся к этой другой постановке, не к выполнению задания.\n\n## Корректность сравнения\n\nФизика Френеля, λ, расстояния и размер 128×128 не менялись. Сохраняется амплитуда,\nне интенсивность. Все финальные метрики считают фактические реконструкции из\nсохранённого 8-битного PNG ДОЭ; нет фильтрации, подгонки gain по target или ROI.\nНастройки выбирались по validation, после чего выполнена оценка на тех же 32 test\nчетвёрках. Для исследовательского подбора использованы первые 4 validation-четвёрки,\nпоэтому эти малые оценки не являются новым независимым test.\nРезультаты с разными split нельзя сравнивать как показатели одной выборки.\nИтеративный solver использует желаемые изображения как входы обратной задачи;\nего качество не выдаётся за качество чистой нейросети. Времена — текущий CPU,\nbatched throughput, без статистических гарантий. Опыты не доказывают математической\nневозможности 0.8 с одним ДОЭ.\n\n## Следующие направления\n\n1. Более мощный обучаемый инициализатор волнового фронта (U-Net/HoloNet),\n   пространственные маски амплитудной свободы и больше обучаемых optical stages.\n   Это перспективные изменения архитектуры, **не проверенное обещание 0.8**.\n2. Teacher/student дистилляция итеративного решателя. Сначала учитель должен\n   показать нужное качество: текущие результаты сами по себе 0.8 не обеспечивают.\n3. При разрешённом изменении задания — отдельные ДОЭ/экспозиции или больше\n   управляемых пикселей. Четыре независимых ДОЭ показали высокий результат в\n   контрольном опыте, но исходное условие одного ДОЭ при этом нарушается.\n\nИсточники: [Neural Holography](https://github.com/computational-imaging/neural-holography)\nи [Tensor Holography](https://github.com/liangs111/tensor_holography). Прочитаны\nофициальные README и опубликованный алгоритм SGD. Их оценки не переносились на\nнашу постановку, код/веса не копировались. Дополнительные источники и исходные\nистории находятся в `docs/quality-research.md` и каталоге probes архива.\n\nПроверки: **17 тестов pytest**. Полные CPU-обучение и benchmark выполнены.\nKaggle/GPU и эксперимент на физическом ПВМС не выполнялись.\n\'\'\'\n    (output/\'report.md\').write_text(markdown)\n    refined=benchmark[\'summary\'][\'neural_refined\']\n    with PdfPages(output/\'report.pdf\') as pdf:\n        fig=plt.figure(figsize=(8.27,11.69))\n        fig.text(.07,.94,\'Исследование качества ДОЭ: цель 0.8\',fontsize=17,weight=\'bold\')\n        paragraphs=[\n            \'Цель: Pearson ≥0.8 и SSIM ≥0.8 одновременно, один общий фазовый ДОЭ 128×128 для четырёх плоскостей. В этом исследовании цель пока не достигнута.\',\n            f"На тех же 32 test-четвёрках исходная сеть: {base[\'mean_pearson\']:.5f}/{base[\'mean_ssim\']:.5f}; сеть + 600 шагов фазы: {refined[\'mean_pearson\']:.5f}/{refined[\'mean_ssim\']:.5f} (Pearson/SSIM).",\n            f"Пользовательская четвёрка: {user_base[\'mean_pearson\']:.5f}/{user_base[\'mean_ssim\']:.5f} → {user[3000][\'mean_pearson\']:.5f}/{user[3000][\'mean_ssim\']:.5f} с 3000 шагами. Это итеративный решатель после сети, не один forward pass.",\n            f"Время сети + 600 шагов: {refined[\'seconds_per_quartet_batched\']:.2f} с/четвёрку в пакетном CPU-тесте. Увеличение качества оплачивается временем расчёта.",\n            \'QAT/perceptual fine-tuning: 12 эпох. На test чистая сеть немного хуже прежней, поэтому production-веса сохранены. Кандидат опубликован отдельно.\',\n            \'Контроль: четыре независимых ДОЭ на первые 4 validation-четвёрки дали около 0.990/0.951. Это другая постановка: исходному условию одного общего ДОЭ она не соответствует.\',\n            \'Физика, входы и метрики сохранены. Оценка после настоящего 8-битного PNG ДОЭ, без фильтрации, ROI и gain по target. Отрицательный результат не является доказательством невозможности 0.8.\',\n            \'Дальше: обучаемый wavefront-инициализатор U-Net/HoloNet и больше optical stages; дистилляция качественного учителя. Успех до 0.8 не подтверждён. Дополнительные ДОЭ требуют изменения задания.\',\n            \'Источники: Neural Holography (Peng et al.) и Tensor Holography (Shi et al.). Прочитаны опубликованные материалы; код и веса не копировались. 17 тестов пройдены. GPU/Kaggle и физический ПВМС не проверены.\'\n        ]\n        y=.88\n        for paragraph in paragraphs:\n            lines=textwrap.wrap(paragraph,91)\n            fig.text(.07,y,\'\\n\'.join(lines),fontsize=10,va=\'top\',linespacing=1.4)\n            y-=len(lines)*.019+.023\n        pdf.savefig(fig);plt.close(fig)\n        for name,title,figsize in [(\'test_quality.png\',\'Одинаковый test: 32 четвёрки\',(10,6)),\n                                   (\'constraint_control.png\',\'Контроль: одинаковые 4 validation-четвёрки\',(10,6)),\n                                   (\'user_comparison.png\',\'Пользовательские изображения: исходы и реконструкции\',(8.27,11.69))]:\n            fig,ax=plt.subplots(figsize=figsize)\n            with Image.open(output/name) as image: ax.imshow(np.array(image))\n            ax.axis(\'off\');ax.set_title(title,fontsize=12)\n            fig.tight_layout();pdf.savefig(fig);plt.close(fig)\n    print(json.dumps({\'report\':str(output),\'goal_achieved\':False,\'test_neural_refined\':refined},indent=2))\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\',required=True)\n    generate(p.parse_args().output)\n', 'scripts/report_texture_finetuning.py': '"""Create a PDF, Markdown and CSV report from actual before/after run outputs."""\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\nimport textwrap\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.backends.backend_pdf import PdfPages\nimport numpy as np\nfrom PIL import Image\n\nplt.rcParams["pdf.fonttype"] = 42  # Embed searchable TrueType text, including Cyrillic.\n\n\ndef read(path):\n    return json.loads(Path(path).read_text())\n\n\ndef generate(training, dataset, before_benchmark, after_benchmark, before_result, after_result, output):\n    training, dataset = read(training), read(Path(dataset) / "manifest.json")\n    before, after = read(before_benchmark), read(after_benchmark)\n    before_result, after_result, output = Path(before_result), Path(after_result), Path(output)\n    old, new = read(before_result / "metrics.json"), read(after_result / "metrics.json")\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if before["sample_count"] != after["sample_count"]:\n        raise ValueError("benchmark sample counts differ")\n    if training["parent_checkpoint"]["sha256"] != old["metadata"]["checkpoint_sha256"]:\n        raise ValueError("before result is not the fine-tuning parent")\n    if new["metadata"]["checkpoint_sha256"] != after["checkpoint_sha256"]:\n        raise ValueError("after benchmark and result use different checkpoints")\n    if old["inputs"] != new["inputs"]:\n        raise ValueError("before/after quartet inputs differ")\n    base, fine = before["summary"]["neural"], after["summary"]["neural"]\n    rows = []\n\n    def comparison(label, a, b):\n        row = {"case": label}\n        for key in ("pearson", "ssim"):\n            row[f"before_{key}"] = a[key]\n            row[f"after_{key}"] = b[key]\n            row[f"delta_{key}"] = b[key] - a[key]\n        rows.append(row)\n\n    comparison("independent_test_mean", {k: base[f"mean_{k}"] for k in ("pearson", "ssim")},\n               {k: fine[f"mean_{k}"] for k in ("pearson", "ssim")})\n    comparison("reserved_user_quartet_mean", {k: old[f"mean_{k}"] for k in ("pearson", "ssim")},\n               {k: new[f"mean_{k}"] for k in ("pearson", "ssim")})\n    for a, b in zip(old["planes"], new["planes"]):\n        comparison(f"reserved_z_{b[\'distance_mm\']:.1f}_mm", a, b)\n    with (output / "before_after.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows(rows)\n    summary = {"training": {k: training[k] for k in ("config", "parent_checkpoint", "initial_validation_loss",\n                 "best_validation_loss", "best_epoch", "elapsed_seconds", "parameter_l2_change_last_vs_initial")},\n               "image_counts": dataset["image_counts"], "families": dataset["families"],\n               "test_count": after["sample_count"], "comparisons": rows,\n               "sampling": new["sampling"], "device": after["device"],\n               "before_checkpoint_sha256": before["checkpoint_sha256"],\n               "after_checkpoint_sha256": after["checkpoint_sha256"]}\n    (output / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\\n")\n\n    history = training["history"]\n    objective = training["config"].get("objective", "physics")\n    fig, ax = plt.subplots(figsize=(8.5, 4.4), constrained_layout=True)\n    epochs = [row["epoch"] for row in history]\n    ax.plot(epochs[1:], [row["train_loss"] for row in history[1:]], label="Train")\n    ax.plot(epochs, [row["validation_loss"] for row in history], label="Validation")\n    ax.axvline(training["best_epoch"], color="gray", linestyle="--", label="Selected checkpoint")\n    ax.set(xlabel="Fine-tuning epoch", ylabel=f"{objective.title()} loss", title="Training history (selection uses validation only)")\n    ax.legend(); ax.grid(alpha=0.25)\n    fig.savefig(output / "learning_curves.png", dpi=160)\n    plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), constrained_layout=True)\n    gs_keys = sorted((k for k in after["summary"] if k.startswith("gs_")), key=lambda k: int(k.split("_")[1]))\n    labels = ["Before", "Fine-tuned", *[key.upper().replace("_", " ") for key in gs_keys]]\n    records = [base, fine, *[after["summary"][key] for key in gs_keys]]\n    for ax, key in zip(axes, ("pearson", "ssim")):\n        values = [r[f"mean_{key}"] for r in records]\n        bars = ax.bar(labels, values, color=["#8a939d", "#3179b8", "#70a46c", "#db8c44"])\n        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)\n        ax.set(ylim=(min(0, min(values) - 0.05), 1.0), ylabel=key.upper(), title=f"Independent textures: {key}")\n        ax.grid(axis="y", alpha=0.2)\n    fig.savefig(output / "holdout_quality.png", dpi=160)\n    plt.close(fig)\n\n    fig, axes = plt.subplots(4, 3, figsize=(9, 10), constrained_layout=True)\n    for i in range(4):\n        paths = [after_result / f"target_{i+1}.png", before_result / f"reconstruction_{i+1}.png", after_result / f"reconstruction_{i+1}.png"]\n        for ax, path in zip(axes[i], paths):\n            with Image.open(path) as im:\n                ax.imshow(np.array(im), cmap="gray", vmin=0, vmax=255)\n            ax.axis("off")\n        axes[i, 0].set_title(f"Target, z={new[\'planes\'][i][\'distance_mm\']:.1f} mm")\n        axes[i, 1].set_title(f"Before: r={old[\'planes\'][i][\'pearson\']:.3f}, SSIM={old[\'planes\'][i][\'ssim\']:.3f}", fontsize=9)\n        axes[i, 2].set_title(f"After: r={new[\'planes\'][i][\'pearson\']:.3f}, SSIM={new[\'planes\'][i][\'ssim\']:.3f}", fontsize=9)\n    fig.savefig(output / "before_after.png", dpi=160)\n    plt.close(fig)\n\n    table = "| Набор/плоскость | Pearson до | После | SSIM до | После |\\n|---|---:|---:|---:|---:|\\n"\n    for row in rows:\n        table += f"| {row[\'case\']} | {row[\'before_pearson\']:.5f} | {row[\'after_pearson\']:.5f} | {row[\'before_ssim\']:.5f} | {row[\'after_ssim\']:.5f} |\\n"\n    markdown = f"""# Отчёт по дообучению на текстурах\n\nОбучение с переносом весов исходной модели. Целевая функция: **{objective}**.\nУстройство: **{after[\'device\']}**. Эпох: {training[\'config\'][\'epochs\']}; выбран checkpoint\nэпохи **{training[\'best_epoch\']}** по минимуму validation loss. Adam запущен с новой\nисторией оптимизатора. Время обучения: {training[\'elapsed_seconds\']:.1f} с.\n\nValidation loss: **{training[\'initial_validation_loss\']:.6f} → {training[\'best_validation_loss\']:.6f}**.\n\n## Данные и отсутствие утечки\n\nИспользован предоставленный ZIP: **262** текстуры 128×128, а не полный набор из 700.\nОбучение: {dataset[\'image_counts\'][\'train\']} исходных изображений;\nvalidation: {dataset[\'image_counts\'][\'validation\']}; test: {dataset[\'image_counts\'][\'test\']};\nрезервные семейства: {dataset[\'image_counts\'][\'reserved\']}.\nСоставлено {len(dataset[\'splits\'][\'train\'])}/{len(dataset[\'splits\'][\'validation\'])}/{len(dataset[\'splits\'][\'test\'])}\nчетвёрок. Семейства между split не пересекаются, проверены дубликаты пикселей.\nCracks/Gabor/Perlin/Voronoi полностью исключены из обучения и выбора checkpoint.\nTrain-аугментации: повороты на 90°, отражение, циклические сдвиги; все записаны в manifest.\n\n## Результаты до и после\n\nОдна и та же независимая выборка: **{after[\'sample_count\']} четвёрки**.\nОдинаковые входы, оптическая модель, 8-битное квантование ДОЭ и амплитуд.\nВыбранная пользовательская четвёрка не участвовала в обучении.\n\n{table}\n\n![Обучение](learning_curves.png)\n\n![Независимая проверка](holdout_quality.png)\n\n![Четыре изображения: до и после](before_after.png)\n\n## Физика, метрики и ограничения\n\nλ=532 нм, входной шаг 10 мкм, 128×128, z=24.1/25.3/26.5/27.7 мм.\nОдин FFT на плоскость, сохраняется модуль поля, а не интенсивность.\nВсе реконструкции рассчитаны из сохранённого квантованного ДОЭ.\nPNG каждой плоскости масштабирован по собственному максимуму без подгонки по target.\nPearson и локальный SSIM (окно 7×7) рассчитаны по фактическим PNG.\nВремя benchmark — batched throughput после прогрева, не статистическая оценка latency.\nПовторяющиеся исходные изображения в разных тестовых четвёрках не считаются\nнезависимыми новыми источниками: это 28 исходников из двух семейств, не 128 уникальных.\nВысокое качество и преимущество над GS не гарантируются; сравнение показывает\nфактическое изменение, включая ухудшения. Оптический эксперимент не проводился.\n\n## Указанные пользователем ссылки\n\n- [TensorFlow README](https://github.com/Jemuna/Image-Classification-Using-Deep-Learning-in-TensorFlow):\n  прочитан; это описание классификации CIFAR-10, не набор голограмм. Код модели и\n  данные CIFAR-10 не копировались и не использовались для этого дообучения.\n- [OpenGameArt](https://opengameart.org/content/700-noise-textures):\n  доступ из среды заблокирован сетевой политикой; использован только загруженный\n  пользователем ZIP с 262 текстурами. Полный набор из 700 не скачивался.\n- [ResearchGate](https://www.researchgate.net/figure/Aliasing-free-area-in-pixel-units-of-the-chirp-functions-under-the-calculation-condition_fig5_236235124):\n  страница недоступна по сетевой политике, её содержимое не проверено.\n  В код независимо добавлена диагностика частоты обеих квадратичных chirp-функций.\n  Входная chirp проходит консервативную проверку Nyquist; выходная требует\n  осторожности при интерполяции комплексной фазы. Она является поточечным множителем\n  единичного модуля и не меняет сохраняемую амплитуду. Полная aliasing-free\n  дискретизация обеих chirp-функций не заявляется.\n\nУдалённый запуск в Kaggle не выполнен: в среде нет настроенной авторизации.\nЛицензия входящего ZIP указана в его License.txt как CC0/Public Domain.\n\nSHA256 до: `{before[\'checkpoint_sha256\']}`.\n\nSHA256 после: `{after[\'checkpoint_sha256\']}`.\n"""\n    (output / "report.md").write_text(markdown)\n    with PdfPages(output / "report.pdf") as pdf:\n        fig = plt.figure(figsize=(8.27, 11.69))\n        fig.text(0.07, 0.95, "Дообучение FresnelNet на текстурах", fontsize=18, weight="bold")\n        paragraphs = [\n            f"CPU: {after[\'device\']}; {training[\'config\'][\'epochs\']} эпох; лучший checkpoint: эпоха {training[\'best_epoch\']}. Время обучения: {training[\'elapsed_seconds\']:.1f} с.",\n            f"Validation physics loss: {training[\'initial_validation_loss\']:.6f} → {training[\'best_validation_loss\']:.6f}.",\n            "Данные: 262 изображения из загруженного ZIP. 140 train, 28 validation, 28 test, 66 в резерве. Семейства не пересекаются. Полный набор из 700 не скачивался.",\n            "Независимая проверка: 32 четвёрки из test. Пользовательские Cracks/Gabor/Perlin/Voronoi полностью исключены из обучения и выбора checkpoint.",\n            f"Test Pearson: {base[\'mean_pearson\']:.5f} → {fine[\'mean_pearson\']:.5f}; SSIM: {base[\'mean_ssim\']:.5f} → {fine[\'mean_ssim\']:.5f}.",\n            f"Выбранная четвёрка: Pearson {old[\'mean_pearson\']:.5f} → {new[\'mean_pearson\']:.5f}; SSIM {old[\'mean_ssim\']:.5f} → {new[\'mean_ssim\']:.5f}.",\n            "Реконструкции из сохранённого 8-битного ДОЭ, модуль поля, без фильтрации и подгонки gain по target. Pearson и локальный SSIM измерены по реальным PNG.",\n            "TensorFlow README описывает классификацию CIFAR-10; эти данные не использованы. OpenGameArt и ResearchGate недоступны по политике сети. Проверка chirp-дискретизации выполнена независимо по формулам модели.",\n            "Качество не гарантируется; улучшения и ухудшения приведены без отбора. Тестовые четвёрки используют повторно 28 исходных текстур. GPU/Kaggle и эксперимент на ПВМС не проверены.",\n        ]\n        y = 0.89\n        for paragraph in paragraphs:\n            lines = textwrap.wrap(paragraph, width=91)\n            fig.text(0.07, y, "\\n".join(lines), fontsize=10, va="top", linespacing=1.5)\n            y -= len(lines) * 0.020 + 0.025\n        pdf.savefig(fig); plt.close(fig)\n        for filename, title, size in (("learning_curves.png", "История обучения", (8.27, 6)),\n                                      ("holdout_quality.png", "Независимая проверка", (10, 6)),\n                                      ("before_after.png", "Выбранная четвёрка: до и после", (8.27, 11.69))):\n            fig, ax = plt.subplots(figsize=size)\n            with Image.open(output / filename) as im:\n                ax.imshow(np.array(im))\n            ax.axis("off"); ax.set_title(title, fontsize=14)\n            fig.tight_layout(); pdf.savefig(fig); plt.close(fig)\n    print(json.dumps({"reports": str(output), "comparisons": rows}, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("training", "dataset", "before-benchmark", "after-benchmark", "before-result", "after-result", "output"):\n        parser.add_argument("--" + name, required=True)\n    generate(**vars(parser.parse_args()))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/stage_kaggle_job.py': '"""Prepare a private Kaggle dataset and a reproducible, larger adaptive GPU job."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\nimport nbformat\n\n\ndef stage(archive,checkpoint,output,username=\'fiskarik\',dataset_slug=\'nuclear-hack-doe-training-v3\'):\n    root=Path(__file__).resolve().parents[1]\n    archive,checkpoint,output=Path(archive),Path(checkpoint),Path(output)\n    if output.exists() and any(output.iterdir()):raise FileExistsError(output)\n    dataset=output/\'dataset\';notebook=output/\'notebook\'\n    dataset.mkdir(parents=True);notebook.mkdir()\n    shutil.copy2(archive,dataset/\'sbs-textures-128.zip\')\n    shutil.copy2(checkpoint,dataset/\'adaptive-parent.pt\')\n    metadata={\'title\':\'Nuclear Hack DOE Training V3\',\'id\':f\'{username}/{dataset_slug}\',\n              \'licenses\':[{\'name\':\'other\'}],\n              \'description\':\'Private inputs: Screaming Brain Studios textures carry CC0/Public Domain; model weights remain copyright of the repository owner and are staged for private training only.\'}\n    (dataset/\'dataset-metadata.json\').write_text(json.dumps(metadata,indent=2)+\'\\n\')\n    (dataset/\'README.md\').write_text(\'Private training inputs. SBS texture archive includes its CC0/Public Domain License.txt. \'\n        \'Model weights belong to the repository owner; no new public license is granted by staging. \'\n        \'Use datasets create without --public.\\n\')\n    nb=nbformat.read(root/\'notebooks/fresnel_doe_kaggle.ipynb\',as_version=4)\n    code=[c for c in nb.cells if c.cell_type==\'code\']\n    code[0].source+=\'\'\'\n# Resolve only the files from the attached private dataset; credentials stay outside the notebook.\ndef unique_input(filename):\n    paths = list(Path(\'/kaggle/input\').rglob(filename))\n    if len(paths) != 1:\n        raise RuntimeError(f\'Expected one {filename}, found {len(paths)}\')\n    return str(paths[0])\nif os.environ.get(\'DOE_QUICK_CHECK\') != \'1\':\n    os.environ[\'DOE_TEXTURE_ZIP\'] = unique_input(\'sbs-textures-128.zip\')\n    os.environ[\'DOE_INITIAL_CHECKPOINT\'] = unique_input(\'adaptive-parent.pt\')\nos.environ[\'DOE_MODEL_KIND\'] = \'adaptive\'\n\'\'\'\n    code[1].source+=\'\'\'\n# Larger GPU experiment; no claim that 0.8 is guaranteed.\nconfig.objective = \'perceptual\'\nif not GPU and not QUICK_CHECK:\n    raise RuntimeError(\'This job requires an enabled Kaggle GPU; check account verification and quota\')\nif not QUICK_CHECK:\n    counts = (2048, 32, 32)\n    config.stages = max(24, config.stages)\n    config.initializer_depth = 4\n    config.epochs = 80\n    config.batch_size = 8\n    config.learning_rate = 0.0003\nprint(\'GPU experiment configuration:\', config)\n\'\'\'\n    nb.cells.insert(1,nbformat.v4.new_markdown_cell(\'\'\'## Подготовленный GPU-эксперимент\n\nПриватный входной набор: SBS ZIP и родительские веса. 24 обучаемых оптических слоя,\n80 эпох, 2048 train-четвёрок. Validation/test остаются по 32 четвёрки и не используются\nкак обучающие данные. Требуется Kaggle GPU и доступная квота. Цель 0.8 пока не\nподтверждена; этот запуск предназначен для дальнейшего исследования.\n\'\'\'))\n    nbformat.validate(nb)\n    nbformat.write(nb,notebook/\'adaptive_gpu.ipynb\')\n    kernel={\'id\':f\'{username}/nuclear-hack-adaptive-gpu\',\'title\':\'Nuclear Hack Adaptive GPU\',\n            \'code_file\':\'adaptive_gpu.ipynb\',\'language\':\'python\',\'kernel_type\':\'notebook\',\'is_private\':True,\n            \'enable_gpu\':True,\'enable_internet\':False,\'dataset_sources\':[f\'{username}/{dataset_slug}\'],\n            \'competition_sources\':[],\'kernel_sources\':[],\'model_sources\':[]}\n    (notebook/\'kernel-metadata.json\').write_text(json.dumps(kernel,indent=2)+\'\\n\')\n    manifest={\'archive_sha256\':hashlib.sha256(archive.read_bytes()).hexdigest(),\n              \'checkpoint_sha256\':hashlib.sha256(checkpoint.read_bytes()).hexdigest(),\n              \'dataset\':metadata[\'id\'],\'kernel\':kernel[\'id\'],\'privacy\':\'private; never add --public\',\n              \'runtime_status\':\'not submitted; authentication and applied Kaggle network access required\'}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    (output/\'README.md\').write_text(f\'\'\'# Prepared Kaggle GPU experiment\n\nNo remote submission has occurred. Keep the dataset and notebook private.\nUse the existing cloud KAGGLE_API_TOKEN binding and HTTPS proxy. Never paste tokens into code or chat.\nRun the following commands from this job directory.\n\n1. Verify authentication: `kaggle kernels list --mine --page-size 1`.\n2. Check `{metadata[\'id\']}` is not an unrelated dataset. If unused, run\n   `kaggle datasets create -p dataset` without `--public`.\n3. Wait until the dataset is ready, then run\n   `kaggle kernels push -p notebook --accelerator NvidiaTeslaT4 -t 21600`.\n4. Inspect `kaggle kernels status {kernel[\'id\']}`. After complete, download the output\n   with `kaggle kernels output {kernel[\'id\']} -p NEW_DIRECTORY` and inspect actual metrics.\n\nSaved files and a successful upload are not evidence of a completed GPU training run.\n\'\'\')\n    print(json.dumps(manifest,indent=2));return manifest\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'archive\',\'checkpoint\',\'output\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--username\',default=\'fiskarik\');p.add_argument(\'--dataset-slug\',default=\'nuclear-hack-doe-training-v3\')\n    stage(**vars(p.parse_args()))\n', 'scripts/validate_notebook.py': '"""Execute every notebook cell; quick mode validates integration, not model quality."""\nimport argparse\nimport os\nfrom pathlib import Path\n\nimport nbformat\nfrom nbclient import NotebookClient\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", default="notebooks/fresnel_doe_kaggle.ipynb")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--workdir", required=True)\n    parser.add_argument("--kernel", default="python3")\n    parser.add_argument("--quick-check", action="store_true")\n    args = parser.parse_args()\n    notebook = nbformat.read(args.notebook, as_version=4)\n    workdir = Path(args.workdir).resolve()\n    workdir.mkdir(parents=True, exist_ok=True)\n    output = Path(args.output)\n    if output.exists():\n        raise FileExistsError(output)\n    if args.quick_check:\n        os.environ["DOE_QUICK_CHECK"] = "1"\n    else:\n        os.environ.pop("DOE_QUICK_CHECK", None)\n    client = NotebookClient(notebook, timeout=1800, kernel_name=args.kernel,\n                            resources={"metadata": {"path": str(workdir)}})\n    client.execute()\n    executed = 0\n    for cell in notebook.cells:\n        if cell.cell_type == "code":\n            assert cell.execution_count is not None\n            assert not any(x.output_type == "error" for x in cell.outputs)\n            executed += 1\n    notebook.metadata["validation"] = {"quick_check": args.quick_check, "executed_code_cells": executed}\n    output.parent.mkdir(parents=True, exist_ok=True)\n    nbformat.write(notebook, output)\n    print(f"Executed {executed} code cells; quick_check={args.quick_check}; {output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'docs/adaptive-training.md': '# Обученный adaptive wavefront и запуск Kaggle\n\nРезультаты: [v3, PDF, веса и изображения](../deliverables/quality_research/v3/README.md).\nЧистая сеть улучшилась на прежнем test из 32 четвёрок до **0.60747 Pearson /\n0.43629 SSIM**. Оба 0.8 пока не достигнуты. Более тяжёлая архитектура и большее\nчисло эпох сами по себе не гарантируют этот порог.\n\n## Воспроизведение CPU-обучения\n\nИспользуйте новый пустой output-каталог для каждой команды.\nСначала подготовьте набор из пользовательского SBS ZIP:\n\n```bash\npython scripts/prepare_texture_dataset.py --archive /path/to/sbs-textures.zip \\\n  --output data/textures_v1 --train 256 --validation 32 --test 32\npython -m doe train --data data/textures_v1 --output artifacts/wavefront_v1 \\\n  --initial-checkpoint deliverables/texture_finetuned/best.pt \\\n  --architecture wavefront --stages 6 --width 12 --initializer-width 8 \\\n  --initializer-depth 3 --objective perceptual --epochs 20 --batch-size 4 \\\n  --learning-rate 0.001 --device cpu --threads 2\npython -m doe train --data data/textures_v1 --output artifacts/adaptive_v1 \\\n  --initial-checkpoint artifacts/wavefront_v1/best.pt \\\n  --architecture adaptive --stages 12 --width 12 --initializer-width 8 \\\n  --initializer-depth 4 --objective perceptual --epochs 16 --batch-size 4 \\\n  --learning-rate 0.0005 --device cpu --threads 2\npython scripts/compare_models.py --data data/textures_v1 \\\n  --checkpoint baseline=deliverables/texture_finetuned/best.pt \\\n  --checkpoint adaptive=artifacts/adaptive_v1/best.pt \\\n  --output artifacts/adaptive-comparison.json --threads 1 --batch-size 4\n```\n\nКвантование фазы и реконструкций в 8 бит одинаково для всех методов.\nУлучшение сети сравнивается только при одинаковой оптике и входных изображениях.\nСтарые checkpoint без поля `kind` поддерживаются. Переход к U-Net и увеличение\nчисла слоёв сохраняют прежние параметры; новые блоки инициализируются отдельно.\nAdam начинает с нового состояния, как указано в training.json.\n\n## Подготовка приватного Kaggle job\n\n```bash\npython scripts/stage_kaggle_job.py --archive /path/to/sbs-textures.zip \\\n  --checkpoint deliverables/quality_research/v3/best.pt \\\n  --output artifacts/kaggle-v3 --username fiskarik\n```\n\nСкрипт не обращается к Kaggle. Он готовит dataset с исходным ZIP и весами,\nсамодостаточный notebook, приватные metadata и SHA256. В репозиторий включена\nкопия notebook/metadata, а входные бинарные файлы собираются командой выше.\nSBS ZIP содержит CC0/Public Domain; для весов новая публичная лицензия не выдаётся.\nDataset metadata использует `other` с пояснением, dataset и notebook приватные.\n\nПолная конфигурация: 24 оптических слоя, 4 уровня U-Net, 80 эпох, batch 8,\nAdam lr=0.0003, perceptual loss, 2048 train-четвёрок. Это больше комбинаций\nпрежних train-текстур, а не 2048 независимых исходных наборов. Seed и 32\nvalidation/test-четвёрки прежние. Без GPU полный job останавливается с понятной\nошибкой. Нужны доступная GPU-квота и подтверждённый аккаунт Kaggle.\n\nПеред отправкой подключите KAGGLE_API_TOKEN через защищённые настройки среды,\nразрешите HTTPS к api.kaggle.com и www.kaggle.com, сохраните и опубликуйте настройки.\nТокен не нужен внутри notebook и не должен попадать в код, Git или чат.\nОткройте сгенерированный README: там команды проверки входа, создания приватного\ndataset, отправки job, статуса и скачивания результатов. Если указанный dataset\nили notebook уже занят другим проектом, сначала выберите новый slug.\n\nПолный GPU-запуск пока не отправлен. Сохранение файлов и локальный quick-прогон\nне являются подтверждением удалённого обучения. После выполнения проверяются\nфактические benchmark.json и PNG, а не только статус job.\n\n## Отдельные исследовательские режимы\n\n`probe_progressive.py`, `probe_raar.py`, `probe_homotopy.py`, `probe_quality.py`\nи `probe_neural_fit.py` содержат диагностические эксперименты. Подгонка весов\nпо одному target не является измерением обобщения сети.\n\n`doe/camera_optics.py` и `probe_camera.py` задают отдельную физическую сетку\nкамеры через нулевое дополнение перед FFT. Меняются шаг и поле зрения; эти\nоценки вынесены в отдельные строки отчёта и не выдаются за улучшение нейросети\nна прежней сетке. В основном CLI и checkpoint используется прежний Fresnel.\n', 'docs/quality-research.md': '# Исследование цели Pearson ≥ 0.8 и SSIM ≥ 0.8\n\n**Новая обученная архитектура:** [v3: adaptive wavefront и PDF](../deliverables/quality_research/v3/README.md).\nЧистая сеть на том же test улучшилась с 0.57364/0.41852 до 0.60747/0.43629.\nСтарый v2-гибрид после уточнения фазы остаётся лучше нового гибрида.\nПолный [следующий Kaggle job](adaptive-training.md) подготовлен; для отправки\nне хватает подключённого секрета и применённого доступа к Kaggle.\n\n**Новый этап завершён:** [полные результаты, PDF, изображения и код](../deliverables/quality_research/v2/README.md).\nНа test из 32 четвёрок сеть + уточнение общей фазы дали 0.67111/0.60637,\nна выбранной пользовательской четвёрке — 0.59732/0.46088 с 3000 шагами.\nПорог 0.8 не достигнут. Контроль с четырьмя отдельными ДОЭ даёт около 0.990/0.951,\nно не соответствует исходному ТЗ. Далее сохранён журнал первого этапа.\n\nЦель пользователя — оба показателя одновременно. Исходные физические условия,\nPNG-квантование, 4 входа, 1 фазовый ДОЭ и протокол метрик сохраняются.\nНиже — промежуточные эксперименты, **не заявление о достижении 0.8**.\n\n## Проверенные источники\n\n- [Neural Holography, Peng et al., SIGGRAPH Asia 2020](https://github.com/computational-imaging/neural-holography):\n  README и algorithms.py прочитаны. Подходы включают GS, SGD, HoloNet, U-Net,\n  camera-in-the-loop и дифференцируемую оптику. Их пример использует angular\n  spectrum и обучаемый масштаб амплитуды; автоматически переносить численные\n  оценки на нашу одно-FFT четырёхплоскостную задачу нельзя.\n- [Tensor Holography V1/V2, Shi et al.](https://github.com/liangs111/tensor_holography):\n  README прочитан. Возможные направления — обучение phase-only синтезу и\n  представление комплексного поля. Их постановка и кодировщики отличаются от\n  нашего ДОЭ 128×128. Код и веса из этих проектов не копировались.\n\n## Первый контролируемый эксперимент\n\nВзяты первые 4 **validation**-четвёрки текстур, до просмотра новых результатов.\nИспользованы дообученные веса `deliverables/texture_finetuned/best.pt`.\nОптимизируется сама входная фаза, а не параметры сети. Это диагностический\nитеративный расчёт для входных изображений, **не ускоренная нейросеть**.\nНачальные метрики: Pearson **0.40725**, SSIM **0.24378**.\n\n| Loss для 600 шагов Adam | Pearson после | SSIM после |\n|---|---:|---:|\n| Существующий physics loss | 0.53670 | 0.34401 |\n| 0.5×ошибка Pearson + 0.5×ошибка SSIM + 0.05×MSE | **0.55028** | **0.45044** |\n\nLR=0.05 с cosine decay, CPU, одинаковая начальная фаза и входы. Новый loss\nиспользует straight-through 8-битное квантование и ту же нормировку каждого\nмодуля по его максимуму. Метрики таблицы вычислены отдельной существующей\nфункцией `phase_metrics`, включая реальное квантование фазы и PNG-амплитуды.\nИстория каждого прогона сохранена в `deliverables/quality_research`.\n\nSSIM-loss проверен против `skimage` на точность и конечными разностями по градиенту;\nфазовое квантование сверено с существующим PNG-кодировщиком. Новые тесты прошли.\n\n## Вывод и дальнейшая работа\n\nСогласование loss с метриками помогает, но **эти два эксперимента не достигли\n0.8 и не доказывают математическую невозможность цели**. Нужны проверки\nинициализации фазы, других оптимизаторов и глубины сети. Настройка выполняется\nпо validation; тестовая выборка и выбранная четвёрка не используются для\nвыбора параметров этих экспериментов.\n\nНельзя выдавать за достижение цели оценки после фильтрации реконструкций,\nобрезки неудобных областей, замены PNG входов или изменения SSIM. Более\nсложный итеративный метод нужно измерять отдельно от одного forward pass сети.\n\nВоспроизведение:\n\n```bash\npython scripts/probe_quality.py --objective perceptual --steps 600 --count 4 \\\n  --lr 0.05 --output NEW_RESULT.json\npython scripts/probe_quality.py --objective physics --steps 600 --count 4 \\\n  --lr 0.05 --output ANOTHER_RESULT.json\n```\n', 'docs/texture-finetuning.md': '# Дообучение на пользовательских текстурах\n\nДообучаются существующие веса `deliverables/best.pt`, а не новая случайно\nинициализированная сеть. Архитектура сохраняется: 6 оптических слоёв, CNN width=12.\nДля fine-tuning добавлен параметр `--initial-checkpoint`; оптимизатор Adam\nинициализируется заново. В checkpoint и training.json записывается SHA256 родителя.\n\n## Данные\n\nДоступен загруженный пользователем ZIP с 262 PNG 128×128. Лицензия внутри архива —\nCC0/Public Domain, Screaming Brain Studios. RGB/RGBA преобразуются в L через\nalpha-composite на чёрный фон; размер и контраст не меняются.\n\nРазделение фиксировано до обучения, на уровне семейств:\n\n| Назначение | Семейства | Исходных PNG | Четвёрок |\n|---|---|---:|---:|\n| train | Craters, Grainy, Manifold, Marble, Melt, Milky, Spokes, Super Noise, Swirl, Techno | 140 | 256 |\n| validation | Super Perlin, Turbulence | 28 | 32 |\n| test | Streak, Vein | 28 | 32 |\n| резерв для пользовательской четвёрки | Cracks, Gabor, Perlin, Voronoi | 66 | отдельный расчёт |\n\nИсточник не попадает в разные split; дополнительно проверяются точные дубликаты\ngrayscale-пикселей. Train-аугментации: отражение, поворот на 90°, циклический сдвиг.\nКаждый исходник и преобразование записываются в manifest. Test и validation не\nаугментируются. 32 тестовые четвёрки используют 28 исходных изображений повторно,\nа не представляют 128 независимых оригиналов.\n\n## Воспроизведение\n\n```bash\npython scripts/prepare_texture_dataset.py --archive PATH_TO_ZIP --output data/textures-new\npython -m doe train --data data/textures-new --output artifacts/textures-new \\\n  --initial-checkpoint deliverables/best.pt --epochs 30 --learning-rate 0.001 \\\n  --batch-size 4 --stages 6 --width 12 --device auto\n```\n\nЗатем оцените исходную и дообученную модели на **том же** test split командой\n`doe evaluate`. Примените обе к одной четвёрке командой `doe predict` и соберите\nотчёт:\n\n```bash\npython scripts/report_texture_finetuning.py \\\n  --training artifacts/textures-new/training.json --dataset data/textures-new \\\n  --before-benchmark BEFORE_BENCHMARK_JSON --after-benchmark AFTER_BENCHMARK_JSON \\\n  --before-result BEFORE_RESULT_DIR --after-result AFTER_RESULT_DIR --output NEW_REPORT_DIR\n```\n\nСоздаются PDF, Markdown, CSV, JSON и графики. Генератор проверяет SHA256 родителя,\nсоответствие checkpoint и одинаковые входы до/после. Результат включает фактические\nухудшения, если они есть; улучшение не предполагается заранее.\n\nВ Kaggle notebook задайте `TEXTURE_ZIP` и `INITIAL_CHECKPOINT` на файлы,\nприкреплённые как Kaggle Input. Он автоматически наследует архитектуру checkpoint,\nготовит family-disjoint набор, дообучает сеть, сравнивает до/после и собирает отчёт.\nВнешний интернет внутри notebook не требуется. Короткий интеграционный прогон\nпроверяет этот режим отдельно от полного обучения.\n\n## Предоставленные ссылки\n\n- [TensorFlow README](https://github.com/Jemuna/Image-Classification-Using-Deep-Learning-in-TensorFlow/blob/main/README.md)\n  прочитан: это описание классификации grayscale CIFAR-10. Из него не взяты данные\n  для обучения; задача синтеза фазового ДОЭ не подменяется классификацией.\n- [700 Noise Textures](https://opengameart.org/content/700-noise-textures) и\n  [иллюстрация на ResearchGate](https://www.researchgate.net/figure/Aliasing-free-area-in-pixel-units-of-the-chirp-functions-under-the-calculation-condition_fig5_236235124)\n  недоступны по текущей политике сети. Домены добавлены в черновик настроек среды,\n  но сохранение черновика само по себе не применяет разрешение. Содержимое этих\n  страниц не подтверждено. Полный набор из 700 не скачивался; использован только\n  реально доступный пользовательский ZIP с 262 PNG.\n\n## Дискретизация Френеля\n\nНезависимо от недоступной иллюстрации добавлена диагностика обеих chirp-функций.\nКонсервативная граница входной chirp: `z_critical=N*du²/λ≈24.06015 мм`.\nВсе расстояния задачи чуть выше этой границы и проходят проверку входной chirp.\nСоответствующие отношения к Nyquist для выходной chirp, напротив, выше единицы:\nэто важно для интерполяции комплексной фазы. В нашем одно-FFT расчёте выходная\nchirp только поточечно умножает поле на фазовый множитель единичного модуля и не\nменяет сохраняемый модуль. Утверждение, что обе chirp-функции полностью свободны\nот aliasing во всём поле, не делается. Диагностика включена в новые metrics.json.\n', 'docs/validation.md': '# Фактическая проверка решения\n\nОбновление: [обучение adaptive wavefront и полный отчёт v3](../deliverables/quality_research/v3/README.md).\nСейчас проходят 26 тестов; все 6 ячеек нового Kaggle job выполнены локально\nв quick-режиме с родительскими adaptive-весами. Отдельно проверены один CPU-epoch\nи перенос 12 → 24 слоёв (8 train, 4 validation). Это интеграционная проверка,\nне полное GPU-обучение. Ниже сохранён первоначальный benchmark процедурного набора.\n\nПроверено в текущей облачной среде на CPU. GPU и удалённое выполнение в Kaggle\n**не проверены**: в текущей среде нет Kaggle API-токена и применённого сетевого\nразрешения для Kaggle. Notebook можно импортировать и выполнить вручную в Kaggle\nбез передачи токена в код.\n\n## Обучение\n\n- Python 3.12.14, PyTorch 2.9.1+cpu, NumPy 2.3.5; полный состав —\n  `docs/cpu-environment.txt`.\n- Процедурный набор `data/procedural_v1`: 96/16 изображений-четвёрок train/validation;\n  seed 2026. Каждая четвёрка содержит четыре PNG/L/128×128.\n- 6 оптических слоёв, CNN width=12, batch=4, Adam lr=0.005, 24 эпохи, seed=42,\n  2 CPU-потока, cosine scheduler.\n- Validation loss до обучения: **0.9402181**; лучший: **0.9188025**, эпоха 24.\n- Обучение заняло около **140 секунд** в этом запуске. Веса действительно\n  обновлены; величина изменения параметров и полная история есть в `training.json`.\n- Сеть с обучаемым over-relaxation выбрана по validation. Ранний вариант\n  использовался для разработки; его тестовые результаты не выдаются за финальную\n  независимую оценку. Данные ниже — новая выборка с другим seed.\n\n## Независимая финальная выборка\n\n`data/independent_holdout`, seed **91007**, 32 новых четвёрки. Эта выборка не\nиспользовалась для обучения и выбора checkpoint. В manifest сохранены seed и\nSHA256; в `benchmark.json` — результаты каждой четвёрки. Показатели усреднены\nпо четырём плоскостям и затем по 32 четвёркам.\n\n| Метод | Pearson ↑ | SSIM ↑ | мс/четвёрку ↓ |\n|---|---:|---:|---:|\n| FresnelNet, обученная | 0.59643 | 0.09554 | 43.03 |\n| Та же сеть до обучения | 0.58689 | 0.09028 | 44.61 |\n| GS, 6 шагов | 0.58689 | 0.09028 | 25.94 |\n| GS, 80 шагов | 0.61833 | 0.12444 | 158.99 |\n\nИзмерение времени — один проход после прогрева, batch=4; это batched throughput,\nне latency одиночного запроса и не статистическая оценка производительности.\nЗагрузка PNG, обучение, расчёт метрик и запись результатов исключены из времени\nсинтеза. Округление phase/amplitude в 8 бит включено в качество всех методов.\n\nОбучение даёт умеренное улучшение относительно такой же сети до обучения и GS\nс тем же числом шагов. GS за 80 шагов качественнее, но медленнее в данном CPU-прогоне.\n**Преимущество по качеству над классическим методом не заявляется.** Малый SSIM\nи видимые помехи означают, что это рабочая исходная реализация, а не решение с\nгарантированно высоким качеством. Для соревнования нужны проверка на реальных\nизображениях, расширение обучающего набора и дальнейшая настройка.\n\n## Экспортированная демонстрация\n\nИспользована первая четвёрка независимой выборки, `test/sample_0000`, а не отобранный\nлучший пример. Исходные PNG, ДОЭ и реконструкции находятся в `run/result` архива.\n\n| z, мм | Pearson | SSIM |\n|---:|---:|---:|\n| 24.1 | 0.68916 | 0.14880 |\n| 25.3 | 0.39648 | 0.08306 |\n| 26.5 | 0.62391 | 0.07223 |\n| 27.7 | 0.65105 | 0.07878 |\n\nСреднее: Pearson **0.59015**, SSIM **0.09572**. PNG амплитуд масштабированы по\nсобственному максимуму; gain по целевым изображениям не подбирался. Видимые спекл\nи межплоскостные помехи сохранены, фильтрация не применялась.\n\n## Что прошло\n\n- **8 тестов pytest**, 0 пропущенных: прямая квадратура дифракции без FFT,\n  сопряжённость, сохранение физической мощности на разных сетках, finite-difference\n  градиент фазы, градиенты CNN, кодирование фазы, Pearson/SSIM, строгие PNG,\n  воспроизводимые данные, реконструкции сохранённого квантованного ДОЭ.\n- Все **6 исполняемых ячеек** автономного notebook выполнены локально через Jupyter.\n  Для этой интеграционной проверки включён `DOE_QUICK_CHECK=1`: 8/4/4 примера,\n  2 эпохи, 2 слоя, width=4. Проверены обучение, benchmark, экспорт изображений и\n  сборка ZIP. Это **не** проверка полной GPU-конфигурации notebook в Kaggle.\n- Полное обучение основной CPU-модели и оценка 32 новых четвёрок выполнены отдельно\n  от короткого notebook-прогона.\n\n## Что остаётся\n\nВ исходных DOCX/PDF нет четырёх PNG объектов организаторов: результаты выше\nотносятся только к процедурной демонстрации. Для выполнения на нужной четвёрке\nпередайте её в `doe predict` или `INPUT_IMAGES` notebook. Код не подменяет и не\nдорисовывает официальные входы.\n\nДля автоматического запуска в аккаунте `fiskarik` требуются защищённый\n`KAGGLE_API_TOKEN` и разрешённый сетевой доступ. Настройки сохранены в черновике\nсреды ранее; их сохранение и публикация не подтверждают выполнение в Kaggle.\nЭкспериментальная проверка оптики на физическом ПВМС не проводилась.\n', 'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "fresnel-doe"\nversion = "0.1.0"\ndescription = "Physics-informed neural synthesis of a four-plane phase-only diffractive optical element"\nrequires-python = ">=3.11"\ndependencies = [\n  "torch>=2.4,<3",\n  "numpy>=1.26,<3",\n  "Pillow>=10.1,<13",\n  "scikit-image>=0.24,<1",\n  "matplotlib>=3.8,<4",\n]\n\n[project.optional-dependencies]\ndev = ["pytest>=8,<9", "nbformat>=5.10,<6", "nbclient>=0.10,<1", "ipykernel>=6.29,<7"]\n\n[project.scripts]\nfresnel-doe = "doe.cli:main"\n\n[tool.setuptools.packages.find]\ninclude = ["doe*"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n', 'requirements.txt': '# PyTorch is already installed on Kaggle; retain its CUDA build there.\n# For local CPU setup, install torch from https://download.pytorch.org/whl/cpu first.\ntorch==2.9.1\nnumpy==2.3.5\nPillow==12.0.0\nscikit-image==0.25.2\nmatplotlib==3.10.7\n', 'requirements-dev.txt': 'pytest==8.4.2\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.30.1\n', 'README.md': "# Нейросетевой синтез четырёхплоскостного ДОЭ\n\nРешение задания хакатона из «Хакатон_задача_q2.docx» и «AI фотоника.pdf»:\nчетыре изображения → один фазовый дифракционный оптический элемент → четыре\nчисленно восстановленных изображения и метрики качества.\n\n**В документах нет исходной четвёрки объектных PNG.** В репозитории есть\nпроцедурный демонстрационный набор, обучение и проверка на независимом test split.\nКачество на изображениях организаторов нужно измерить отдельно. Симуляция не\nзаменяет эксперимент на физическом ПВМС.\n\n## Готовые материалы\n\n**Новая обученная архитектура:** [adaptive wavefront, веса и PDF-отчёт](deliverables/quality_research/v3/README.md).\nЧистая сеть на том же test: **0.60747 Pearson / 0.43629 SSIM** против прежних\n0.57364/0.41852. Обучены U-Net и 12 адаптивных оптических слоёв.\n[Подготовлен следующий Kaggle GPU job](docs/adaptive-training.md), удалённый запуск\nпока блокируют секрет и сетевые настройки. Оба порога 0.8 пока не достигнуты.\n\nИсследование новой цели **Pearson ≥0.8 и SSIM ≥0.8**:\n[методика, источники и промежуточные результаты](docs/quality-research.md).\nЦель пока не достигнута; текущие опубликованные веса не заменены экспериментом.\n\n**Результаты исследования:** [PDF, код и режим уточнения фазы](deliverables/quality_research/v2/README.md).\nОдин общий ДОЭ: test **0.671 Pearson / 0.606 SSIM** после 600 шагов.\nДля выбранной четвёрки: **0.597 / 0.461** после 3000 шагов. Это сеть плюс\nитеративный solver; результаты чистой сети и контроль с отдельными ДОЭ приведены\nотдельно. Прежние production-веса сохранены.\n\n- **Последнее дообучение на текстурах:** [отчёты и результаты](deliverables/texture_finetuned/README.md),\n  [PDF](deliverables/texture_finetuned/reports/report.pdf),\n  [новые веса](deliverables/texture_finetuned/best.pt),\n  [полный архив](deliverables/texture_finetuned/submission.zip).\n- [Архив базовой версии до дообучения](deliverables/submission.zip): исходники, данные,\n  checkpoint, обучение, независимая проверка и изображения.\n- [Обученные веса](deliverables/best.pt) — готовая CPU-модель, 6 слоёв, 24 эпохи.\n- [Пример ДОЭ и четырёх реконструкций](deliverables/result/).\n- [Результаты независимой проверки](deliverables/benchmark.json) и\n  [история обучения](deliverables/training.json).\n- [Kaggle Notebook](notebooks/fresnel_doe_kaggle.ipynb).\n\nОтдельно рассчитана предоставленная пользователем четвёрка текстур:\n[входы и порядок плоскостей](examples/noise_quartet/README.md),\n[ДОЭ, реконструкции и метрики](deliverables/noise_quartet/README.md).\nИспользована готовая сеть, без переобучения.\n\nРежим дообучения на текстурах и описание указанных источников:\n[docs/texture-finetuning.md](docs/texture-finetuning.md). В Kaggle notebook можно\nзадать `TEXTURE_ZIP` и `INITIAL_CHECKPOINT`; он сформирует PDF/CSV-отчёт до/после.\n\nДля инференса без повторного обучения используйте `--checkpoint deliverables/best.pt`\nв команде `doe predict`. После распаковки полного архива этот checkpoint также\nнаходится в `run/best.pt`. Веса загружаются с `weights_only=True`.\n\n## Что реализовано\n\n- `FresnelNet`: обучаемая развёрнутая сеть с физическими слоями и CNN-коррекциями;\n  вход `[B,4,128,128]`, выход `[B,128,128]` в пределах `[-π,π]`.\n- Дифференцируемый прямой расчёт Френеля с **одним FFT на плоскость**.\n- Фазовый ДОЭ `doe_phase.png`: PNG/L/8 бит/128×128, `0 → −π`, `255 → +π`.\n- Четыре `reconstruction_1.png` … `reconstruction_4.png`: модули полей,\n  полученных **из сохранённого, квантованного PNG ДОЭ**.\n- Pearson и SSIM для каждой пары исходное/восстановленное; JSON и CSV.\n- Неокруглённые амплитуды и комплексные поля `.npy`, физические параметры,\n  контрольные суммы входов, ДОЭ и checkpoint.\n- Обучение без размеченных голограмм, train/validation/test, контроль до обучения,\n  сравнение с многоплоскостным алгоритмом Герчберга–Сэкстона.\n- Самодостаточный Kaggle notebook, тесты, веса и материалы для экспертов.\n\n## Запуск в Kaggle\n\nИмпортируйте **[notebooks/fresnel_doe_kaggle.ipynb](notebooks/fresnel_doe_kaggle.ipynb)**\nчерез Kaggle → New Notebook → File → Import Notebook. Включите GPU при доступной\nквоте и выполните Run All. Весь Python-код встроен в notebook обычным текстом;\nклонирование GitHub, интернет и API-токен **внутри Kaggle** не требуются.\nИспользуются предустановленные PyTorch, NumPy, Pillow, scikit-image и matplotlib.\nВерсии фактического окружения сохраняются в `runtime.json`.\n\nNotebook генерирует данные, обучает сеть, оценивает test split, экспортирует\nрезультат и собирает `submission.zip` со всеми материалами. На GPU по умолчанию\n512/64/32 четвёрки train/validation/test, 20 эпох, 8 оптических слоёв. На CPU —\n96/16/16, 24 эпохи, 6 слоёв. При наличии своих PNG добавьте их как Kaggle Dataset и\nзадайте **четыре конкретных пути** в `INPUT_IMAGES` в порядке расстояний ниже.\nДемонстрационный режим явно отмечается в выходном отчёте.\n\nДля отправки из облачной среды через CLI нужен её настроенный доступ к Kaggle:\n\n```bash\n/workspace/.tools/kaggle/bin/kaggle kernels push -p notebooks --accelerator NvidiaTeslaT4\n/workspace/.tools/kaggle/bin/kaggle kernels status fiskarik/nuclear-hack-fresnel-doe\n/workspace/.tools/kaggle/bin/kaggle kernels output fiskarik/nuclear-hack-fresnel-doe -p artifacts/kaggle\n```\n\n`push` создаёт/обновляет и запускает приватный notebook; перед первым push убедитесь,\nчто slug не занят другой вашей работой. API-токен вводится только в защищённых\nнастройках облачной среды как `KAGGLE_API_TOKEN`. Статус `complete` и проверка\nрезультатов необходимы, прежде чем считать удалённый запуск успешным. Сетевой доступ,\nверификация аккаунта и доступность ускорителя определяются средой и Kaggle.\n\n## Локальный запуск\n\nPython 3.11+, рекомендуется 3.12. Для CPU можно установить PyTorch отдельно:\n\n```bash\npython3 -m venv .venv\nsource .venv/bin/activate\npython -m pip install torch==2.9.1 --index-url https://download.pytorch.org/whl/cpu\npython -m pip install -r requirements.txt\npython -m pip install -e '.[dev]'\npython -m pytest -q\n```\n\nНе заменяйте CUDA-сборку PyTorch в Kaggle локальной CPU-сборкой. `requirements.txt`\nфиксирует основной проверенный стек; полный состав CPU-окружения записан в\n`docs/cpu-environment.txt`. В подготовленной облачной среде интерпретатор находится\nв `/workspace/.tools/photonics/bin/python`; команды выполняются из `/workspace/Nuclear-hack`.\n\n```bash\npython -m doe generate --output data/my-run --train 96 --validation 16 --test 16\npython -m doe train --data data/my-run --output artifacts/my-run \\\n  --epochs 24 --learning-rate 0.005 --stages 6 --width 12 --batch-size 4 --device auto\npython -m doe evaluate --data data/my-run --checkpoint artifacts/my-run/best.pt \\\n  --output artifacts/my-run/benchmark.json\npython -m doe predict --checkpoint artifacts/my-run/best.pt \\\n  --images image1.png image2.png image3.png image4.png --output artifacts/result\n```\n\nРезультаты предыдущего запуска не перезаписываются: используйте новый каталог.\nИзображения должны быть именно PNG/L/8 бит/128×128; код не меняет размер и не\nпревращает RGB в grayscale молча. Порядок файлов — `24.1, 25.3, 26.5, 27.7 мм`.\nПолностью чёрная плоскость отклоняется: фазовый ДОЭ при равномерном ненулевом\nосвещении сохраняет энергию и не может дать нулевое поле на всей выходной плоскости.\n\n## Физическая модель\n\n| Параметр | Значение |\n|---|---|\n| Размер | 128×128 |\n| Длина волны λ | 532 нм |\n| Входной шаг Δu=Δv | 10 мкм |\n| Расстояния z | 24.1, 25.3, 26.5, 27.7 мм |\n| Освещение | равномерное, амплитуда 1 |\n| ДОЭ | H(u,v)=exp(iφ(u,v)), φ∈[−π,π] |\n\nИспользуется формула из задания с дискретизационным множителем Δu²:\n\n```text\nU_z(x,y) = exp(ikz)/(i λ z) · exp[iπ(x²+y²)/(λz)]\n           · Δu² · FFT{exp(iφ(u,v)) · exp[iπ(u²+v²)/(λz)]},  k=2π/λ.\n```\n\nНачало координат находится в центре массива. FFT центрированное. При FFT с\n`norm='ortho'` восстанавливающий множитель равен `N Δu²/(iλz) · exp(ikz)`.\nНикакой подмены одно-FFT модели на angular spectrum или свёртку нет.\n\n**Сетка восстановления зависит от расстояния**:\n`Δx_z=Δy_z=λz/(NΔu)`, то есть примерно\n`10.0166, 10.5153, 11.0141, 11.5128 мкм`. Каждый входной объект трактуется на\nсвоей соответствующей выходной сетке. Это не четыре изображения с одним и тем же\nфизическим шагом. В отчёте сохранён шаг для каждой плоскости.\n\nДля устойчивого обучения оператор нормирован до унитарного: удалён только\nоднородный комплексный множитель каждой плоскости. При экспорте полный физический\nмножитель возвращается. Это не меняет форму модуля, Pearson и масштабированных PNG.\n\n## Нейросетевой метод и обучение\n\n1. Четыре входные амплитуды нормируются до единичного RMS: энергия равномерного\n   фазового ДОЭ фиксирована. Градации серого интерпретируются как **амплитуда**, как\n   требуется при сравнении с модулями в задании, а не как интенсивность.\n2. Детерминированные случайные фазы в плоскостях объектов и обратные операторы\n   Френеля дают начальное поле на ДОЭ. В тестовом режиме используются те же правила.\n3. Каждый слой сети вычисляет четыре поля, накладывает ограничение на амплитуду,\n   переносит их обратно и объединяет с обучаемыми весами. Обучаемое смешивание\n   и общая небольшая CNN корректируют фазу. Модуль поля на ДОЭ остаётся единичным.\n4. Число слоёв фиксировано. При инференсе выполняется один проход сети;\n   **шагов Adam, поиска лучшей фазы и дообучения на тестовом изображении нет**.\n   Каждый слой содержит прямые и обратные FFT: это физически информированная\n   развёрнутая сеть, а не обещание «синтеза без дифракционных расчётов».\n5. Loss состоит из MSE амплитуд с единичным RMS, многомасштабной ошибки и ошибки\n   корреляции. Используются Adam, ограничение градиента и cosine learning rate.\n   Голограммы-эталоны для обучения не требуются.\n\nПроверочные данные — цифры, диски, прямоугольники, кольца и линии с варьируемой\nгеометрией и сглаживанием. Генератор и seed входят в поставку; PNG, manifest и\nSHA256 позволяют воспроизвести набор. Три split имеют разные seed. Checkpoint\nвыбирается **только** по validation loss; отдельный test split используется для\nфинальной оценки. Обучение не гарантирует идеального восстановления четырёх\nнезависимых объектов одним фазовым экраном: остаются спекл, межплоскостные помехи\nи ограничения числа степеней свободы.\n\n## Метрики и честное сравнение\n\n- Фаза сначала округляется в 8 бит, PNG читается обратно. Реконструкции считаются\n  из этой фазы — потеря качества квантования включена в оценку.\n- Сохраняется `|U|`, не `|U|²`. Для PNG каждая плоскость делится на собственный\n  максимум и округляется до 8 бит. Нет подгонки gain по target, вычитания фона,\n  фильтрации или изменения размера. Исходные физические модули сохранены в `.npy`.\n- Pearson вычисляется по центрированным пикселям; для постоянного изображения\n  он математически не определён и записывается как `null`.\n- SSIM: `skimage.metrics.structural_similarity`, окно 7×7, `data_range=1` после\n  деления фактических PNG на 255; используется стандартная локальная версия SSIM.\n- Сравниваются обученная сеть, такая же сеть до обучения и GS с малым/большим\n  числом шагов. Все используют одинаковую физику, входы и квантование.\n- Время измеряется после прогрева с синхронизацией CUDA и не включает загрузку\n  файлов и метрики. В benchmark указано **batched seconds per quartet**, не latency\n  одиночного запроса. Сбор данных и обучение в время инференса не входят.\n- Превосходство по скорости или качеству не предполагается заранее: результаты\n  конкретного измерения находятся в `benchmark.json` и `docs/validation.md`.\n\n## Проверки и структура\n\n```text\ndoe/optics.py       — физика, центрированные FFT и сопряжённый оператор\ndoe/model.py        — нейросеть, loss, классический GS\ndoe/data.py         — строгий ввод PNG и воспроизводимый набор\ndoe/engine.py       — обучение, выбор checkpoint, оценка, инференс\ndoe/results.py      — квантование, реконструкции, метрики, визуализация\ntests/             — независимая квадратура Френеля, adjoint, мощность,\n                     finite differences, CNN-градиенты, PNG и метрики\nnotebooks/         — автономный notebook и приватные метаданные Kaggle\nscripts/           — генерация notebook и архива для экспертов\ndocs/              — фактический отчёт валидации и состав окружения\n```\n\nОптический оператор сравнивается с прямой суммой интеграла дифракции **без FFT**\nна малой сетке. Проверяются сохранение физической мощности с учётом выходного\nшага, сопряжённость, градиент конечными разностями, границы фазы, строгий формат\nPNG, повторное восстановление сохранённого ДОЭ и метрики известных случаев.\n\nОбновление notebook после изменения кода: `python scripts/build_notebook.py`.\nСборка материалов:\n\n```bash\npython scripts/package_submission.py --data data/procedural_v1 \\\n  --run artifacts/cpu_v2 --holdout data/independent_holdout --output artifacts/submission.zip\n```\n"}
for name, source in SOURCE_FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text() != source:
        raise RuntimeError(f'Existing source differs: {path}; preserve it and use a fresh work directory')
    path.write_text(source)
sys.path.insert(0, str(PROJECT))
import torch, numpy as np, PIL, skimage, matplotlib
print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'numpy': np.__version__})
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


## Настройки

Четыре входных файла должны быть PNG, режим L, 8 бит, 128×128. Пустая плоскость
несовместима с фазовым ДОЭ при равномерном освещении и не принимается.
`INPUT_IMAGES=[]` означает демонстрацию, а не выполнение на официальных данных.
CPU использует меньший учебный набор; на GPU можно увеличить число эпох и примеров.
При повторном запуске создаётся новый каталог, предыдущие результаты сохраняются.


In [ ]:
from datetime import datetime, timezone
from doe.engine import TrainConfig, train, evaluate, predict
from doe.data import generate_dataset
import runpy

INPUT_IMAGES = []  # ['/kaggle/input/YOUR_DATA/image1.png', ..., 'image4.png']
REFINE_STEPS = int(os.environ.get('DOE_REFINE_STEPS', '0'))  # Optional, slower phase solver; raw network outputs remain separate
MODEL_KIND = os.environ.get('DOE_MODEL_KIND', 'inherit')  # 'wavefront' or 'adaptive' upgrades a matching-width checkpoint
TEXTURE_ZIP = os.environ.get('DOE_TEXTURE_ZIP') or None  # '/kaggle/input/YOUR_DATA/sbs...128x128.zip'
INITIAL_CHECKPOINT = os.environ.get('DOE_INITIAL_CHECKPOINT') or None  # '/kaggle/input/YOUR_MODEL/best.pt'
if TEXTURE_ZIP and not INITIAL_CHECKPOINT:
    raise ValueError('Texture fine-tuning requires INITIAL_CHECKPOINT')
QUICK_CHECK = os.environ.get('DOE_QUICK_CHECK') == '1'  # integration validation only
GPU = torch.cuda.is_available()
RUN = BASE / ('doe-run-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f'))
RUN.mkdir(parents=True)
DATA = RUN / 'data'
TRAINING = RUN / 'training'
if QUICK_CHECK:
    counts = (8, 4, 4)
    config = TrainConfig(epochs=2, batch_size=2, stages=2, width=4)
else:
    counts = (512, 64, 32) if GPU else (96, 16, 16)
    config = TrainConfig(epochs=20 if GPU else 24, batch_size=8 if GPU else 4,
                         stages=8 if GPU else 6, width=16 if GPU else 12)
if TEXTURE_ZIP:
    counts = (8, 4, 4) if QUICK_CHECK else (256, 32, 32)
    config.epochs = 2 if QUICK_CHECK else 30
    config.learning_rate = 0.001
if INITIAL_CHECKPOINT:
    parent = torch.load(INITIAL_CHECKPOINT, map_location='cpu', weights_only=True)
    config.stages = parent['architecture']['stages']
    config.width = parent['architecture']['width']
    config.architecture = parent['architecture'].get('kind','fixed')
    config.initializer_width = parent['architecture'].get('initializer_width',8)
    config.initializer_depth = parent['architecture'].get('initializer_depth',3)
    config.initial_checkpoint = INITIAL_CHECKPOINT
if MODEL_KIND != 'inherit':
    if MODEL_KIND not in ('fixed','wavefront','adaptive'):
        raise ValueError('Unknown model architecture')
    config.architecture = MODEL_KIND
    if MODEL_KIND == 'adaptive':
        config.stages = max(config.stages, 12)
        config.initializer_depth = 4
print('Run:', RUN, '\nTraining:', config)


## Данные и обучение

Разные seed для train/validation/test; сеть минимизирует ошибку амплитуд после
дифференцируемого одно-FFT оператора Френеля. Лучшая модель выбирается по validation loss.
`initial.pt` нужен для контроля того, чему научилась сеть. `best.pt` — для инференса.


In [ ]:
if TEXTURE_ZIP:
    prepare = runpy.run_path(str(PROJECT / 'scripts/prepare_texture_dataset.py'))['prepare']
    prepare(TEXTURE_ZIP, DATA, train=counts[0], validation=counts[1], test=counts[2])
else:
    generate_dataset(DATA, train=counts[0], validation=counts[1], test=counts[2])
training_report = train(DATA, TRAINING, config)
assert training_report['parameter_l2_change_last_vs_initial'] > 0
print('Best validation epoch:', training_report['best_epoch'])


## Независимая тестовая оценка

Сравнение с той же сетью до обучения и многоплоскостным Герчбергом–Сэкстоном
при одинаковом и большем числе оптических шагов. Все методы оцениваются после
квантования фазы и амплитуд в 8 бит. Время отражает batched throughput на текущем
устройстве; ускорение относительно классического метода не предполагается заранее.


In [ ]:
benchmark = evaluate(DATA, TRAINING / 'best.pt', RUN / 'benchmark.json',
                     batch_size=config.batch_size, gs_steps=12 if QUICK_CHECK else 80)
print(json.dumps(benchmark['summary'], indent=2))
if INITIAL_CHECKPOINT:
    before_benchmark = evaluate(DATA, INITIAL_CHECKPOINT, RUN / 'benchmark-before.json',
                                batch_size=config.batch_size, gs_steps=12 if QUICK_CHECK else 80)


## ДОЭ и восстановленные изображения

Сеть выдаёт фазу, кодируемую как `0 → −π`, `255 → +π` с округлением.
Реконструкции рассчитываются заново **из сохранённого PNG ДОЭ**. Сохраняется модуль
комплексного поля, не квадрат модуля. Для отображения каждый модуль делится на свой
максимум; без вычитания фона и без подгонки к целевому изображению.
Метрики считаются по фактическим сохранённым 8-битным PNG. В `.npy` сохранены также
модули до масштабирования и комплексные поля. Шаг выходной сетки зависит от z:
Δx_out = λz/(NΔu); изображения разных плоскостей задаются на соответствующих сетках.


In [ ]:
if INPUT_IMAGES:
    if len(INPUT_IMAGES) != 4:
        raise ValueError('Specify exactly four paths in increasing distance order')
    input_paths = [Path(p) for p in INPUT_IMAGES]
    data_source = 'user-provided quartet'
elif TEXTURE_ZIP:
    names = runpy.run_path(str(PROJECT / 'scripts/prepare_noise_quartet.py'))['SELECTION']
    input_paths = [DATA / 'pool' / Path(name).parent.name.replace(' ', '_') / Path(name).name for name in names]
    data_source = 'reserved texture families: Cracks, Gabor, Perlin, Voronoi; not used in training or validation'
else:
    input_paths = [DATA / 'test/sample_0000' / f'plane_{i}.png' for i in range(1, 5)]
    data_source = 'synthetic held-out demonstration, not official hackathon inputs'
result = predict(TRAINING / 'best.pt', input_paths, RUN / 'result', source=data_source)
print(json.dumps(result['planes'], indent=2))
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(RUN / 'result/comparison.png')))
display(DisplayImage(filename=str(RUN / 'result/phase_preview.png')))
if REFINE_STEPS:
    refined = predict(TRAINING / 'best.pt', input_paths, RUN / 'result-refined', source=data_source,
                      refine_steps=REFINE_STEPS, refine_lr=0.08)
    print('Neural initializer + phase optimizer (not raw network):', refined['mean_pearson'], refined['mean_ssim'])
    display(DisplayImage(filename=str(RUN / 'result-refined/comparison.png')))
if INITIAL_CHECKPOINT:
    before_result = predict(INITIAL_CHECKPOINT, input_paths, RUN / 'result-before', source=data_source)
if TEXTURE_ZIP:
    report = runpy.run_path(str(PROJECT / 'scripts/report_texture_finetuning.py'))['generate']
    report(TRAINING / 'training.json', DATA, RUN / 'benchmark-before.json', RUN / 'benchmark.json',
           RUN / 'result-before', RUN / 'result', RUN / 'reports')


## Материалы для экспертов

Архив содержит код, полный сгенерированный набор, веса, историю обучения,
тестовое сравнение, входные PNG, фазовый ДОЭ, четыре восстановленных PNG,
метрики и исходные комплексные поля. Качество на официальной четвёрке необходимо
измерить отдельно после её предоставления. Это численное моделирование; оптическая
проверка на ПВМС здесь не выполняется.


In [ ]:
import shutil
submission = RUN / 'submission'
submission.mkdir()
for name, source in SOURCE_FILES.items():
    path = submission / 'source' / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
for name in ('data', 'training', 'result'):
    shutil.copytree(RUN / name, submission / name)
for name in ('result-before', 'reports', 'result-refined'):
    if (RUN / name).exists(): shutil.copytree(RUN / name, submission / name)
if INITIAL_CHECKPOINT:
    shutil.copy2(INITIAL_CHECKPOINT, submission / 'parent.pt')
    shutil.copy2(RUN / 'benchmark-before.json', submission / 'benchmark-before.json')
shutil.copy2(RUN / 'benchmark.json', submission / 'benchmark.json')
(submission / 'runtime.json').write_text(json.dumps({'torch': torch.__version__, 'numpy': np.__version__,
    'pillow': PIL.__version__, 'skimage': skimage.__version__, 'device': benchmark['device'],
    'input_source': data_source, 'quick_check': QUICK_CHECK}, indent=2))
archive = shutil.make_archive(str(RUN / 'submission'), 'zip', submission)
print('Submission archive:', archive)
print('Saved results:', RUN / 'result')
